# OpenMorph on ARC-AGI-3 — a digital organism that learns to play

This notebook is two things at once:

1. **A hands-on intro to [OpenMorph](https://github.com/openmorph/openmorph-arc)** — a new kind of learning
   substrate: *digital organisms* that learn with **no backprop, no gradient tape, no optimizer**. An
   organism is a membrane of cells that **fires on change, settles to a fixed point, learns locally, and
   grows new structure when surprised**.
2. **A working ARC-AGI-3 agent** whose entire brain is one OpenMorph organism, learning each hidden game
   online from the reward signal alone.

The Python API is designed to feel like a familiar ML library (`predict` / `learn` / `act`), so you can
read it top-to-bottom. Let's build it up step by step.

## 0. Install OpenMorph (offline)

The competition kernel has no internet, so we install from wheels: the ARC toolkit from the competition
dataset, and OpenMorph from an attached dataset (`openmorph-arc3`). OpenMorph ships as a single
`abi3` wheel — one binary for every Python ≥ 3.8.

In [ ]:
import glob
# ARC-AGI-3 toolkit (bundled with the competition)
!pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv
# OpenMorph (attached private dataset) — a single abi3 wheel, works on any Python >= 3.8.
# PINNED: the one wheel this notebook installs, by exact version AND SHA-256, so a wheel in any other attached
# dataset (a higher version number, the same file name) is never installed. To ship a new wheel, upload it and set
# both lines to its version and `sha256sum` (kaggle/arc-agi-3/README.md, "The pinned wheel").
OM_WHEEL_VERSION = '1.8.0'
OM_WHEEL_SHA256 = '45bfde8bf267b844d851446a00f5139058e73403ffbefe88743ef34cf1e3d6be'
import hashlib, os
print('inputs:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()
def _pinned_wheel(root, version, sha256):
    """The attached wheel of exactly `version` whose SHA-256 is `sha256`; every other candidate is reported, never used."""
    found = sorted(glob.glob(os.path.join(root, '**', f'openmorph-{version}-*.whl'), recursive=True))
    good = [p for p in found if _sha256(p) == sha256]
    for p in found:
        if p not in good:
            print('REFUSED (hash mismatch):', p, _sha256(p))
    others = [p for p in glob.glob(os.path.join(root, '**', 'openmorph-*.whl'), recursive=True) if p not in found]
    if others:
        print('ignored (not the pinned version):', others)
    assert good, (f'no openmorph-{version} wheel with sha256 {sha256} under {root} — is the dataset attached, '
                  f'and do OM_WHEEL_VERSION / OM_WHEEL_SHA256 name the wheel it holds?')
    return good[0]
_om = _pinned_wheel('/kaggle/input', OM_WHEEL_VERSION, OM_WHEEL_SHA256)
print('installing', _om)
!pip install -q --no-index {_om}
import openmorph as om
assert om.__version__ == OM_WHEEL_VERSION, f'installed openmorph {om.__version__}, pinned {OM_WHEEL_VERSION}'
import sys
print('OpenMorph', om.__version__, '| Python', sys.version.split()[0], '| Agent:', hasattr(om, 'Agent'),
      '| API:', [x for x in dir(om) if not x.startswith('_')])
os.makedirs('/kaggle/working/muse/weights', exist_ok=True)   # kit X: v3's modules go here


## 1. The raw organism: `Morph`

At the bottom is `Morph` — the Rust core. You **stimulate** some cells, the membrane **settles**, and you
**read** the resulting cell values. No layers, no forward(): just a field relaxing to a fixed point.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):  # skip demo during scored rerun
    m = om.Morph(4)                     # a 4-cell membrane
    events = m.stimulate([0, 1], [1.0, -0.5], budget=64)   # perturb cells 0 and 1, let it settle
    print('settled in', events, 'events ->', [round(c, 3) for c in m.cells()])

## 2. Learning by *growing*: `ValueHead`

`ValueHead` turns the organism into a function approximator `y ≈ f(x)`. But unlike a fixed-size network,
each `learn` call performs a local **grow** step: when the organism is surprised it *births new structure*,
then tunes it. So the model grows exactly the capacity the task needs. Watch it fit a nonlinearity and
report how many units it grew.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):  # skip demo during scored rerun
    import numpy as np
    head = om.ValueHead(in_dim=1, hidden=4, seed=0)
    for _ in range(3000):
        x = np.random.uniform(-1, 1, size=1)
        head.learn(x, np.tanh(1.5 * x[0]), lr=0.1)
    for xv in (-1.0, 0.0, 1.0):
        print(f'f({xv:+.1f}) pred={head.predict(np.array([xv])):+.3f}  true={np.tanh(1.5*xv):+.3f}')
    print('grown units:', head.n_units)

## 3. Acting with curiosity: `Actor`

`Actor` is a value-based RL agent: one grown `ValueHead` per action (`Q_a(x)`) plus an **intrinsic novelty**
drive that rewards visiting under-explored states — so it explores on its own. Everything is online: no
replay buffer, no target network, no backprop. The loop is just `act` then `learn`.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):  # skip demo during scored rerun
    import numpy as np
    actor = om.Actor(n_features=4, n_actions=3, beta=0.3, seed=0)
    state = np.zeros(4)
    a = actor.act(state, epsilon=0.1)          # choose
    err = actor.learn(state, a, reward=1.0, x_next=state, done=True)   # learn online
    print(f'chose action {a}, TD error {err:.3f}, total grown units {actor.n_units}')

## 4. The ARC-AGI-3 agent (kit RU: kit DG13 + rules kept across levels)

`explore` grows a model of **which actions change the world** and spends its budget on them. It has no authored
solver.
- **Logical grid.** It plays on a logical grid read from the frame.
- **Clock.** Cells that change in most transitions are learned as the clock; an action's effect is what changes
  outside it.
- **Effect model.** An RLS readout (a `Field`, `readout_rls`) learns from every transition how likely each action is
  to have an effect. Its inputs are the action's id and, for a click, the clicked object's colour and size.
- **Progress head.** A second readout learns from level completions, discounted back over 20 actions.
- **Routing.** It walks the known state graph to the nearest state with untried actions.

What is authored: the grid reading, the clock rule, the features, the 20-action credit window and the routing.
What is learned per game: the effect and progress readouts and the clock. It is the gpu-track branch's v3
(`a446b4d1`), shipped byte for byte; see `kaggle/arc-agi-3/kit-x/README.md`.

**Kit TK adds a planner** (`muse/think.py`, stream TK). Only actions are scored, so before each action it searches its
own models for the most expected discovery per action spent:
- **On the graph** (never wrong): the transitions it has seen, searched up to 16 actions ahead; known deaths avoided.
- **Off the graph**: an untried action's outcome is predicted only from rates verified on this game's own tries, per
  action type (new state, known state, nothing, death), and how often one action undoes another.
- **A life's budget**: when the longest lives keep ending at the same length, targets beyond it are skipped.
- **An arbiter**: each game checks whether v3's own effect head or the planner's verified rate predicts new states
  better, and the better one orders untried actions.
- **A blind reading**: when the logical grid shows nothing changing on a level, it plays on the raw frame instead,
  with v4's border line clock.

What is authored: the search, the outcome classes, the value form and the two fallback rules. What is learned per
game: every rate, the graph, the budget and v3's heads. Nothing names a game.

**Kit HV adds verified facts** (`facts.py`, `hy_arms.py`, stream HY). Before each action the morph predicts facts about
the next frame (does anything change, does a never-changed cell change, does the game end, does the self move), and
after the action it checks each one. Every check is a learning step for that fact's readout (a `Field`), and every
fact type keeps a running record of how often it was right, per confidence.
- The planner's risk for each move here is the death rate **verified** in that prediction's confidence bin over the
  game, not a guess per action type.
- A verified "nothing will change" means no new state; a verified "a new cell will change" means a new one.

What is authored: the fact list, their truth tests, the feature lists and the substitution rule. What is learned per
game: every readout and every rate. Nothing names a game.

**Kit BH reads what ticks** (`bs.py`, stream BS). A colour change that was a move's only change under two different
actions (a bar losing a pill whatever the morph does), or that came with every one of many different clicks and never
at the clicked cell, is a clock: the cells it changes are painted out of the state, so the planner's graph stops
minting a new state per tick.

**Kit BH adds clicks probed by effect** (`bs.py`, stream BS). The morph offers a click on every object it sees (not
only the first 32), learns which clicks change nothing (a click that changed nothing at a place on a colour, or on a
colour that changed nothing at three places and something nowhere), and holds those back until every other click of
the state was tried; two clicks that took one state to the same next state are one button.

What is authored: these rules. What is learned per game: which clicks are inert, which are one button. Nothing names a
game.

**Kit DG3 reads the budget bar** (`dg3.py`, stream DG3). Most games draw a bar that shortens by one cell on every
action (or every few) until the life ends. It changes together with whatever the action does, never alone, so the tick
test above misses it, and while it is part of the state every state is new: a move and its undo, a toggle pressed
twice, the same useless click look like discoveries. A colour change that, within one life, comes in small parts
touching no other change, never changes a cell twice, never comes with its inverse and stays on one row or column,
seen four times under two kinds of action (or eight under one), is a bar: its line is painted out of the state like a
tick.

What is authored: these conditions and their numbers. What is learned per game: which changes, which cells. Nothing
names a game.

**Kit DG13 looks where things touch** (`dg1.py`, stream DG1, over `dg3.py`). Most games ask for a correspondence with
something on screen: a piece onto its outline, pins onto pins, a block into a box. When a level has taken 300 actions
without falling, the morph weighs an untried move more where the objects touch in a way it has seldom seen this level
(the set of colour pairs that touch, counted per level), and less where it would only be one more position in open
space.

What is authored: the touching-pairs cell, the weight 1 + 1 / sqrt(n) and the 300 actions. What is learned per level:
the counts. Nothing names a game.

**Kit RU learns rules, not states** (`rules.py` and `ru.py`, stream RU, over `dg13.py`). From every step the morph
learns what an action does to the objects, keyed by what the self meets: the colour ahead, the colour beyond it,
whether either is the last of its colour. The effect is where the self went, what happened to the object ahead
(pushed, gone, recoloured), which colours changed elsewhere, and whether it died. Every rule predicts before it learns,
so each carries a verified rate. The rules are kept when a level ends; only the positions are forgotten.

At each level completion the morph also notes which conditions became true (the self entered a colour, a colour
reached zero, a colour was clicked); one that comes true without finishing a level is refuted. From level 2 on it
searches its own predicted screens for the shortest way to make the learned win true, follows that plan while each
prediction comes true, and plans again on a miss. With no plan, kit DG13 decides as before.

What is authored: the object reading, the relation key, the effect classes, the win vocabulary, the search and its
cap (2,000 predictions). What is learned per game: the rules and their rates, the self, the win. Nothing names a game.


In [ ]:
%%writefile /kaggle/working/muse/muse.py
# SPDX-License-Identifier: MIT-0
"""The muse claim-quality test: prompts from recorded stuck states, engine ground truth, scoring.

Pipeline (no network, no GPU):

    python muse.py build data/arcengine.jsonl data/bench.jsonl --out data/      # -> prompts.jsonl, gt.jsonl.gz
    python muse.py score data/ answers/*.jsonl                                   # -> the table per model config
    python muse.py selftest data/                                                # oracle and null answers

Inputs are episode records (`record_arcengine.py`, `muse_export.rs`): the 64x64 frames of a level, the actions and
events, and every candidate action PROBED from every state (event + next frame). Everything below is derived from
those transitions, generically, with no hand labels:

* GRID. The frame is reduced to its logical grid: the largest block size k (with offsets) whose k x k blocks are
  uniform in >= 90 % of cases over all the record's frames, each block its modal colour; constant border rows and
  columns of the letterbox colour (the frame's corner pixel) are cropped. This is the description a model sees; a click (x, y) is a logical cell.
* SELF. A colour "translates" in a transition when the objects of it that changed moved rigidly (other objects of
  that colour unchanged). The self is the colour that translates in the most non-terminal simple-action
  transitions of the level (>= 2); the self group adds colours that translate
  at least half as often and always with the self's shift.
* ACTION RULES (simple actions). Over the probes where the self translated: the dominant direction (sign of dx, dy),
  if it holds in >= 80 % of them (else `other`).
  An action that never moves the self is `none` when it never changes the grid, else `other` (not scored).
* GOAL. Candidate `reach(c)`: a move action completes the level iff the self, shifted one cell by the action's
  direction, overlaps a cell of colour c. A candidate is ground truth iff it predicts EVERY probe's completion event
  of the level exactly (F1 = 1, at least one completion). Levels no candidate explains have no claim-form goal truth
  (their goal is scored in the code form only).

Two answer forms (the same state, the same known facts):

* `claims`: JSON {self, actions, goal}. A fact the prompt already gives as verified is not scored (not novel);
  every withheld fact is scored correct / wrong / abstained. `novel_correct` = correct / withheld.
* `code`: Python `step(grid, action)` and `is_goal(grid)`, executed in a sandboxed subprocess over every probe of
  the level: step accuracy on the transitions shown in the prompt (what the morph can verify) and on the held-out
  probes, and the goal as `is_goal(step(grid, a))` against the engine's completion events.
"""
import argparse
import collections
import glob
import gzip
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import tempfile

import numpy as np

SIZE = 64


def opener(path, mode="rt"):
    """open() that reads and writes `.gz` transparently."""
    return gzip.open(path, mode) if path.endswith(".gz") else open(path, mode)
STATE_TIMES = (0, 10, 20, 39)
HISTORY = 6
MAX_DIFF_CELLS = 12

# --------------------------------------------------------------------------------------------------------- grids


def decode(s):
    a = np.array([int(ch, 16) for ch in s], dtype=np.int64)
    return a.reshape(SIZE, SIZE)


class Grid:
    """Frame -> logical grid for one record: block size k, offsets, crop."""

    def __init__(self, frames):
        # the largest block size whose cropped grid is at least 4 x 4 (a near-uniform frame can pass the uniformity
        # test at a large k and then crop to nothing)
        kmax = 8
        while True:
            self.k, self.ox, self.oy = self._block(frames, kmax)
            grids = [self._reduce(f) for f in frames]
            self.y0, self.y1, self.x0, self.x1 = self._crop(grids, int(frames[0][0, 0]))
            if (self.y1 - self.y0 >= 4 and self.x1 - self.x0 >= 4) or self.k == 1:
                break
            kmax = self.k - 1

    @staticmethod
    def _block(frames, kmax=8):
        for k in [k for k in (8, 7, 6, 5, 4, 3, 2) if k <= kmax]:
            best = None
            for oy in range(k):
                for ox in range(k):
                    uni = tot = 0
                    for f in frames:
                        sub = f[oy:oy + ((SIZE - oy) // k) * k, ox:ox + ((SIZE - ox) // k) * k]
                        h, w = sub.shape[0] // k, sub.shape[1] // k
                        if h == 0 or w == 0:
                            continue
                        b = sub.reshape(h, k, w, k).transpose(0, 2, 1, 3).reshape(h, w, k * k)
                        uni += int((b == b[:, :, :1]).all(axis=2).sum())
                        tot += h * w
                    if tot and uni / tot >= 0.9 and (best is None or uni / tot > best[0]):
                        best = (uni / tot, ox, oy)
            if best:
                return k, best[1], best[2]
        return 1, 0, 0

    def _reduce(self, f):
        k, ox, oy = self.k, self.ox, self.oy
        h, w = (SIZE - oy) // k, (SIZE - ox) // k
        sub = f[oy:oy + h * k, ox:ox + w * k]
        b = sub.reshape(h, k, w, k).transpose(0, 2, 1, 3).reshape(h, w, k * k)
        out = np.zeros((h, w), dtype=np.int64)
        for y in range(h):
            for x in range(w):
                out[y, x] = np.bincount(b[y, x], minlength=16).argmax()
        return out

    @staticmethod
    def _crop(grids, letterbox):
        """Drop constant border rows/columns of the letterbox colour (the frame's corner pixel); a border of any
        other colour (a maze's outer wall) is part of the game."""
        g0 = grids[0]
        h, w = g0.shape

        def const_row(y):
            v = g0[y, 0]
            return all((g[y, :] == v).all() and g[y, 0] == v for g in grids)

        def const_col(x):
            v = g0[0, x]
            return all((g[:, x] == v).all() and g[0, x] == v for g in grids)

        corner = letterbox
        y0, y1, x0, x1 = 0, h, 0, w
        while y0 < y1 - 1 and const_row(y0) and g0[y0, 0] == corner:
            y0 += 1
        while y1 - 1 > y0 and const_row(y1 - 1) and g0[y1 - 1, 0] == corner:
            y1 -= 1
        while x0 < x1 - 1 and const_col(x0) and g0[0, x0] == corner:
            x0 += 1
        while x1 - 1 > x0 and const_col(x1 - 1) and g0[0, x1 - 1] == corner:
            x1 -= 1
        return y0, y1, x0, x1

    def grid(self, f):
        return self._reduce(f)[self.y0:self.y1, self.x0:self.x1]

    def cell_of_pixel(self, px, py):
        return (px - self.ox) // self.k - self.x0, (py - self.oy) // self.k - self.y0


def grid_text(g):
    return "\n".join("".join("%x" % v for v in row) for row in g)


def diff_text(a, b):
    if a.shape != b.shape:
        return "the whole grid changed (new size %dx%d)" % (b.shape[1], b.shape[0])
    ys, xs = np.nonzero(a != b)
    if len(ys) == 0:
        return "no change"
    parts = ["(%d,%d) %x->%x" % (x, y, a[y, x], b[y, x]) for y, x in zip(ys.tolist(), xs.tolist())]
    more = len(parts) - MAX_DIFF_CELLS
    return ", ".join(parts[:MAX_DIFF_CELLS]) + (f", ... {more} more cells" if more > 0 else "")


# --------------------------------------------------------------------------------------------------- truth


def _components(mask):
    """4-connected components of a boolean mask, as lists of (y, x)."""
    seen = np.zeros(mask.shape, bool)
    comps = []
    for y0, x0 in np.argwhere(mask).tolist():
        if seen[y0, x0]:
            continue
        stack, comp = [(y0, x0)], []
        seen[y0, x0] = True
        while stack:
            y, x = stack.pop()
            comp.append((y, x))
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < mask.shape[0] and 0 <= xx < mask.shape[1] and mask[yy, xx] and not seen[yy, xx]:
                    seen[yy, xx] = True
                    stack.append((yy, xx))
        comps.append(comp)
    return comps


def translation(a, b, c):
    """The (dx, dy) by which the objects of colour c that changed moved rigidly from a to b (the rest of colour c
    unchanged), else None. Other objects of the same colour may stay put."""
    if a.shape != b.shape:
        return None
    A, B = a == c, b == c
    gone, new = A & ~B, B & ~A
    if not gone.any() or not new.any():
        return None
    moved = [comp for comp in _components(A) if any(gone[y, x] for y, x in comp)]
    cells = [p for comp in moved for p in comp]
    gy, gx = np.argwhere(gone)[0]
    h, w = a.shape
    for ny, nx in np.argwhere(new).tolist():
        for py, px in cells:
            if (py, px) != (gy, gx):
                continue
            dy, dx = ny - py, nx - px
            if dy == 0 and dx == 0:
                continue
            exp = A.copy()
            for y, x in cells:
                exp[y, x] = False
            ok = True
            for y, x in cells:
                yy, xx = y + dy, x + dx
                if not (0 <= yy < h and 0 <= xx < w):
                    ok = False
                    break
                exp[yy, xx] = True
            if ok and (exp == B).all():
                return int(dx), int(dy)
    return None


def sign(v):
    return (v > 0) - (v < 0)


def action_token(tok, G):
    """Engine token -> (kind, logical): '3' -> (3, '3'); '6@px,py' -> (6, '6 x y')."""
    if tok.startswith("6@"):
        px, py = map(int, tok[2:].split(","))
        x, y = G.cell_of_pixel(px, py)
        return 6, f"6 {x} {y}", (x, y)
    return int(tok), tok, None


def level_truth(recs):
    """Ground truth of one (game, seed, level) from its episodes (walk + demo)."""
    frames = [decode(s) for r in recs for s in r["pool"]]
    G = Grid(frames)
    trans = []  # (pre grid, token, event, post grid)
    for r in recs:
        gs = [G.grid(decode(s)) for s in r["pool"]]
        for t, probes in enumerate(r["probes"]):
            pre = gs[r["frames"][t]]
            for tok, (ev, fi) in probes.items():
                trans.append((pre, tok, ev, gs[fi]))
    colours = sorted({int(v) for g in (G.grid(f) for f in frames) for v in np.unique(g)})
    tcount, tshift = collections.Counter(), collections.defaultdict(list)
    for pre, tok, ev, post in trans:
        if ev != "none" or tok == "0" or tok.startswith("6@"):
            continue  # the self is what the simple actions move
        for c in colours:
            d = translation(pre, post, c)
            if d:
                tcount[c] += 1
                tshift[c].append((id(pre), tok, d))
    truth = dict(block=G.k, dims=None, self=None, self_group=[], rules={}, goal=None, goal_candidates=[])
    if tcount and tcount.most_common(1)[0][1] >= 2:
        me = tcount.most_common(1)[0][0]
        truth["self"] = me
        mine = {(p, t): d for p, t, d in tshift[me]}
        group = [me]
        for c, n in tcount.items():
            if c != me and n >= tcount[me] / 2:
                if all(mine.get((p, t)) == d for p, t, d in tshift[c]):
                    group.append(c)
        truth["self_group"] = sorted(group)
        # action rules
        per = collections.defaultdict(collections.Counter)
        changed = collections.Counter()
        seen_act = set()
        for pre, tok, ev, post in trans:
            if tok.startswith("6@") or tok == "0":
                continue
            a = int(tok)
            seen_act.add(a)
            if ev == "none":
                d = translation(pre, post, me)
                if d:
                    per[a][(sign(d[0]), sign(d[1]))] += 1
                elif pre.shape != post.shape or (pre != post).any():
                    changed[a] += 1
        for a in sorted(seen_act):
            if per[a]:
                (dx, dy), k = per[a].most_common(1)[0]
                if k >= 0.8 * sum(per[a].values()):
                    truth["rules"][str(a)] = {"moves": me, "dx": dx, "dy": dy}
                else:  # no consistent direction (e.g. a jump to a fixed place): not a scorable rule
                    truth["rules"][str(a)] = {"effect": "other"}
            elif changed[a]:
                truth["rules"][str(a)] = {"effect": "other"}
            else:
                truth["rules"][str(a)] = {"effect": "none"}
        # goal: reach(c)
        group_set = set(group)
        for c in colours:
            if c in group_set:
                continue
            tp = fp = fn = 0
            for pre, tok, ev, post in trans:
                if ev == "death" or tok == "0":
                    continue
                pred = False
                rule = truth["rules"].get(tok)
                if rule and "moves" in rule:
                    cells = np.argwhere(np.isin(pre, list(group_set)))
                    for y, x in cells.tolist():
                        yy, xx = y + rule["dy"], x + rule["dx"]
                        if 0 <= yy < pre.shape[0] and 0 <= xx < pre.shape[1] and pre[yy, xx] == c:
                            pred = True
                            break
                real = ev == "complete"
                tp += pred and real
                fp += pred and not real
                fn += real and not pred
            if tp and not fp and not fn:
                truth["goal_candidates"].append(c)
        if truth["goal_candidates"]:
            truth["goal"] = {"kind": "reach", "targets": truth["goal_candidates"]}
    truth["completions"] = sum(ev == "complete" for _, _, ev, _ in trans)
    return G, trans, truth


# -------------------------------------------------------------------------------------------------- prompts

SYSTEM = (
    "You are the muse of a game-playing agent. The agent plays an unknown grid game with no instructions. "
    "It records every transition, and it tests every claim you make against them before acting on it, so a "
    "wrong claim costs little but a vague one is useless. Be concrete."
)

GRID_HELP = (
    "The grid is {w} wide and {h} high. Each cell is a colour 0-15 written as one hex digit; row 0 is the top, "
    "x grows to the right, y grows downward."
)


def actions_text(avail):
    names = []
    for a in avail:
        if a == 6:
            names.append("ACTION6 x y (click the cell at x, y)")
        elif 1 <= a <= 7:
            names.append(f"ACTION{a}")
    return "Available actions: " + ", ".join(names) + ". RESET restarts the level."


def objects_text(g):
    """The grid as objects: the most common colour, then every 4-connected one-colour component, grouped when a
    colour has several objects of the same shape. Generic: no colour or shape is named."""
    vals, counts = np.unique(g, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    h, w = g.shape
    lines = [f"Most common colour: {bg} ({int(counts.max())} of {h * w} cells), not listed below. Objects:"]
    for c in [int(v) for v in vals if v != bg]:
        comps = _components(g == c)
        shapes = collections.defaultdict(list)
        for comp in comps:
            ys, xs = [p[0] for p in comp], [p[1] for p in comp]
            x0, y0 = min(xs), min(ys)
            shapes[tuple(sorted((y - y0, x - x0) for y, x in comp))].append((x0, y0, max(xs), max(ys), len(comp)))
        for shape, objs in sorted(shapes.items(), key=lambda kv: (-len(kv[1]), kv[1][0])):
            n = objs[0][4]
            bw, bh = objs[0][2] - objs[0][0] + 1, objs[0][3] - objs[0][1] + 1
            if n == 1:
                desc = "1 cell"
            elif n == bw * bh:
                desc = f"a solid {bw}x{bh} block"
            elif n <= 8:
                desc = f"{n} cells, shape " + " ".join(f"({x},{y})" for y, x in shape) + " from its top-left"
            else:
                desc = f"{n} cells spanning {bw}x{bh} (a large irregular shape: walls, a frame or a region)"
            where = ", ".join(f"({x0},{y0})" for x0, y0, _, _, _ in objs[:12]) + (" ..." if len(objs) > 12 else "")
            if len(objs) == 1:
                lines.append(f"- colour {c}: one object, {desc}, top-left at {where}")
            else:
                lines.append(f"- colour {c}: {len(objs)} objects, each {desc}, top-left at {where}")
    return "\n".join(lines)


def object_diff_text(a, b):
    """What changed between two grids, per colour: a rigid move of its changed objects, or cells (dis)appearing.
    The most common colour's changes (the ground filling in behind a moving object) are reported only when nothing
    else changed."""
    if a.shape != b.shape:
        return "the whole grid changed (new size %dx%d)" % (b.shape[1], b.shape[0])
    if (a == b).all():
        return "no change"
    vals, counts = np.unique(a, return_counts=True)
    common = int(vals[np.argmax(counts)])
    changed = sorted({int(v) for v in np.unique(np.concatenate([a[a != b], b[a != b]]))})
    if len(changed) > 1:
        changed = [c for c in changed if c != common]
    parts = []
    for c in changed:
        d = translation(a, b, c)
        if d:
            parts.append(f"colour {c} moved by dx={d[0]}, dy={d[1]}")
            continue
        gone, new = int(((a == c) & (b != c)).sum()), int(((b == c) & (a != c)).sum())
        bits = []
        if new:
            bits.append(f"{new} cell{'s' if new > 1 else ''} appeared")
        if gone:
            bits.append(f"{gone} cell{'s' if gone > 1 else ''} disappeared")
        parts.append(f"colour {c}: " + " and ".join(bits))
    return "; ".join(parts)


def history_text(hist, style="grid"):
    if not hist:
        return "Recent transitions: none yet (this is the level's first frame)."
    lines = ["Recent transitions, oldest first (cell changes as (x,y) old->new):" if style == "grid"
             else "Recent transitions, oldest first (what each colour's objects did):"]
    for i, (act, ev, d) in enumerate(hist, 1):
        if act == "0":
            name = "RESET"
        elif act == "outside":
            name = "ACTION6 outside the grid"
        else:
            name = "ACTION" + act if " " not in act else "ACTION6 " + act[2:]
        tail = {"complete": " -> the level was COMPLETED", "death": " -> the agent DIED (game over)"}.get(ev, "")
        lines.append(f"{i}. {name}: {d}{tail}")
    return "\n".join(lines)


def known_text(known, truth):
    facts = []
    if "self" in known:
        facts.append(f"the agent's own piece (the self) is colour {truth['self']:x}")
    for a in sorted(k for k in known if k != "self"):
        r = truth["rules"][a]
        if "moves" in r:
            facts.append(f"ACTION{a} moves the self by dx={r['dx']}, dy={r['dy']} (when not blocked)")
        else:
            facts.append(f"ACTION{a} does nothing")
    if not facts:
        return "Already verified: nothing yet."
    return "Already verified (do not repeat these): " + "; ".join(facts) + "."


CLAIMS_ASK = """The agent has not completed this level and has no idea what to do. Reply with ONE JSON object and nothing else:
{{"self": <the colour of the piece the agent controls, as an integer 0-15>,
 "actions": {{"<action number>": {{"moves": <colour>, "dx": <-1|0|1>, "dy": <-1|0|1>}} or {{"effect": "none"}} or {{"effect": "other", "note": "<short>"}}}},
 "goal": {{"kind": "reach", "target": <colour the self must touch to complete the level>}} or {{"kind": "other", "note": "<short>"}}}}
Give "actions" for every simple action ({simple}) that is not already verified. Colours are integers (hex b = 11)."""

CODE_ASK = """The agent has not completed this level and has no idea what to do. Write the game's rules as two Python functions:

def step(grid, action):
    # grid: list of rows, each a list of ints 0-15 (grid[y][x]); action: an int ({simple}) or a tuple (6, x, y) for a click.
    # Return the grid right after the action, within the same level.

def is_goal(grid):
    # Return True if this grid means the level is complete.

Standard library only, no input/output. Reply with exactly one ```python code block."""


def prompt_messages(form, G, grid, avail, hist, known, truth, style="grid"):
    """`grid` style: the hex grid and raw cell diffs. `objects` style: the objects and what each colour did (the
    code form keeps the grid too, since its functions take one)."""
    simple = ", ".join(str(a) for a in avail if a != 6) or "none"
    h, w = grid.shape
    if style == "grid":
        scene = "Current grid:\n" + grid_text(grid)
    elif form == "claims":
        scene = "Current scene:\n" + objects_text(grid)
    else:
        scene = "Current scene:\n" + objects_text(grid) + "\n\nThe same, as the grid:\n" + grid_text(grid)
    body = "\n\n".join([
        GRID_HELP.format(w=w, h=h),
        actions_text(avail),
        scene,
        history_text(hist, style),
        known_text(known, truth),
        (CLAIMS_ASK if form == "claims" else CODE_ASK).format(simple=simple),
    ])
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": body}]


def avail_list(rec, t):
    if "available" in rec:
        return list(rec["available"][t])
    m = rec["available_mask"][t]
    return [i for i in range(1, 8) if m & (1 << i)]


def build(paths, out_dir, style="grid"):
    recs = [json.loads(l) for p in paths for l in opener(p) if l.strip()]
    groups = collections.defaultdict(list)
    for r in recs:
        groups[(r["source"], r["set"], r["game"], r["seed"], r["level"])].append(r)
    os.makedirs(out_dir, exist_ok=True)
    np_ = gt_ = 0
    with open(os.path.join(out_dir, "prompts.jsonl"), "w") as fp, \
            opener(os.path.join(out_dir, "gt.jsonl.gz"), "wt") as fg, \
            open(os.path.join(out_dir, "levels.jsonl"), "w") as fl:
        for key, rs in sorted(groups.items()):
            src, set_, game, seed, level = key
            G, trans, truth = level_truth(rs)
            lid = f"{src}:{game}:s{seed}:L{level}"
            fl.write(json.dumps(dict(level_id=lid, source=src, set=set_, game=game, seed=seed, level=level,
                                     block=G.k, truth=truth)) + "\n")
            walk = next((r for r in rs if r["kind"] == "walk"), None)
            if walk is None:
                continue
            gs = [G.grid(decode(s)) for s in walk["pool"]]
            # every probe of the level, as logical cases, for the code form
            cases = []
            for r in rs:
                rg = [G.grid(decode(s)) for s in r["pool"]]
                for t, probes in enumerate(r["probes"]):
                    for tok, (ev, fi) in probes.items():
                        _, logical, xy = action_token(tok, G)
                        g0 = rg[r["frames"][t]]
                        if xy and not (0 <= xy[0] < g0.shape[1] and 0 <= xy[1] < g0.shape[0]):
                            continue  # a click on the letterbox
                        cases.append(dict(ep=r["kind"], t=t, grid=rg[r["frames"][t]].tolist(), action=logical,
                                          event=ev, post=rg[fi].tolist()))
            T = len(walk["actions"])
            for si, t in enumerate(sorted({min(x, T - 1) for x in STATE_TIMES})):
                if not walk["probes"][t]:
                    continue  # game over: only RESET
                grid = gs[walk["frames"][t]]
                hist, shown = [], []
                for u in range(max(0, t - HISTORY), t):
                    tok = walk["actions"][u]
                    a, b = gs[walk["frames"][u]], gs[walk["frames"][u + 1]]
                    _, logical, xy = action_token(tok, G) if tok != "0" else (0, "0", None)
                    if xy and not (0 <= xy[0] < a.shape[1] and 0 <= xy[1] < a.shape[0]):
                        logical = "outside"
                    hist.append((logical, walk["events"][u], diff_text(a, b) if style == "grid" else object_diff_text(a, b)))
                    shown.append(dict(t=u, action=logical))
                avail = avail_list(walk, t)
                rng = random.Random(f"{lid}:{t}")
                variants = [("none", set())]
                if truth["self"] is not None and si % 2 == 1:  # every other state also asks with some facts known
                    scored = [a for a in truth["rules"] if truth["rules"][a].get("effect") != "other"]
                    half = set(rng.sample(scored, len(scored) // 2)) if scored else set()
                    variants.append(("some", {"self"} | half))
                for vname, known in variants:
                    for form in ("claims", "code"):
                        pid = f"{lid}:t{t}:{vname}:{form}" + ("" if style == "grid" else f":{style}")
                        msgs = prompt_messages(form, G, grid, avail, hist, known, truth, style)
                        fp.write(json.dumps(dict(prompt_id=pid, level_id=lid, form=form, known=vname,
                                                 messages=msgs)) + "\n")
                        fg.write(json.dumps(dict(prompt_id=pid, level_id=lid, form=form, t=t,
                                                 known=sorted(known), truth=truth, shown=shown,
                                                 cases=cases if form == "code" else None)) + "\n")
                        np_ += 1
            gt_ += 1
    print(f"{gt_} levels, {np_} prompts -> {out_dir}", file=sys.stderr)


# ---------------------------------------------------------------------------------------------------- scoring


def parse_json(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S)
    cand = [m.group(1)] if m else []
    i = text.find("{")
    if i >= 0:
        depth = 0
        for j in range(i, len(text)):
            depth += text[j] == "{"
            depth -= text[j] == "}"
            if depth == 0:
                cand.append(text[i:j + 1])
                break
    for c in cand:
        try:
            return json.loads(c)
        except Exception:
            continue
    return None


def as_colour(v):
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return v
    if isinstance(v, str):
        s = v.strip().lower()
        try:
            return int(s, 16) if len(s) == 1 else int(s)
        except ValueError:
            return None
    return None


def score_claims(ans, g):
    truth, known = g["truth"], set(g["known"])
    out = collections.Counter()
    obj = parse_json(ans)
    out["parsed"] = obj is not None and isinstance(obj, dict)
    obj = obj if isinstance(obj, dict) else {}
    # (c) self
    if truth["self"] is not None and "self" not in known:
        out["self_n"] += 1
        c = as_colour(obj.get("self"))
        if c is None:
            out["self_abstain"] += 1
        elif c in truth["self_group"]:
            out["self_ok"] += 1
        else:
            out["self_wrong"] += 1
    # (a) actions
    acts = obj.get("actions") if isinstance(obj.get("actions"), dict) else {}
    acts = {str(k).replace("ACTION", ""): v for k, v in acts.items()}
    for a, r in truth["rules"].items():
        if r.get("effect") == "other":
            continue
        if a in known:
            if a in acts:
                out["act_repeat"] += 1
            continue
        out["act_n"] += 1
        claim = acts.get(a)
        if not isinstance(claim, dict) or claim.get("effect") == "other":
            out["act_abstain"] += 1
            continue
        if "moves" in r:
            ok = (as_colour(claim.get("moves")) in truth["self_group"]
                  and sign(int(claim.get("dx", 0) or 0)) == r["dx"] and sign(int(claim.get("dy", 0) or 0)) == r["dy"])
        else:
            ok = claim.get("effect") == "none"
        out["act_ok" if ok else "act_wrong"] += 1
    # (b) goal
    if truth["goal"]:
        out["goal_n"] += 1
        goal = obj.get("goal") if isinstance(obj.get("goal"), dict) else {}
        if goal.get("kind") != "reach":
            out["goal_abstain"] += 1
        elif as_colour(goal.get("target")) in truth["goal"]["targets"]:
            out["goal_ok"] += 1
        else:
            out["goal_wrong"] += 1
    return out


RUNNER = r'''
import json, sys, resource
resource.setrlimit(resource.RLIMIT_AS, (1 << 30, 1 << 30))
src = sys.stdin.readline()
cases = json.loads(sys.stdin.readline())
out = []
try:
    ns = {}
    exec(compile(json.loads(src), "<muse>", "exec"), ns)
    step, is_goal = ns.get("step"), ns.get("is_goal")
except Exception as e:
    print(json.dumps({"error": "compile: %s" % e})); sys.exit()
for c in cases:
    a = c["action"]
    act = tuple(int(v) for v in a.split()) if " " in a else int(a)
    r = {}
    try:
        g = [list(row) for row in c["grid"]]
        post = step(g, act)
        r["post"] = [list(map(int, row)) for row in post]
    except Exception as e:
        r["err"] = type(e).__name__
    try:
        r["goal"] = bool(is_goal(r["post"])) if "post" in r else None
    except Exception as e:
        r["goal"] = None
    out.append(r)
print(json.dumps({"results": out}))
'''


def extract_code(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S)
    code = max(blocks, key=len) if blocks else (text if "def step" in text else None)
    return code


def run_code(code, cases, timeout=20):
    with tempfile.TemporaryDirectory() as d:
        runner = os.path.join(d, "run.py")
        open(runner, "w").write(RUNNER)
        payload = json.dumps(code) + "\n" + json.dumps([dict(grid=c["grid"], action=c["action"]) for c in cases]) + "\n"
        try:
            p = subprocess.run([sys.executable, "-I", "-S", runner], input=payload, capture_output=True, text=True,
                               timeout=timeout, cwd=d, env={})
        except subprocess.TimeoutExpired:
            return {"error": "timeout"}
        try:
            return json.loads(p.stdout.strip().splitlines()[-1])
        except Exception:
            return {"error": "crash: " + p.stderr[-200:]}


def score_code(ans, g):
    out = collections.Counter()
    code = extract_code(ans)
    out["parsed"] = bool(code and "def step" in code)
    if not out["parsed"]:
        return out
    cases = g["cases"]
    res = run_code(code, cases)
    if "error" in res:
        out["exec_error"] = 1
        return out
    out["ran"] = 1
    shown = {(s["t"], s["action"]) for s in g["shown"]}
    for c, r in zip(cases, res["results"]):
        if c["event"] == "death":
            continue
        is_shown = c["ep"] == "walk" and (c["t"], c["action"]) in shown
        if c["event"] == "none":
            ok = r.get("post") == c["post"]
            key = "seen" if is_shown else "held"
            out[f"step_{key}_n"] += 1
            out[f"step_{key}_ok"] += ok
            if not is_shown and c["post"] != c["grid"]:
                out["step_moved_n"] += 1
                out["step_moved_ok"] += ok
        real = c["event"] == "complete"
        pred = bool(r.get("goal"))
        out["goal_tp"] += pred and real
        out["goal_fp"] += pred and not real
        out["goal_fn"] += real and not pred
    out["verified"] = int(out["step_seen_n"] == out["step_seen_ok"])
    return out


def load_jsonl(path):
    return [json.loads(l) for l in opener(path) if l.strip()]


def score(data_dir, answer_paths):
    gt = {g["prompt_id"]: g for g in load_jsonl(os.path.join(data_dir, "gt.jsonl.gz"))}
    tallies = collections.defaultdict(collections.Counter)
    n = collections.Counter()
    rows = []
    for p in answer_paths:
        for a in load_jsonl(p):
            g = gt.get(a["prompt_id"])
            if g is None or a.get("error"):
                continue
            cfg = a["config"]
            s = score_claims(a["answer"], g) if g["form"] == "claims" else score_code(a["answer"], g)
            tallies[(cfg, g["form"])].update(s)
            n[(cfg, g["form"])] += 1
            n[(cfg, "tokens")] += a.get("usage", {}).get("completion_tokens", 0) or 0
            n[(cfg, "latency")] += a.get("latency_s", 0) or 0
            rows.append(dict(prompt_id=a["prompt_id"], config=cfg, form=g["form"], **{k: int(v) for k, v in s.items()}))
    cfgs = sorted({c for c, _ in tallies})
    pct = lambda a, b: f"{100 * a / b:5.1f}" if b else "    -"
    print("CLAIMS FORM (novel = withheld facts only; ok / wrong / abstain, % of withheld)")
    print(f"{'config':34} {'n':>4} {'parse':>6} | {'self ok':>7} {'wrong':>6} | {'act ok':>6} {'wrong':>6} | {'goal ok':>7} {'wrong':>6} | {'tok/ans':>7} {'s/ans':>6}")
    for c in cfgs:
        t, k = tallies[(c, "claims")], n[(c, "claims")]
        if not k:
            continue
        tot = n[(c, "claims")] + n[(c, "code")]
        print(f"{c:34} {k:4d} {pct(t['parsed'], k)} | {pct(t['self_ok'], t['self_n'])}   {pct(t['self_wrong'], t['self_n'])} |"
              f" {pct(t['act_ok'], t['act_n'])} {pct(t['act_wrong'], t['act_n'])} | {pct(t['goal_ok'], t['goal_n'])}  "
              f" {pct(t['goal_wrong'], t['goal_n'])} | {n[(c, 'tokens')] / tot:7.0f} {n[(c, 'latency')] / tot:6.1f}")
    print("\nCODE FORM (step = exact next grid on held-out probes, all / only those that change the grid; "
          "goal = is_goal(step(grid, a)) vs the engine's completions)")
    print(f"{'config':34} {'n':>4} {'parse':>6} {'ran':>6} | {'seen ok':>7} {'held ok':>7} {'changed':>7} {'verified':>8} | {'goal P':>6} {'goal R':>6}")
    for c in cfgs:
        t, k = tallies[(c, "code")], n[(c, "code")]
        if not k:
            continue
        print(f"{c:34} {k:4d} {pct(t['parsed'], k)} {pct(t['ran'], k)} | {pct(t['step_seen_ok'], t['step_seen_n'])}  "
              f" {pct(t['step_held_ok'], t['step_held_n'])}   {pct(t['step_moved_ok'], t['step_moved_n'])}    {pct(t['verified'], k)} |"
              f" {pct(t['goal_tp'], t['goal_tp'] + t['goal_fp'])} {pct(t['goal_tp'], t['goal_tp'] + t['goal_fn'])}")
    return rows


# ------------------------------------------------------------------------------------------------- selftest


def oracle_claims(g):
    t = g["truth"]
    return json.dumps({"self": t["self"], "actions": {a: r for a, r in t["rules"].items()},
                       "goal": {"kind": "reach", "target": t["goal"]["targets"][0]} if t["goal"] else {"kind": "other"}})


def selftest(data_dir):
    """Oracle claims must score 100 % where truth exists; a null answer 0 %; a trivial step (identity) shows the
    floor of the code form (unchanged grids are common)."""
    gts = load_jsonl(os.path.join(data_dir, "gt.jsonl.gz"))
    tmp = os.path.join(data_dir, "selftest_answers.jsonl")
    ident = "```python\ndef step(grid, action):\n    return grid\n\ndef is_goal(grid):\n    return False\n```"
    with open(tmp, "w") as f:
        for g in gts:
            if g["form"] == "claims":
                f.write(json.dumps(dict(prompt_id=g["prompt_id"], config="oracle", answer=oracle_claims(g))) + "\n")
                f.write(json.dumps(dict(prompt_id=g["prompt_id"], config="null", answer="I don't know.")) + "\n")
            else:
                f.write(json.dumps(dict(prompt_id=g["prompt_id"], config="identity", answer=ident)) + "\n")
    score(data_dir, [tmp])
    levels = load_jsonl(os.path.join(data_dir, "levels.jsonl"))
    print("\nLEVEL TRUTH")
    for l in levels:
        t = l["truth"]
        print(f"{l['level_id']:42} block {l['block']}  self {t['self']} group {t['self_group']}  "
              f"rules {' '.join(f'{a}:' + (('%+d%+d' % (r['dx'], r['dy'])) if 'moves' in r else r['effect']) for a, r in t['rules'].items())}"
              f"  goal {t['goal']['targets'] if t['goal'] else None}  completions {t['completions']}")


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    b = sub.add_parser("build")
    b.add_argument("records", nargs="+")
    b.add_argument("--out", required=True)
    b.add_argument("--style", default="grid", choices=["grid", "objects"])
    s = sub.add_parser("score")
    s.add_argument("data")
    s.add_argument("answers", nargs="+")
    s.add_argument("--rows", help="write per-answer scores as JSONL")
    t = sub.add_parser("selftest")
    t.add_argument("data")
    args = ap.parse_args()
    if args.cmd == "build":
        build(args.records, args.out, args.style)
    elif args.cmd == "score":
        rows = score(args.data, sorted({p for a in args.answers for p in glob.glob(a)}))
        if args.rows:
            with open(args.rows, "w") as f:
                for r in rows:
                    f.write(json.dumps(r) + "\n")
    else:
        selftest(args.data)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/muse/ask.py
# SPDX-License-Identifier: MIT-0
"""Ask every prompt of the muse test to one or more OpenAI-compatible chat servers; log every answer to JSONL.

Standard library only. Works against the RunPod pod (`pod.py`), a local vLLM / SGLang / llama.cpp server, or the
owner's Mac. Every request and answer is appended to the log as it completes, so a run can be killed and resumed
(prompts already answered for a config are skipped) and scored again for free (`muse.py score`).

    python ask.py data/prompts.jsonl --configs configs.json --out answers/run.jsonl [--limit 20] [--workers 8]

configs.json: a list of {"config": name, "base_url": "http://host:port/v1", "model": served model name,
"max_tokens": int, "temperature": float, "extra": {...merged into the request body...}}. `base_url` may use
`{pod}` (replaced by --pod).
"""
import argparse
import json
import os
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed


def stream(url, body, timeout):
    """POST a streaming chat request; return (content, reasoning, finish_reason, usage). Streaming keeps bytes
    flowing, so a long (thinking) answer is not cut by an HTTP proxy's idle timeout (RunPod's is about 100 s)."""
    body = dict(body, stream=True, stream_options={"include_usage": True})
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json",
                                                                         "User-Agent": "openmorph-muse/1.0"})
    content, reasoning, finish, usage = [], [], None, {}
    with urllib.request.urlopen(req, timeout=timeout) as r:
        for raw in r:
            line = raw.decode("utf-8", "replace").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            ev = json.loads(data)
            if ev.get("usage"):
                usage = ev["usage"]
            for ch in ev.get("choices", []):
                d = ch.get("delta", {})
                if d.get("content"):
                    content.append(d["content"])
                if d.get("reasoning_content"):
                    reasoning.append(d["reasoning_content"])
                finish = ch.get("finish_reason") or finish
    return "".join(content), ("".join(reasoning) or None), finish, usage


def ask_one(cfg, p, timeout, retries=3):
    body = dict(model=cfg["model"], messages=p["messages"], max_tokens=cfg.get("max_tokens", 1024),
                temperature=cfg.get("temperature", 0.0), seed=cfg.get("seed", 0))
    body.update(cfg.get("extra", {}))
    url = cfg["base_url"].rstrip("/") + "/chat/completions"
    err = None
    for attempt in range(retries):
        t0 = time.time()
        try:
            content, reasoning, finish, usage = stream(url, body, timeout)
            return dict(prompt_id=p["prompt_id"], config=cfg["config"], model=cfg["model"],
                        request={k: v for k, v in body.items() if k != "messages"}, answer=content,
                        reasoning=reasoning, finish=finish, usage=usage, latency_s=round(time.time() - t0, 3),
                        ts=time.time())
        except (urllib.error.URLError, TimeoutError, ConnectionError, json.JSONDecodeError, OSError) as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
    return dict(prompt_id=p["prompt_id"], config=cfg["config"], model=cfg["model"], error=err, ts=time.time())


def run(prompts, configs, out, workers=8, timeout=300, limit=None, on_answer=None):
    done = set()
    if os.path.exists(out):
        for line in open(out):
            try:
                a = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not a.get("error"):
                done.add((a["config"], a["prompt_id"]))
    jobs = [(c, p) for c in configs for p in (prompts[:limit] if limit else prompts)
            if (c["config"], p["prompt_id"]) not in done]
    lock = threading.Lock()
    n_ok = n_err = 0
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    with open(out, "a") as f, ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(ask_one, c, p, timeout) for c, p in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            a = fut.result()
            with lock:
                f.write(json.dumps(a) + "\n")
                f.flush()
                if a.get("error"):
                    n_err += 1
                else:
                    n_ok += 1
            if on_answer:
                on_answer(a)
            if i % 25 == 0 or i == len(futs):
                print(f"  {i}/{len(futs)} answered ({n_err} errors)", flush=True)
    return n_ok, n_err


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("prompts")
    ap.add_argument("--configs", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--pod", default="")
    ap.add_argument("--only", help="comma-separated config names")
    ap.add_argument("--limit", type=int)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--timeout", type=float, default=300)
    args = ap.parse_args()
    prompts = [json.loads(l) for l in open(args.prompts) if l.strip()]
    configs = json.load(open(args.configs))
    for c in configs:
        c["base_url"] = c["base_url"].replace("{pod}", args.pod)
    if args.only:
        keep = set(args.only.split(","))
        configs = [c for c in configs if c["config"] in keep]
    ok, err = run(prompts, configs, args.out, args.workers, args.timeout, args.limit)
    print(f"{ok} answers, {err} errors -> {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/muse/play.py
# SPDX-License-Identifier: MIT-0
"""The muse in the loop: an LLM writes the game's rules as code, a planner searches them, reality corrects them.

    python play.py ENV --base-url http://host:port/v1 --model NAME [--games a,b] [--jobs 8] --out runs/x.json

Plays real `arcengine` games through the real-engine gate (`../gate/replay.py`: the real `arc_agi` scorecard,
competition resets, MAX_ACTIONS 1000), with the agent `muse`:

1. probe: every simple action once (and clicks on object centres) from the level's first frames;
2. model: the LLM writes `step(grid, action)` and `is_goal(grid)` from the object scene, the grid and the recorded
   transitions of this level (`muse.py` describes them);
3. verify: `step` is run (sandboxed) on every recorded transition; mismatches go back for a revision (2 rounds);
4. plan: breadth-first search over the LLM's `step` (in the sandbox) to a grid where `is_goal` holds;
5. act: the plan is executed; each real grid is compared with the predicted one; the first surprise ends the plan,
   records the transition and asks for a revision;
6. no plan, or the LLM budget spent: explore an action not yet tried in the current state.

Every prompt and answer is appended to `--log` (JSONL). The LLM is any OpenAI-compatible server (`ask.py`).
"""
import argparse
import collections
import hashlib
import json
import multiprocessing
import os
import random
import subprocess
import sys
import tempfile
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path[:0] = [HERE, os.path.join(HERE, "..", "gate")]
import ask  # noqa: E402
import muse  # noqa: E402

SYSTEM = ("You are the muse of a game-playing agent. The agent plays an unknown grid game with no instructions. "
          "You write the game's rules as Python code; the agent checks your code against everything it has seen and "
          "plans with it, so exact code matters more than prose.")

ASK = """Write the game's rules as two Python functions:

def step(grid, action):
    # grid: list of rows, each a list of ints 0-15 (grid[y][x]); action: an int ({simple}) or a tuple (6, x, y)
    # for a click on cell (x, y). Return the grid right after the action, within the same level.

def is_goal(grid):
    # Return True if this grid means the level is complete. The agent will search for actions that reach it.

Standard library only, no input/output. Reply with exactly one ```python code block."""

RUNNER = r'''
import json, sys, resource, collections, time
resource.setrlimit(resource.RLIMIT_AS, (1 << 30, 1 << 30))
req = json.loads(sys.stdin.read())
ns = {}
try:
    exec(compile(req["code"], "<muse>", "exec"), ns)
    step, is_goal = ns["step"], ns["is_goal"]
except Exception as e:
    print(json.dumps({"error": "compile: %s: %s" % (type(e).__name__, e)})); sys.exit()
def act(a):
    return tuple(int(v) for v in a.split()) if " " in a else int(a)
def norm(g):
    return [list(map(int, r)) for r in g]
out = {}
if req["mode"] == "verify":
    res = []
    for c in req["cases"]:
        try:
            res.append(norm(step([list(r) for r in c["grid"]], act(c["action"]))))
        except Exception as e:
            res.append("%s: %s" % (type(e).__name__, e))
    out["post"] = res
else:  # plan: BFS over the model to is_goal
    t0 = time.time()
    start = norm(req["grid"])
    key = lambda g: json.dumps(g)
    try:
        done0 = bool(is_goal(start))
    except Exception:
        done0 = False
    seen = {key(start)}
    frontier = [(start, [])]
    path = None
    nodes = 0
    while frontier and path is None and len(frontier[0][1]) < req["depth"]:
        nxt = []
        for g, p in frontier:
            for a in req["actions"]:
                try:
                    h = norm(step([list(r) for r in g], act(a)))
                    nodes += 1
                    if is_goal(h):
                        path = p + [(a, h)]
                        break
                except Exception:
                    continue
                k = key(h)
                if k not in seen:
                    seen.add(k)
                    nxt.append((h, p + [(a, h)]))
                if nodes >= req["nodes"] or time.time() - t0 > req["seconds"]:
                    break
            if path is not None or nodes >= req["nodes"] or time.time() - t0 > req["seconds"]:
                break
        frontier = nxt
    out = {"path": path, "nodes": nodes, "goal_at_start": done0}
print(json.dumps(out))
'''


def sandbox(req, timeout=40):
    with tempfile.TemporaryDirectory() as d:
        runner = os.path.join(d, "run.py")
        open(runner, "w").write(RUNNER)
        try:
            p = subprocess.run([sys.executable, "-I", "-S", runner], input=json.dumps(req), capture_output=True,
                               text=True, timeout=timeout, cwd=d, env={})
            return json.loads(p.stdout.strip().splitlines()[-1])
        except subprocess.TimeoutExpired:
            return {"error": "timeout"}
        except Exception as e:
            return {"error": f"crash: {type(e).__name__}"}


def click_cells(g, cap=24):
    """Logical cells to try a click on: one per 4-connected object of a non-modal colour (its member nearest the
    centroid), in reading order."""
    vals, counts = np.unique(g, return_counts=True)
    bg = int(vals[np.argmax(counts)])
    out = []
    for c in [int(v) for v in vals if v != bg]:
        for comp in muse._components(g == c):
            my = sum(p[0] for p in comp) / len(comp)
            mx = sum(p[1] for p in comp) / len(comp)
            y, x = min(comp, key=lambda p: ((p[0] - my) ** 2 + (p[1] - mx) ** 2, p))
            out.append((x, y))
    return sorted(out, key=lambda p: (p[1], p[0]))[:cap]


class MuseAgent:
    """The closed loop above, behind the framework's `is_done` / `choose_action`."""

    def __init__(self, game_id, cfg, log_path, max_calls=16, seed=1):
        self.game_id, self.cfg, self.log_path, self.max_calls = game_id, cfg, log_path, max_calls
        self.rng = random.Random(f"{game_id}:{seed}")
        self.frames = []          # distinct frames seen (for the block grid)
        self.G = None
        self.level = 0
        self.trans = []           # this level's transitions: (grid, action, event, grid)
        self.tried = collections.defaultdict(set)
        self.code = None
        self.plan = []            # [(action, predicted grid)]
        self.prev = None          # (grid, action, predicted)
        self.calls = 0
        self.need_model = True
        self.stats = collections.Counter()

    # -- perception
    def grid_of(self, frame):
        f = np.asarray(frame, dtype=np.int64)
        if len(self.frames) < 12 and not any((f == x).all() for x in self.frames):
            self.frames.append(f)
            self.G = muse.Grid(self.frames)
        return self.G.grid(f)

    def to_action(self, a):
        from arcengine import GameAction
        if " " in a:
            _, x, y = a.split()
            G = self.G
            px = G.ox + (int(x) + G.x0) * G.k + G.k // 2
            py = G.oy + (int(y) + G.y0) * G.k + G.k // 2
            act = GameAction.from_id(6)
            act.set_data({"x": int(min(63, max(0, px))), "y": int(min(63, max(0, py)))})
            return act
        return GameAction.from_id(int(a))

    def actions(self, g, avail):
        acts = [str(a) for a in avail if a not in (0, 6)]
        if 6 in avail:
            acts += [f"6 {x} {y}" for x, y in click_cells(g)]
        return acts

    # -- the LLM
    def prompt(self, g, avail, failures):
        simple = ", ".join(str(a) for a in avail if a not in (0, 6)) or "none"
        hist = []
        for a, ev, b, act in [(t[0], t[2], t[3], t[1]) for t in self.trans[-14:]]:
            hist.append((act, ev, muse.object_diff_text(a, b)))
        parts = [muse.GRID_HELP.format(w=g.shape[1], h=g.shape[0]), muse.actions_text(avail),
                 "Current scene:\n" + muse.objects_text(g), "Current grid:\n" + muse.grid_text(g),
                 muse.history_text(hist, "objects")]
        if self.level:
            parts.append(f"The agent has completed {self.level} level(s) of this game; this is a new level.")
        if self.code and failures:
            parts.append("Your previous code:\n```python\n" + self.code + "\n```\nIt was WRONG on these recorded "
                         "transitions (action: what really happened / what your step predicted):\n" + failures +
                         "\nFix it.")
        elif self.code:
            parts.append("Your previous code (it matched every recorded transition, but planning with it did not "
                         "finish the level, so is_goal or step is still wrong):\n```python\n" + self.code + "\n```")
        parts.append(ASK.format(simple=simple))
        return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(parts)}]

    def ask(self, msgs):
        self.calls += 1
        t0 = time.time()
        try:
            content, reasoning, finish, usage = ask.stream(self.cfg["base_url"].rstrip("/") + "/chat/completions",
                                                           dict(model=self.cfg["model"], messages=msgs,
                                                                max_tokens=self.cfg.get("max_tokens", 4096),
                                                                temperature=self.cfg.get("temperature", 0.6),
                                                                seed=self.calls, **self.cfg.get("extra", {})), 900)
            err = None
        except Exception as e:
            content, reasoning, finish, usage, err = "", None, None, {}, f"{type(e).__name__}: {e}"
        with open(self.log_path, "a") as f:
            f.write(json.dumps(dict(game=self.game_id, level=self.level, call=self.calls, messages=msgs,
                                    answer=content, reasoning=reasoning, finish=finish, usage=usage, error=err,
                                    latency_s=round(time.time() - t0, 2), ts=time.time())) + "\n")
        return muse.extract_code(content or "")

    def failures(self, code):
        cases = [dict(grid=a.tolist(), action=act) for a, act, ev, b in self.trans if ev == "none"]
        if not cases:
            return 0, ""
        res = sandbox(dict(mode="verify", code=code, cases=cases))
        if "error" in res:
            return len(cases), res["error"]
        bad = []
        for (a, act, ev, b), post in zip([t for t in self.trans if t[2] == "none"], res["post"]):
            if post != b.tolist():
                pred = post if isinstance(post, str) else muse.object_diff_text(a, np.array(post))
                bad.append(f"- action {act}: really {muse.object_diff_text(a, b)} / yours {pred}")
        return len(bad), "\n".join(bad[:6])

    def model(self, g, avail):
        fails = ""
        for _ in range(3):
            if self.calls >= self.max_calls:
                return
            code = self.ask(self.prompt(g, avail, fails))
            if not code or "def step" not in code:
                self.stats["no_code"] += 1
                continue
            n_bad, fails = self.failures(code)
            self.code = code
            if n_bad == 0:
                self.stats["verified"] += 1
                return
            self.stats["revise"] += 1

    def make_plan(self, g, avail):
        if not self.code:
            return
        res = sandbox(dict(mode="plan", code=self.code, grid=g.tolist(), actions=self.actions(g, avail),
                           depth=40, nodes=30000, seconds=25))
        path = res.get("path") if isinstance(res, dict) else None
        if path:
            self.plan = [(a, np.array(h)) for a, h in path]
            self.stats["plans"] += 1
        else:
            self.stats["no_plan"] += 1

    # -- the framework interface
    def is_done(self, frames, latest):
        from arcengine import GameState
        return latest.state is GameState.WIN

    def choose_action(self, frames, latest):
        from arcengine import GameAction, GameState
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            if self.prev and latest.state is GameState.GAME_OVER:
                a, act, pred = self.prev
                self.trans.append((a, act, "death", a))
                self.prev, self.plan = None, []
            self.prev = None
            return GameAction.RESET
        g = self.grid_of(latest.frame[-1])
        avail = sorted(set(int(v) for v in (latest.available_actions or [])))
        lv = int(latest.levels_completed or 0)
        if self.prev is not None:
            a, act, pred = self.prev
            if lv > self.level:
                self.trans.append((a, act, "complete", g))
                self.stats["levels"] += 1
                self.level, self.trans, self.tried, self.plan = lv, [], collections.defaultdict(set), []
                self.frames, self.G = [], None
                g = self.grid_of(latest.frame[-1])
                self.need_model = True
            else:
                self.trans.append((a, act, "none", g))
                if pred is not None and (pred.shape != g.shape or (pred != g).any()):
                    self.stats["surprise"] += 1
                    self.plan, self.need_model = [], True
        key = hashlib.blake2b(g.tobytes(), digest_size=8).hexdigest()
        if not self.plan and self.need_model and self.calls < self.max_calls and len(self.trans) >= len(avail):
            self.need_model = False
            self.model(g, avail)
            self.make_plan(g, avail)
        if self.plan:
            a, pred = self.plan.pop(0)
            self.stats["planned_steps"] += 1
        else:
            pred = None
            acts = self.actions(g, avail)
            untried = [x for x in acts if x not in self.tried[key]]
            simple = [x for x in untried if " " not in x]
            a = (simple or untried or acts)[0] if (simple or untried) else self.rng.choice(acts)
            if not untried:
                a = self.rng.choice(acts)
                if self.code and self.calls < self.max_calls:
                    self.need_model = True
            self.stats["explore_steps"] += 1
        self.tried[key].add(a)
        self.prev = (g, a, pred)
        return self.to_action(a)


def main():
    import replay

    ap = argparse.ArgumentParser()
    ap.add_argument("env")
    ap.add_argument("--base-url", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--extra", default="{}", help="JSON merged into each request (e.g. chat_template_kwargs)")
    ap.add_argument("--max-tokens", type=int, default=4096)
    ap.add_argument("--max-calls", type=int, default=16, help="LLM calls per game")
    ap.add_argument("--games", default="")
    ap.add_argument("--jobs", type=int, default=8)
    ap.add_argument("--max-actions", type=int, default=1000)
    ap.add_argument("--out", required=True)
    ap.add_argument("--log", required=True)
    args = ap.parse_args()
    cfg = dict(base_url=args.base_url, model=args.model, max_tokens=args.max_tokens, extra=json.loads(args.extra))

    orig = replay.make_agent

    def make_agent(kind, game_id, seed, env, meta):
        if kind == "muse":
            return MuseAgent(game_id, cfg, args.log, max_calls=args.max_calls, seed=seed)
        return orig(kind, game_id, seed, env, meta)

    replay.make_agent = make_agent
    games = replay.discover(args.env, 40, "openmorph-gate-1")
    if args.games:
        keep = set(args.games.split(","))
        games = [g for g in games if g["base"] in keep or g["game_id"] in keep]
    opts = dict(env=args.env, max_actions=args.max_actions, offline_resets=False)
    tasks = [("muse", 1, g, opts) for g in games]
    ctx = multiprocessing.get_context("fork")
    rows = []
    with ctx.Pool(args.jobs) as pool:
        for r in pool.imap_unordered(replay._run_safe, tasks):
            rows.append(r)
            print(f"{r.get('game'):18} score {r.get('score', 0):7.2f}  levels {r.get('levels', 0)}/"
                  f"{r.get('n_levels', '?')}  actions {r.get('actions', 0)}  {r.get('error', '')}", flush=True)
    final = sum(r.get("score", 0) for r in rows) / max(1, len(rows))
    json.dump(dict(rows=rows, final=final, cfg={k: v for k, v in cfg.items()}), open(args.out, "w"), indent=1)
    print(f"FINAL muse {final:.3f} over {len(rows)} games; levels {sum(r.get('levels', 0) for r in rows)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/muse/weights/eval.py
# SPDX-License-Identifier: MIT-0
"""Score the options of `docs/plans/morph-llm/weights.md` on the common test (CPU, free).

    python eval.py items.jsonl out/

Top-1 per state (the item with the highest score must be a true colour), per question, for: chance, the authored
priors, W-0 (zero-shot logit), W-A (readouts on the hidden states, held out by game), W-B (the held-out LoRA
folds) and every W-C file found (`out/wc_*.jsonl`).

W-A, per held-out game: the layer is chosen by an inner leave-one-game-out ridge on the TRAINING games only; the
hidden state is standardised and projected on the training items' top 16 principal directions; two readouts are
trained on the training games' items: a ridge regression (the reference) and the morph's own grown readout
(`openmorph.nn.ValueHead`, a `Field`, positives oversampled to balance). Held-out items are scored by both.
"""
import argparse
import collections
import glob
import json
import os

import numpy as np

K = 16


def top1(items, score):
    """Top-1 accuracy over states: {q: (correct, states)}."""
    by = collections.defaultdict(lambda: collections.defaultdict(list))
    for it in items:
        if it["item_id"] in score:
            by[it["q"]][it["state_id"]].append(it)
    out = {}
    for q, states in by.items():
        ok = sum(max(v, key=lambda it: score[it["item_id"]])["label"] for v in states.values())
        out[q] = (ok, len(states))
    return out


def chance(items):
    by = collections.defaultdict(lambda: collections.defaultdict(list))
    for it in items:
        by[it["q"]][it["state_id"]].append(it["label"])
    return {q: (sum(sum(v) / len(v) for v in s.values()), len(s)) for q, s in by.items()}


def ridge_fit(X, y, lam=1.0):
    Xb = np.hstack([X, np.ones((len(X), 1))])
    return np.linalg.solve(Xb.T @ Xb + lam * np.eye(Xb.shape[1]), Xb.T @ y)


def ridge_pred(w, X):
    return np.hstack([X, np.ones((len(X), 1))]) @ w


def project(Htr, Hte, k=K):
    """Standardise on the training rows, project both on the training rows' top k principal directions. With fewer
    rows than dimensions the directions come from the n x n Gram matrix (the same subspace as the SVD, up to sign,
    which no readout here depends on), about 10x faster."""
    mu, sd = Htr.mean(0), Htr.std(0) + 1e-6
    A, B = (Htr - mu) / sd, (Hte - mu) / sd
    if A.shape[0] < A.shape[1]:
        vals, U = np.linalg.eigh(A @ A.T)
        top = np.argsort(vals)[::-1][:k]
        P = A.T @ (U[:, top] / np.sqrt(np.maximum(vals[top], 1e-12)))
    else:
        _, _, vt = np.linalg.svd(A, full_matrices=False)
        P = vt[:k].T
    Ztr, Zte = A @ P, B @ P
    s = Ztr.std(0) + 1e-6
    return Ztr / s, Zte / s


def logo_ridge(H, y, games, q_idx):
    """Inner leave-one-game-out top-1 proxy (mean held-out correlation sign agreement is too weak; use AUC-ish:
    the share of held-out positives scored above the held-out mean)."""
    gs = sorted(set(games))
    hits = n = 0
    for g in gs:
        tr, te = games != g, games == g
        if te.sum() == 0 or y[tr].std() == 0:
            continue
        Ztr, Zte = project(H[tr], H[te])
        w = ridge_fit(Ztr, y[tr])
        p = ridge_pred(w, Zte)
        pos = y[te] > 0
        if pos.any():
            hits += float((p[pos] > p.mean()).mean())
            n += 1
    return hits / max(1, n)


class RlsHead:
    """The morph's ridge: a `Field` whose readout unit learns by recursive least squares (`Config::readout_rls`,
    lambda 1 = every event weighted equally, penalty 1/P0), one `learn_step` per training item. With `backbone` > 0
    the readout reads a grown random-feature backbone (`grow_backbone`) of the inputs instead of the inputs."""

    def __init__(self, k, prior=100.0, backbone=0, seed=0):
        import json as _json
        from openmorph import Morph

        self.k = k
        self.m = Morph(k)
        self.m.configure(_json.dumps({"readout_rls": 1.0, "rls_prior": float(prior)}), False)
        self.m.rng_seed(seed)
        self.idx = np.arange(k)
        reads = list(self.m.grow_backbone(list(range(k)), backbone, 1.0)) if backbone else list(range(k))
        self.out = self.m.n_cells() if hasattr(self.m, "n_cells") else None
        self.out = max(reads + [k - 1]) + 1
        self.m.add_unit(reads, [0.0] * len(reads), 0.0, self.out)

    def learn(self, z, target):
        self.m.learn_step(self.idx, np.asarray(z, dtype=np.float64), self.out, float(target), 1.0, 256)

    def predict(self, z):
        self.m.stimulate(self.idx, np.asarray(z, dtype=np.float64), 256)
        return float(self.m.cell(self.out))


def make_readouts(spec):
    """name -> (train(Ztr, ytr) -> predict(z)) for the morph readouts compared in step 1."""
    from openmorph.nn import ValueHead

    def valuehead(hidden, epochs):
        def train(Z, y):
            head = ValueHead(Z.shape[1], hidden=hidden, budget=64, seed=0)
            pos, neg = np.where(y > 0)[0], np.where(y < 0)[0]
            rng = np.random.RandomState(0)
            order = np.concatenate([neg, rng.choice(pos, len(neg), replace=True)]) if len(pos) else neg
            for _ in range(epochs):
                rng.shuffle(order)
                for i in order:
                    head.learn(Z[i], 0.9 * y[i], 0.05)
            return head.predict
        return train

    def rls(prior, backbone):
        def train(Z, y):
            head = RlsHead(Z.shape[1], prior=prior, backbone=backbone)
            pos, neg = np.where(y > 0)[0], np.where(y < 0)[0]
            rng = np.random.RandomState(0)
            order = np.concatenate([neg, rng.choice(pos, len(neg), replace=True)]) if len(pos) else neg
            rng.shuffle(order)
            for i in order:
                head.learn(Z[i], 0.9 * y[i])
            return head.predict
        return train

    all_ = {"morph ValueHead h4": valuehead(4, 8), "morph ValueHead h0": valuehead(0, 8),
            "morph RLS linear P0=1": rls(1.0, 0), "morph RLS linear P0=100": rls(100.0, 0),
            "morph RLS backbone64 P0=100": rls(100.0, 64)}
    return {k: v for k, v in all_.items() if k in spec}


def wa(items, H, layers, use_morph=True, readouts=("morph ValueHead h4",)):
    ridge_s, chosen = {}, collections.Counter()
    heads = make_readouts(readouts) if use_morph else {}
    morph_s = {name: {} for name in heads}
    for q in ("self", "goal"):
        idx = np.array([i for i, it in enumerate(items) if it["q"] == q])
        sub = [items[i] for i in idx]
        y = np.array([1.0 if it["label"] else -1.0 for it in sub])
        games = np.array([it["game"] for it in sub])
        for g in sorted(set(games)):
            tr, te = games != g, games == g
            # layer chosen on the training games only
            best = max(range(len(layers)), key=lambda l: logo_ridge(H[idx][tr][:, l, :], y[tr], games[tr], None))
            chosen[(q, layers[best])] += 1
            Ztr, Zte = project(H[idx][tr][:, best, :], H[idx][te][:, best, :])
            w = ridge_fit(Ztr, y[tr])
            for it, v in zip([sub[i] for i in np.where(te)[0]], ridge_pred(w, Zte)):
                ridge_s[it["item_id"]] = float(v)
            for name, train in heads.items():
                predict = train(Ztr, y[tr])
                for it, z in zip([sub[i] for i in np.where(te)[0]], Zte):
                    morph_s[name][it["item_id"]] = predict(z)
    return ridge_s, morph_s, chosen


def load_scores(path):
    return {r["item_id"]: r["score"] for r in map(json.loads, open(path))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("items")
    ap.add_argument("out")
    ap.add_argument("--no-morph", action="store_true")
    ap.add_argument("--readouts", default="morph ValueHead h4", help="comma-separated morph readouts to compare")
    args = ap.parse_args()
    items = [json.loads(l) for l in open(args.items)]
    rows = []
    fmt = lambda r: f"{100 * r[0] / r[1]:5.1f}% ({r[0]:.0f}/{r[1]})" if r else "     -"
    rows.append(("chance", chance(items)))
    rows.append(("authored prior", top1(items, {it["item_id"]: it["prior"] for it in items})))
    zs_p = os.path.join(args.out, "zs.jsonl")
    if os.path.exists(zs_p):
        rows.append(("W-0 zero-shot 8B (Yes-No logit)", top1(items, load_scores(zs_p))))
    hp = os.path.join(args.out, "hidden.npy")
    if os.path.exists(hp):
        meta = json.load(open(os.path.join(args.out, "hidden_ids.json")))
        pos = {i: k for k, i in enumerate(meta["ids"])}
        H = np.load(hp).astype(np.float32)[[pos[it["item_id"]] for it in items]]
        r, m, chosen = wa(items, H, meta["layers"], use_morph=not args.no_morph, readouts=args.readouts.split(","))
        rows.append(("W-A ridge readout on hidden states", top1(items, r)))
        for name, sc in m.items():
            rows.append((f"W-A {name}", top1(items, sc)))
        print("W-A layers chosen (per held-out game):", dict(chosen))
    lp = os.path.join(args.out, "lora.jsonl")
    if os.path.exists(lp) and os.path.getsize(lp):
        rows.append(("W-B LoRA, held-out folds", top1(items, load_scores(lp))))
    for p in sorted(glob.glob(os.path.join(args.out, "wc_*.jsonl"))):
        rows.append((f"W-C {os.path.basename(p)[3:-6]} zero-shot", top1(items, load_scores(p))))
    print(f"\n{'option':44} {'Q-goal (primary)':>18} {'Q-self':>18}")
    for name, r in rows:
        print(f"{name:44} {fmt(r.get('goal')):>18} {fmt(r.get('self')):>18}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/muse/explore.py
# SPDX-License-Identifier: MIT-0
"""`explore`: an agent with no authored solver. It grows a model of WHICH actions change the world and spends its
budget on them.

    python explore.py ENV [--games a,b] [--jobs 4] --out runs/explore.json

Per step, on the logical grid (`muse.Grid`):
* CLOCK. Cells that change in most transitions (the step/energy bar every real game draws, a timer) are learned as
  the clock: a cell is clock when it changed in >= 60 % of the level's transitions (after 8 of them). An action's
  EFFECT is the change outside the clock.
* EFFECT MODEL. A `Field` readout (the RLS readout, `Config::readout_rls` 1, P0 1: the morph's ridge) learns, from
  every transition, how likely an action is to have an effect: inputs are the action's id (one-hot), and for a click
  the clicked object's colour (one-hot), log size and whether it is one cell; target +0.9 effect, -0.9 none, -0.9
  for a death.
* PROGRESS. A second RLS readout over the same features learns from completions: the completing action gets +0.9,
  the 19 before it 0.9 * 0.85^k. It carries what finished one level into the next.
* CHOICE. Score = predicted effect + predicted progress. Among the actions not yet tried in this exact grid, the one the model predicts most effective (ties: the
  first); when every action was tried here, the most effective one with a small seeded noise. A death marks the
  (grid, action) as tried. RESET only after a game over.
Nothing names a game, a colour or a mechanic; what is effective is learned per game from its own transitions.
"""
import argparse
import collections
import hashlib
import json
import multiprocessing
import os
import random
import sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path[:0] = [HERE, os.path.join(HERE, "..", "gate"), os.path.join(HERE, "weights")]
import muse  # noqa: E402
from play import click_cells  # noqa: E402

N_FEAT = 7 + 16 + 2


class EffectModel:
    """The morph's RLS readout over action features (see `weights/eval.py` RlsHead)."""

    def __init__(self, seed=0):
        from eval import RlsHead
        self.head = RlsHead(N_FEAT, prior=1.0, backbone=0, seed=seed)
        self.n = 0

    def learn(self, f, target):
        self.head.learn(f, target)
        self.n += 1

    def predict(self, f):
        return self.head.predict(f) if self.n else 0.0


def features(a, g, comp_of):
    f = np.zeros(N_FEAT)
    if " " in a:
        _, x, y = a.split()
        x, y = int(x), int(y)
        f[5] = 1.0                                   # action 6
        c = int(g[y, x]) if 0 <= y < g.shape[0] and 0 <= x < g.shape[1] else 0
        f[7 + c] = 1.0
        size = comp_of.get((x, y), 1)
        f[23] = np.log1p(size) / 5.0
        f[24] = 1.0 if size == 1 else 0.0
    else:
        f[int(a) - 1] = 1.0
    return f


class ExploreAgent:
    def __init__(self, game_id, seed=1):
        self.rng = random.Random(f"{game_id}:{seed}:explore")
        self.model = EffectModel()
        self.progress = EffectModel(seed=1)         # credit from completions, discounted backward
        self.recent = []                             # this level's action features, newest last
        self.w_progress = float(os.environ.get("EXPLORE_PROGRESS", "1.0"))
        self.frames, self.G = [], None
        self.level = 0
        self.prev = None                              # (grid, action, features)
        self.tried = collections.defaultdict(set)
        self.change_count = None
        self.n_trans = 0
        self.graph = collections.defaultdict(dict)    # state key -> {action: next state key}
        self.acts_of = {}                             # state key -> its candidate actions
        self.route = []                               # actions toward the nearest state with untried actions
        self.frontier = bool(int(os.environ.get("EXPLORE_FRONTIER", "1")))
        self.stats = collections.Counter()

    def grid_of(self, frame):
        f = np.asarray(frame, dtype=np.int64)
        if len(self.frames) < 12 and not any((f == x).all() for x in self.frames):
            self.frames.append(f)
            self.G = muse.Grid(self.frames)
        return self.G.grid(f)

    def to_action(self, a):
        from arcengine import GameAction
        if " " in a:
            _, x, y = a.split()
            G = self.G
            act = GameAction.from_id(6)
            act.set_data({"x": int(min(63, max(0, G.ox + (int(x) + G.x0) * G.k + G.k // 2))),
                          "y": int(min(63, max(0, G.oy + (int(y) + G.y0) * G.k + G.k // 2)))})
            return act
        return GameAction.from_id(int(a))

    def effect(self, a, b):
        """Changed cells outside the learned clock."""
        if a.shape != b.shape:
            return True
        d = a != b
        if self.change_count is None or self.change_count.shape != d.shape:
            self.change_count = np.zeros(d.shape)
            self.n_trans = 0
        self.change_count += d
        self.n_trans += 1
        clock = (self.change_count >= 0.6 * self.n_trans) if self.n_trans >= 8 else np.zeros(d.shape, bool)
        return bool((d & ~clock).any())

    def clock_mask(self, shape):
        if self.change_count is None or self.change_count.shape != shape or self.n_trans < 8:
            return np.zeros(shape, bool)
        return self.change_count >= 0.6 * self.n_trans

    def key(self, g):
        """The state with the learned clock masked out (the step bar must not make every state new)."""
        h = g.copy()
        h[self.clock_mask(g.shape)] = -1
        return hashlib.blake2b(h.tobytes(), digest_size=8).hexdigest()

    def route_to_frontier(self, start):
        """Shortest known action path from `start` to a state that still has untried actions."""
        prev = {start: None}
        q = collections.deque([start])
        while q:
            k = q.popleft()
            if k != start and any(a not in self.tried[k] for a in self.acts_of.get(k, [])):
                path = []
                while prev[k] is not None:
                    k, a = prev[k]
                    path.append(a)
                return path[::-1]
            for a, n in self.graph[k].items():
                if n not in prev:
                    prev[n] = (k, a)
                    q.append(n)
        return []

    def is_done(self, frames, latest):
        from arcengine import GameState
        return latest.state is GameState.WIN

    def choose_action(self, frames, latest):
        from arcengine import GameAction, GameState
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            if self.prev is not None and latest.state is GameState.GAME_OVER:
                g0, a0, f0 = self.prev
                self.model.learn(f0, -0.9)
                self.stats["deaths"] += 1
            self.prev = None
            return GameAction.RESET
        g = self.grid_of(latest.frame[-1])
        lv = int(latest.levels_completed or 0)
        if self.prev is not None:
            g0, a0, f0 = self.prev
            if lv > self.level:
                self.model.learn(f0, 0.9)
                for i, f in enumerate(reversed((self.recent + [f0])[-20:])):  # the completing action first
                    self.progress.learn(f, 0.9 * (0.85 ** i))
                self.recent = []
                self.level = lv
                self.frames, self.G, self.change_count = [], None, None
                self.tried = collections.defaultdict(set)
                self.graph, self.acts_of, self.route = collections.defaultdict(dict), {}, []
                g = self.grid_of(latest.frame[-1])
                self.stats["levels"] += 1
            else:
                eff = self.effect(g0, g)
                self.model.learn(f0, 0.9 if eff else -0.9)
                self.stats["effective" if eff else "no_effect"] += 1
                self.graph[self.key(g0)][a0] = self.key(g)
        avail = sorted(set(int(v) for v in (latest.available_actions or [])))
        acts = [str(a) for a in avail if a not in (0, 6)]
        comp_of = {}
        if 6 in avail:
            cells = click_cells(g, cap=32)
            vals, counts = np.unique(g, return_counts=True)
            for x, y in cells:
                c = int(g[y, x])
                size = next((len(comp) for comp in muse._components(g == c) if (y, x) in comp), 1)
                comp_of[(x, y)] = size
            acts += [f"6 {x} {y}" for x, y in cells]
        key = self.key(g)
        self.acts_of[key] = acts
        untried = [a for a in acts if a not in self.tried[key]]
        if self.frontier and not untried:
            if not self.route:
                self.route = self.route_to_frontier(key)
            if self.route:
                a = self.route.pop(0)
                if a in acts:
                    self.stats["route_steps"] += 1
                    self.prev = (g, a, features(a, g, comp_of))
                    return self.to_action(a)
                self.route = []
        else:
            self.route = []
        pool = untried or acts
        scored = [(self.model.predict(features(a, g, comp_of))
                   + self.w_progress * self.progress.predict(features(a, g, comp_of))
                   + (0 if untried else self.rng.random() * 0.3), -i, a)
                  for i, a in enumerate(pool)]
        a = max(scored)[2]
        self.tried[key].add(a)
        self.prev = (g, a, features(a, g, comp_of))
        self.recent.append(self.prev[2])
        return self.to_action(a)


def main():
    import replay

    ap = argparse.ArgumentParser()
    ap.add_argument("env")
    ap.add_argument("--games", default="")
    ap.add_argument("--jobs", type=int, default=4)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--split", default="all", choices=["all", "dev", "sealed"])
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    orig = replay.make_agent

    def make_agent(kind, game_id, seed, env, meta):
        return ExploreAgent(game_id, seed) if kind == "explore" else orig(kind, game_id, seed, env, meta)

    replay.make_agent = make_agent
    games = replay.discover(args.env, 40, "openmorph-gate-1")
    if args.split != "all":
        games = [g for g in games if g["split"] == args.split]
    if args.games:
        keep = set(args.games.split(","))
        games = [g for g in games if g["base"] in keep or g["game_id"] in keep]
    opts = dict(env=args.env, max_actions=1000, offline_resets=False)
    rows = []
    with multiprocessing.get_context("fork").Pool(args.jobs) as pool:
        for r in pool.imap_unordered(replay._run_safe, [("explore", args.seed, g, opts) for g in games]):
            rows.append(r)
            print(f"{r.get('game'):18} [{r.get('split')}] score {r.get('score', 0):6.2f}  levels "
                  f"{r.get('levels', 0)}/{r.get('n_levels', '?')}  deaths {r.get('deaths', 0)}  {r.get('error', '')}",
                  flush=True)
    by = collections.defaultdict(list)
    for r in rows:
        by[r.get("split")].append(r)
    for s, rs in sorted(by.items()):
        print(f"FINAL {s:7} {sum(r.get('score', 0) for r in rs) / len(rs):6.3f}  levels "
              f"{sum(r.get('levels', 0) for r in rs)}")
    print(f"FINAL all     {sum(r.get('score', 0) for r in rows) / len(rows):6.3f}  levels "
          f"{sum(r.get('levels', 0) for r in rows)}")
    json.dump(dict(rows=rows), open(args.out, "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/muse/think.py
# SPDX-License-Identifier: MIT-0
"""`think`: kit X's explorer (`explore.ExploreAgent`, v3) that plans before it acts (stream TK, 2026-10-26).

Compute is free and actions are scored, so each decision searches the agent's own models for the action with the
most expected discovery per real action spent. All of it is a usage protocol over v3's state graph and heads; nothing
here is in a crate.

The imagined world, per decision:
* ON THE GRAPH (never wrong): the observed transitions `graph[state][action] -> state`, searched breadth-first from
  the current state up to `TK_DEPTH` real actions (1, 4, 16, 64). Known deaths are edges the search never takes.
* OFF THE GRAPH (an untried action in a reached state): its outcome is predicted only from rates VERIFIED on this
  game's own real tries, per action type (the key, or a click's colour and whether it is one cell): the share of
  first tries that reached a NEW state, a known one, no change, or a death. One more verified rate, per ordered pair
  of action types: how often `b` right after `a` returned to the state before `a` (an undo). An untried `b` in a
  state entered by `a` is predicted to lead back at that rate.
* VALUE: gain = P(new state) + w_p * progress(action) (v3's progress head, learned from completions);
  cost = the path's actions + 1 + P(death) * (1 + the known distance from the level's start back to that state).
  The pick is the best gain / cost; the first action of its path is taken, its real outcome checked, and the next
  decision re-plans (MPC). Nothing in reach -> v3's own choice (its frontier route, then its noise).

AUTHORED (labelled, as the rules ask): the outcome classes (new / known / none / death), the Beta(1,1)-style priors,
the gain / cost form, the undo pairing and the depth horizon. LEARNED per game: every rate, the graph, v3's two
heads and v3's clock. Nothing names a game, a colour or a mechanic.

Switches (environment): TK_DEPTH (default 16), TK_PROGRESS (w_p, default 1.0), TK_UNDO (1), TK_DEATH (1).
"""
import collections
import os
import time

import numpy as np

import explore
from explore import features


class ThinkAgent(explore.ExploreAgent):
    def __init__(self, game_id, seed=1):
        super().__init__(game_id, seed)
        self.depth = int(os.environ.get("TK_DEPTH", "16"))
        self.w_p = float(os.environ.get("TK_PROGRESS", "1.0"))
        self.use_undo = bool(int(os.environ.get("TK_UNDO", "1")))
        self.use_death = bool(int(os.environ.get("TK_DEATH", "1")))
        # verified outcome counts of first tries, per action type: [new, known, none, death]
        self.outcome = collections.defaultdict(lambda: [0, 0, 0, 0])
        self.undo = collections.defaultdict(lambda: [0, 0])     # (type a, type b) -> [undid, tries]
        self.parent = {}                                          # state -> (previous state, action type into it)
        self.dead = set()                                         # (state, action) known deaths
        self.start_key = None                                     # the level's first state (where a death returns)
        self.pend = None                                          # (state key, action, type, first try?, parent)
        self.think_s = 0.0
        self.pred_log = collections.defaultdict(lambda: [0, 0])   # source -> [one-step predictions right, made]
        self.pend_key, self.pend_pred = None, None
        self.n_steps = 0
        self.game_id = game_id
        self.diag = os.environ.get("TK_DIAG", "")                # a directory: one JSONL line per 100 steps (off)
        self.kinds_of = {}                                        # state -> {action: type}
        self.awaiting_start = True
        self.use_raw = bool(int(os.environ.get("TK_RAW", "1")))
        self.raw = False                                          # reading the raw frame (after the grid was blind)
        self.level_steps = 0
        self.use_budget = bool(int(os.environ.get("TK_BUDGET", "1")))
        self.use_commit = bool(int(os.environ.get("TK_COMMIT", "1")))
        self.life = 0                                             # actions since this life began
        self.life_ends = []                                       # life lengths that ended in a death
        self.goal = None                                          # the committed target (state, untried action)
        self.v3_order = bool(int(os.environ.get("TK_V3ORDER", "0")))  # untried actions here: v3's order (off)
        self.arbiter = bool(int(os.environ.get("TK_ARBITER", "0")))   # ... when v3's head is better calibrated (off)
        self.brier = [0.0, 0.0, 0]                                    # P(new) squared error: planner, v3, count
        self.pend_brier, self.pend_first = None, False
        self.line = int(os.environ.get("TK_LINE", "1"))           # v4's line clock: 0 off, 1 in raw mode, 2 always

    # ---- the blind-reading fallback (AUTHORED rule) ----------------------------------------------------------------
    class _Raw:
        """The identity reading: one cell per pixel (the `Grid` interface `to_action` uses)."""
        k, ox, oy, x0, y0 = 1, 0, 0, 0, 0

    def grid_of(self, frame):
        if self.raw:
            return np.asarray(frame, dtype=np.int64)
        return super().grid_of(frame)

    def blind(self):
        """The logical grid shows (almost) nothing this level: every known state's actions tried, or under one state
        per ten actions after 200. Verified on this game's own play, so only a stagnant level switches."""
        if not self.use_raw or self.raw or self.level_steps < 50:
            return False
        front = any(any(x not in self.tried[k] for x in acts) for k, acts in self.acts_of.items())
        return not front or (self.level_steps >= 200 and 10 * len(self.acts_of) < self.level_steps)

    def go_raw(self):
        self.raw = True
        self.G = self._Raw()
        self.frames, self.change_count, self.row_count = [], None, None
        self.tried = collections.defaultdict(set)
        self.graph, self.acts_of, self.route = collections.defaultdict(dict), {}, []
        self.parent, self.dead, self.kinds_of = {}, set(), {}
        self.start_key, self.prev, self.pend, self.goal = None, None, None, None
        self.stats["went_raw"] = self.n_steps

    # ---- v4's line clock (gpu-track 34e1a31d, AUTHORED there: border lines changing by <= 2 cells at a time) -------
    def line_on(self):
        return self.line == 2 or (self.line == 1 and self.raw)

    def effect(self, a, b):
        if not self.line_on() or a.shape != b.shape:
            return super().effect(a, b)
        d = a != b
        if (self.change_count is None or self.change_count.shape != d.shape or getattr(self, "row_count", None) is None
                or self.row_count.shape[0] != d.shape[0] or self.col_count.shape[0] != d.shape[1]):
            self.change_count = np.zeros(d.shape)
            self.row_count, self.col_count = np.zeros(d.shape[0]), np.zeros(d.shape[1])
            self.row_small, self.col_small = np.zeros(d.shape[0]), np.zeros(d.shape[1])
            self.n_trans = 0
        self.change_count += d
        rows, cols = d.sum(1), d.sum(0)
        self.row_count += rows > 0
        self.col_count += cols > 0
        self.row_small += (rows > 0) & (rows <= 2)
        self.col_small += (cols > 0) & (cols <= 2)
        self.n_trans += 1
        return bool((d & ~self.clock_mask(d.shape)).any())

    def clock_mask(self, shape):
        m = super().clock_mask(shape)
        if not self.line_on() or self.n_trans < 8 or getattr(self, "row_count", None) is None \
                or self.row_count.shape[0] != shape[0] or self.col_count.shape[0] != shape[1]:
            return m
        n = self.n_trans
        h, w = shape
        edge_r = np.zeros(h, bool)
        edge_r[:2] = edge_r[-2:] = True
        edge_c = np.zeros(w, bool)
        edge_c[:2] = edge_c[-2:] = True
        m = m.copy()
        m[edge_r & (self.row_count >= 0.6 * n) & (self.row_small >= 0.9 * self.row_count), :] = True
        m[:, edge_c & (self.col_count >= 0.6 * n) & (self.col_small >= 0.9 * self.col_count)] = True
        return m

    # ---- action types -------------------------------------------------------------------------------------------
    @staticmethod
    def kind(a, g, comp_of):
        if " " in a:
            _, x, y = a.split()
            x, y = int(x), int(y)
            c = int(g[y, x]) if 0 <= y < g.shape[0] and 0 <= x < g.shape[1] else 0
            return f"6:{c}:{int(comp_of.get((x, y), 1) == 1)}"
        return a

    def rates(self, t):
        n, k, z, d = self.outcome[t]
        tot = n + k + z + d
        p_new = (n + 1.0) / (tot + 2.0)
        p_death = (d + 0.1) / (tot + 2.0) if self.use_death else 0.0
        return p_new, p_death

    def undo_rate(self, ta, tb):
        u, n = self.undo[(ta, tb)]
        return (u + 0.0) / (n + 1.0)

    # ---- the one-step prediction (THE HOOK: what will the next state be) --------------------------------------------
    def predict_next(self, n, a, t, par, graph=True):
        """What taking `a` in state `n` (action type `t`, entered from `par` = (state, type) or None) leads to.

        Returns {"source", "next", "p_new", "p_death", "p_undo"}: on a seen edge, source "graph" and `next` the exact
        next state key; a known death, source "dead"; otherwise source "rates", `next` None and the probabilities
        verified on this game's own tries (`graph=False`: the search's call, which scores untried actions by rates as
        measured, even where a clock-mask change left an edge under the same key). The search reads only `p_new` and
        `p_death` off the graph, so a model of
        trusted facts (stream HY) can replace this method (subclass or assign) without touching the search."""
        if (n, a) in self.dead:
            return dict(source="dead", next=None, p_new=0.0, p_death=1.0, p_undo=0.0)
        nxt = self.graph.get(n, {}).get(a) if graph else None
        if nxt is not None:
            return dict(source="graph", next=nxt, p_new=0.0, p_death=0.0, p_undo=1.0 if par and nxt == par[0] else 0.0)
        p_new, p_death = self.rates(t)
        u = self.undo_rate(par[1], t) if self.use_undo and par is not None else 0.0
        if self.use_undo and par is not None:
            p_new *= 1.0 - u
        return dict(source="rates", next=None, p_new=p_new, p_death=p_death, p_undo=u)

    def predicted_class(self, pred, t):
        """The single most likely outcome, for scoring the prediction (logging only; the search never reads it)."""
        if pred["source"] == "dead":
            return "death"
        if pred["source"] == "graph":
            return "none" if pred["next"] == self.pend_key else "known"
        n, k, z, d = self.outcome[t]
        tot = n + k + z + d + 2.0
        q = {"new": (n + 0.5) / tot, "known": (k + 0.5) / tot, "none": (z + 0.5) / tot, "death": (d + 0.5) / tot}
        q["known"] += q["new"] * pred["p_undo"]
        q["new"] *= 1.0 - pred["p_undo"]
        return max(("new", "known", "none", "death"), key=lambda c: q[c])

    def note_prediction(self, key, a, t, par):
        self.pend_key = key
        self.pend_first = bool(self.pend is not None and self.pend[3])     # this is the action's first try here
        pred = self.predict_next(key, a, t, par)
        self.pend_pred = (pred["source"], pred["next"], self.predicted_class(pred, t))
        # both orderers' P(new state) for this action, scored on first tries off the graph (the arbiter)
        f = self.prev[2] if self.prev is not None else None
        v3 = min(1.0, max(0.0, (self.model.predict(f) + 1.0) / 2.0)) if f is not None else None
        self.pend_brier = (pred["p_new"], v3) if pred["source"] == "rates" and v3 is not None else None

    def score_prediction(self, k0, k_now, died, known_before):
        src, nxt, cls = self.pend_pred
        actual = "death" if died else ("none" if k_now == k0 else ("known" if known_before else "new"))
        ok = (nxt == k_now) if (src == "graph" and not died) else (cls == actual)
        if self.pend_brier is not None and k0 is not None and self.pend_first:
            y = 1.0 if actual == "new" else 0.0
            self.brier[0] += (self.pend_brier[0] - y) ** 2
            self.brier[1] += (self.pend_brier[1] - y) ** 2
            self.brier[2] += 1
        rec = self.pred_log[src]
        rec[0] += int(ok)
        rec[1] += 1

    # ---- the real outcome of the last action -----------------------------------------------------------------------
    def observe(self, k_now, died, leveled):
        if self.pend is None:
            return
        k0, a, t, first, par = self.pend
        self.pend = None
        if leveled:
            return
        if died:
            self.score_prediction(k0, None, True, False)
            self.dead.add((k0, a))
            if first:
                self.outcome[t][3] += 1
            return
        known_before = k_now in self.graph or k_now in self.parent or k_now == self.start_key
        self.score_prediction(k0, k_now, False, known_before)
        if first:
            self.outcome[t][2 if k_now == k0 else (1 if known_before else 0)] += 1
        if par is not None and k_now != k0:
            pk, pt = par
            self.undo[(pt, t)][1] += 1
            if k_now == pk:
                self.undo[(pt, t)][0] += 1
        if k_now != k0 and k_now not in self.parent and k_now != self.start_key:
            self.parent[k_now] = (k0, t)

    # ---- the search -------------------------------------------------------------------------------------------------
    def bfs(self, start, limit):
        dist, first = {start: 0}, {start: None}
        q = collections.deque([start])
        while q:
            k = q.popleft()
            if dist[k] >= limit:
                continue
            for a, n in self.graph[k].items():
                if n not in dist and (k, a) not in self.dead:
                    dist[n] = dist[k] + 1
                    first[n] = a if k == start else first[k]
                    q.append(n)
        return dist, first

    def budget(self):
        """A life's step budget, once verified: the two longest lives ended within 2 actions of each other."""
        if not self.use_budget or len(self.life_ends) < 2:
            return None
        a, b = sorted(self.life_ends)[-2:]
        return a if b - a <= 2 else None

    def plan(self, key, acts_here, g, comp_of):
        if self.v3_order or (self.arbiter and self.brier[2] >= 10 and self.brier[1] < self.brier[0]):
            here = [a for a in acts_here if a not in self.tried[key] and (key, a) not in self.dead]
            if here:                                   # v3's own pick where v3 had one (its effect + progress order)
                self.goal = None
                return max((self.model.predict(features(a, g, comp_of))
                            + self.w_progress * self.progress.predict(features(a, g, comp_of)), -i, a)
                           for i, a in enumerate(here))[2]
        dist, first = self.bfs(key, self.depth)
        budget = self.budget()
        left = None if budget is None else budget - self.life       # actions before the budget kills this life
        if self.use_commit and self.goal is not None:
            n, a = self.goal
            if (n in dist and a not in self.tried[n] and (n, a) not in self.dead
                    and (left is None or dist[n] + 1 < left)):
                return a if n == key else first[n]
            self.goal = None
        back = self.bfs(self.start_key, 10 ** 6)[0] if self.start_key is not None else {}
        best = None
        for n, d in dist.items():
            if left is not None and d + 1 >= left:
                continue
            acts = acts_here if n == key else self.acts_of.get(n, [])
            par = self.parent.get(n)
            for i, a in enumerate(acts):
                if a in self.tried[n] or (n, a) in self.dead:
                    continue
                t = self.kind(a, g, comp_of) if n == key else self.kinds_of.get(n, {}).get(a, a)
                pred = self.predict_next(n, a, t, par, graph=False)
                p_new, p_death = pred["p_new"], pred["p_death"]
                gain = p_new
                if self.w_p and n == key:
                    gain += self.w_p * max(0.0, self.progress.predict(features(a, g, comp_of)))
                cost = d + 1 + p_death * (1 + back.get(n, d))
                s = (gain / cost, -d, -i)
                if best is None or s > best[0]:
                    best = (s, n, a)
        if best is None:
            return None
        _, n, a = best
        self.goal = (n, a) if n != key else None
        return a if n == key else first[n]

    # ---- one decision -------------------------------------------------------------------------------------------------
    def choose_action(self, frames, latest):
        from arcengine import GameState
        t0 = time.perf_counter()
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            if latest.state is GameState.GAME_OVER:
                self.observe(None, True, False)
                if not self.awaiting_start:
                    self.life_ends.append(self.life)
                self.goal = None
            self.awaiting_start = True
            a = super().choose_action(frames, latest)       # v3: learns the death, returns RESET
        else:
            a = self._think(latest)
        self.think_s += time.perf_counter() - t0
        self.n_steps += 1
        if self.diag and self.n_steps % 100 == 0:
            self.dump()
        return a

    def dump(self):
        import json
        front = sum(1 for k, acts in self.acts_of.items() if any(x not in self.tried[k] for x in acts))
        rec = dict(game=self.game_id, step=self.n_steps, level=self.level, nodes=len(self.acts_of), frontier=front,
                   edges=sum(len(v) for v in self.graph.values()), dead=len(self.dead), stats=dict(self.stats),
                   outcome={t: v for t, v in self.outcome.items()}, pred=dict(self.pred_log), brier=self.brier, think_ms=round(1000 * self.think_s / self.n_steps, 3))
        with open(os.path.join(self.diag, f"{self.game_id}.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")

    def _think(self, latest):
        self.level_steps += 1
        if self.blind():
            self.go_raw()
        import muse
        from play import click_cells
        # v3's bookkeeping (identical to explore.ExploreAgent.choose_action up to its choice)
        g = self.grid_of(latest.frame[-1])
        lv = int(latest.levels_completed or 0)
        if self.prev is not None:
            g0, a0, f0 = self.prev
            if lv > self.level:
                self.model.learn(f0, 0.9)
                for i, f in enumerate(reversed((self.recent + [f0])[-20:])):
                    self.progress.learn(f, 0.9 * (0.85 ** i))
                self.recent = []
                self.level = lv
                self.frames, self.G, self.change_count = [], None, None
                self.tried = collections.defaultdict(set)
                self.graph, self.acts_of, self.route = collections.defaultdict(dict), {}, []
                g = self.grid_of(latest.frame[-1])
                self.stats["levels"] += 1
                self.parent, self.dead, self.kinds_of = {}, set(), {}
                self.start_key, self.goal = None, None
                self.level_steps = 0
                self.life_ends = []
                if self.raw:
                    self.G = self._Raw()
                self.observe(None, False, True)
                self.awaiting_start = True
            else:
                eff = self.effect(g0, g)
                self.model.learn(f0, 0.9 if eff else -0.9)
                self.stats["effective" if eff else "no_effect"] += 1
                self.graph[self.key(g0)][a0] = self.key(g)
        avail = sorted(set(int(v) for v in (latest.available_actions or [])))
        acts = [str(a) for a in avail if a not in (0, 6)]
        comp_of = {}
        if 6 in avail:
            cells = click_cells(g, cap=64 if self.raw else 32)
            for x, y in cells:
                c = int(g[y, x])
                size = next((len(comp) for comp in muse._components(g == c) if (y, x) in comp), 1)
                comp_of[(x, y)] = size
            acts += [f"6 {x} {y}" for x, y in cells]
        if not acts:
            # no candidate at all (a click-only level whose logical grid is one colour): the reading is blind now,
            # not after 50 steps; v3 (and this agent before) raised here and the wrapper clicked the frame's centre
            if self.use_raw and not self.raw:
                self.go_raw()
                return self._think(latest)
            if 6 in avail:
                acts = [f"6 {g.shape[1] // 2} {g.shape[0] // 2}"]
            else:
                from arcengine import GameAction
                return GameAction.RESET
        key = self.key(g)
        if self.awaiting_start:
            self.start_key = key
            self.awaiting_start = False
            self.life = 0
        self.life += 1
        self.observe(key, False, False)
        self.acts_of[key] = acts
        self.kinds_of[key] = {a: self.kind(a, g, comp_of) for a in acts}
        a = self.plan(key, acts, g, comp_of)
        if a is not None and a in acts:
            self.stats["planned"] += 1
            self.route = []
            first = a not in self.tried[key]
            if first:
                self.tried[key].add(a)
                self.recent.append(features(a, g, comp_of))
            self.prev = (g, a, features(a, g, comp_of))
            self.pend = (key, a, self.kinds_of[key][a], first, self.parent.get(key))
            self.note_prediction(key, a, self.kinds_of[key][a], self.parent.get(key))
            return self.to_action(a)
        # nothing untried within reach: v3's own choice (its frontier route, then its noise)
        untried = [x for x in acts if x not in self.tried[key]]
        if self.frontier and not untried:
            if not self.route:
                self.route = self.route_to_frontier(key)
            if self.route:
                a = self.route.pop(0)
                if a in acts:
                    self.stats["route_steps"] += 1
                    self.prev = (g, a, features(a, g, comp_of))
                    self.pend = (key, a, self.kinds_of[key][a], False, self.parent.get(key))
                    self.note_prediction(key, a, self.kinds_of[key][a], self.parent.get(key))
                    return self.to_action(a)
                self.route = []
        else:
            self.route = []
        pool = untried or acts
        scored = [(self.model.predict(features(x, g, comp_of))
                   + self.w_progress * self.progress.predict(features(x, g, comp_of))
                   + (0 if untried else self.rng.random() * 0.3), -i, x) for i, x in enumerate(pool)]
        a = max(scored)[2]
        first = a not in self.tried[key]
        self.tried[key].add(a)
        self.prev = (g, a, features(a, g, comp_of))
        self.recent.append(self.prev[2])
        self.pend = (key, a, self.kinds_of[key][a], first, self.parent.get(key))
        self.note_prediction(key, a, self.kinds_of[key][a], self.parent.get(key))
        self.stats["fallback"] += 1
        return self.to_action(a)


In [ ]:
%%writefile /kaggle/working/facts.py
# SPDX-License-Identifier: MIT-0
"""Stream HY: the screen as many small hypotheses.

Before every action the `Verifier` predicts many separate facts about the next frame; after the action it checks
each one against the frame that came back. Every fact is a confirmed or refuted hypothesis, and both are learning
signal. The verifier is an observer: it never chooses an action (the agent it wraps does), so the same layer scores
any agent, and an agent may ask it what it expects (`expect`) before choosing.

The fact types (all yes/no; the prediction is the sign of a `Field` RLS readout, `weights/eval.py` RlsHead, unless
the line says COMPOSED):

  effect        a cell outside the clock changes
  death         the game is over after the action
  level         the level is completed by the action
  reset         the grid returns to the level's first grid (outside the clock) without a game over
  clock_tick    a clock cell changes (the clock: a cell changing in >= 60 % of the level's transitions, after 8)
  novel         a cell that had never changed in this level changes
  cell_change   per active cell (one that has changed in this level): it changes now
  cell_value    per active cell that changed: its new value is the predicted one (a learned table, below)
  self_moves    the self is displaced
  self_pos      COMPOSED: the self's next position (mode displacement of the action if `self_moves`, else none)
  contact       COMPOSED: the self touches another object after the action (from the predicted position)
  obj_gone      per object: no cell of it keeps its colour and no translated copy exists
  obj_moved     per object: it moved rigidly
  obj_recolour  per object: all its cells now hold one other colour
  obj_reshape   per object: its colour's component over it has another cell set (not a translation)

What is AUTHORED (game-blind, nothing names a game, a colour or a mechanic): the logical grid reading (`muse.Grid`,
kit X's), the clock rule (kit X's), the list of fact types above and their truth tests, the feature lists below, the
choice of the self (the colour that moves rigidly most often under non-click actions), the 4-connected object
reading, and the cell-value table's key (old value -> new value, per cell, then for the whole game). LEARNED per game:
every readout, the self's colour, its displacement per action, the value table, the clock, and every rate.

Scores (`report`): per level and per fact type a confusion matrix. The understanding score U is the macro mean over
fact types of the AUC (the chance that a random "yes" case got a higher predicted probability than a random "no"
case), taken over the types that saw both answers: a constant predictor scores 0.5 on every type, so U = 0.5 is no
understanding and U = 1 is every fact ranked right. The composed `self_pos` and `cell_value` have no probability and
report a raw accuracy only (with `self_pos` the baseline "the self stays" is reported beside it).
U_acc is the macro mean of the raw accuracy (flattered by facts that are almost always no), kept for reference.
"""
import collections
import json
import math
import os
import sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
for _p in (os.path.join(HERE, "..", "kit-x", "muse"), os.path.join(HERE, "..", "kit-x", "muse", "weights")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import muse  # noqa: E402  (kit X's grid reader and components)

LEARNED = ("effect", "death", "level", "reset", "clock_tick", "novel", "cell_change", "self_moves",
           "obj_gone", "obj_moved", "obj_recolour", "obj_reshape")
COMPOSED = ("cell_value", "self_pos", "self_pos_base", "contact")
TYPES = LEARNED + COMPOSED
N_ACT = 25            # kit X's action features: id one-hot (7), clicked colour (16), log size, one cell
N_LIFE = 5            # steps in this life (log), / mean past life, past it, clock cells used since the life began, none
N_AHEAD = 18          # the colour the self would enter (16), out of the grid, nothing known
N_CELL = 7 + 10
N_OBJ = 7 + 8 + 16
MAX_OBJECTS = int(os.environ.get("HY_MAX_OBJECTS", "48"))     # objects with facts per step (cost)
MAX_CELLS = int(os.environ.get("HY_MAX_CELLS", "160"))       # active cells with facts per step (cost)
WIN = 50              # steps per window of the understanding curve


class Head:
    """One yes/no readout: the morph's RLS readout (`Config::readout_rls`), target +-0.9."""

    def __init__(self, k, seed):
        from eval import RlsHead
        self.h = RlsHead(k + 1, prior=1.0, backbone=0, seed=seed)
        self.n = 0

    def predict(self, f):
        return self.h.predict(np.append(f, 1.0)) if self.n else 0.0

    def learn(self, f, yes, times=1):
        for _ in range(times):
            self.h.learn(np.append(f, 1.0), 0.9 if yes else -0.9)
        self.n += 1


NB = 20               # coarse probability bins: the trust verdicts and the arms' rates
NBF = 200             # fine bins of the raw readout over [-1.5, 1.5]: every reported AUC (rare facts sit near -0.9)


class Confusion:
    __slots__ = ("tp", "fp", "fn", "tn", "surprise", "pos", "neg", "posf", "negf")

    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = self.surprise = 0
        self.pos, self.neg = [0] * NB, [0] * NB        # histogram of the predicted probability, by the truth
        self.posf, self.negf = None, None               # the same over the raw readout, fine (allocated on use)

    def add(self, pred, truth, conf, prob=None, raw=None):
        if raw is not None:
            if self.posf is None:
                self.posf, self.negf = [0] * NBF, [0] * NBF
            b = min(NBF - 1, max(0, int((raw + 1.5) / 3.0 * NBF)))
            (self.posf if truth else self.negf)[b] += 1
        if prob is not None:
            b = min(NB - 1, max(0, int(prob * NB)))
            (self.pos if truth else self.neg)[b] += 1
        if pred and truth:
            self.tp += 1
        elif pred:
            self.fp += 1
        elif truth:
            self.fn += 1
        else:
            self.tn += 1
        if pred != truth and conf >= 0.5:
            self.surprise += 1

    @property
    def n(self):
        return self.tp + self.fp + self.fn + self.tn

    def acc(self):
        return (self.tp + self.tn) / self.n if self.n else None

    def bal(self):
        pos, neg = self.tp + self.fn, self.tn + self.fp
        if not pos or not neg:
            return None
        return 0.5 * (self.tp / pos + self.tn / neg)

    @staticmethod
    def _auc(pos, neg):
        P, N = sum(pos), sum(neg)
        if not P or not N:
            return None
        below = a = 0
        for p, q in zip(pos, neg):
            a += p * (below + 0.5 * q)
            below += q
        return a / (P * N)

    def auc(self):
        """Threshold-free discrimination on the 20 coarse bins (what the arms read as a fact type's rate)."""
        return self._auc(self.pos, self.neg)

    def aucf(self):
        """The reported AUC: the chance a random yes got a higher readout than a random no, on 200 fine bins of the
        raw readout (the coarse bins tie every rare prediction near P = 0 and pull the AUC toward 0.5)."""
        if self.posf is None:
            return self.auc()
        return self._auc(self.posf, self.negf)

    def merge(self, o):
        for s in ("tp", "fp", "fn", "tn", "surprise"):
            setattr(self, s, getattr(self, s) + getattr(o, s))
        self.pos = [a + b for a, b in zip(self.pos, o.pos)]
        self.neg = [a + b for a, b in zip(self.neg, o.neg)]
        if o.posf is not None:
            if self.posf is None:
                self.posf, self.negf = [0] * NBF, [0] * NBF
            self.posf = [a + b for a, b in zip(self.posf, o.posf)]
            self.negf = [a + b for a, b in zip(self.negf, o.negf)]

    def row(self):
        return dict(n=self.n, tp=self.tp, fp=self.fp, fn=self.fn, tn=self.tn, surprise=self.surprise,
                    acc=_r(self.acc()), bal=_r(self.bal()), auc=_r(self.aucf()), auc20=_r(self.auc()),
                    base=_r((self.tp + self.fn) / self.n if self.n else None))


def _r(v):
    return None if v is None else round(v, 4)


def understanding(conf):
    """(U, U_acc) of {type: Confusion}: macro AUC over types that saw both answers; macro raw accuracy."""
    bals = [c.aucf() for c in conf.values() if c.aucf() is not None]
    accs = [c.acc() for c in conf.values() if c.n >= 5]
    return (sum(bals) / len(bals) if bals else None), (sum(accs) / len(accs) if accs else None)


def prob(p):
    """A readout's output (target +-0.9) as a probability of yes."""
    return min(1.0, max(0.0, 0.5 + p / 1.8))


def one_hot(i, n):
    v = np.zeros(n)
    if 0 <= i < n:
        v[i] = 1.0
    return v


def background(g):
    vals, counts = np.unique(g, return_counts=True)
    return int(vals[np.argmax(counts)])


def objects(g, bg):
    """4-connected same-colour components that are not the background, smallest first (walls last), capped."""
    out = []
    big = 0.25 * g.size
    for c in [int(v) for v in np.unique(g) if v != bg]:
        for comp in muse._components(g == c):
            if len(comp) <= big:
                out.append((c, frozenset(comp)))
    out.sort(key=lambda o: (len(o[1]), min(o[1])))
    return out[:MAX_OBJECTS]


def shape_key(cells):
    y0 = min(p[0] for p in cells)
    x0 = min(p[1] for p in cells)
    return frozenset((y - y0, x - x0) for y, x in cells), (y0, x0)


class Verifier:
    """Predicts the facts of the next frame before each action, checks them after, learns from both."""

    def __init__(self, seed=0, replay_surprise=0):
        self.frames, self.G = [], None
        self.replay_surprise = int(replay_surprise)       # arm +4: extra learning passes on a refuted confident fact
        dims = dict(effect=N_ACT, death=N_ACT + N_AHEAD + N_LIFE, level=N_ACT, reset=N_ACT + N_AHEAD + N_LIFE, clock_tick=N_ACT,
                    novel=N_ACT + N_AHEAD, cell_change=N_CELL, self_moves=7 + N_AHEAD, obj_gone=N_OBJ,
                    obj_moved=N_OBJ, obj_recolour=N_OBJ, obj_reshape=N_OBJ)
        self.heads = {t: Head(d, seed + i) for i, (t, d) in enumerate(sorted(dims.items()))}
        self.level = 0
        self.conf = collections.defaultdict(lambda: collections.defaultdict(Confusion))   # level -> type -> C
        self.steps = collections.Counter()                 # level -> checked steps
        self.game = collections.defaultdict(Confusion)     # type -> Confusion over the whole game (the rates)
        self.img = collections.defaultdict(collections.Counter)   # level -> imagined-frame scores
        self.horizon = {}                                  # checked steps -> U and per-type AUC then
        self.win = collections.defaultdict(lambda: collections.defaultdict(Confusion))   # 50-step window -> type -> C
        self.info = collections.defaultdict(collections.Counter)   # 100-step bin -> information per action
        self.cur = collections.Counter()
        self.level_up_at = []                              # checked-step index of each level completion
        self.tau = float(os.environ.get("HY_TAU", "0.95"))
        self.pending = None
        self.frozen = False                                # a probe: readouts are read, never trained
        self.lite = bool(int(os.environ.get("HY_LITE", "0")))
        self._read_key, self._read_val, self._obj_cache = None, None, {}   # only the step facts and the self (the kit's cost)
        self.errors = 0
        self.replay = []                                   # (type, features, truth) of refuted confident facts
        # carried across levels: the self, its displacement per action, the value table
        self.self_votes = collections.Counter()
        self.lives = []                                    # past life lengths (steps from a level start or a death)
        self.life = 0
        self.life_start = None
        self.disp = collections.defaultdict(collections.Counter)      # action id -> Counter((dx, dy))
        self.value_game = collections.defaultdict(collections.Counter)  # old -> Counter(new)
        self._new_level()

    # ------------------------------------------------------------------------------------------- level state
    def _new_level(self):
        self.frames, self.G = [], None
        self.change_count, self.n_trans = None, 0
        self.last_diff = None
        self.start = None
        self.self_cells = None
        self.value_cell = collections.defaultdict(collections.Counter)  # (y, x, old) -> Counter(new)

    def grid_of(self, frame):
        f = np.asarray(frame, dtype=np.int64)
        if len(self.frames) < 12 and not any((f == x).all() for x in self.frames):
            self.frames.append(f)
            self.G = muse.Grid(self.frames)
        return self.read(f)

    def read(self, f):
        """`self.G.grid(f)`, cached for the last frame read under this grid (the same frame is read up to 3 times a
        step). The returned grid is shared: callers copy before writing."""
        key = (id(self.G), f.tobytes())
        if self._read_key != key:
            self._read_key, self._read_val = key, self.G.grid(f)
        return self._read_val

    def objects_of(self, g):
        """(background, objects) of g, cached for the last few grids (every candidate action reads the same grid)."""
        key = (g.shape, g.tobytes())
        hit = self._obj_cache.get(key)
        if hit is None:
            bg = background(g)
            hit = (bg, objects(g, bg))
            if len(self._obj_cache) >= 4:
                self._obj_cache.pop(next(iter(self._obj_cache)))
            self._obj_cache[key] = hit
        return hit

    def clock(self, shape):
        if self.change_count is None or self.change_count.shape != shape or self.n_trans < 8:
            return np.zeros(shape, bool)
        return self.change_count >= 0.6 * self.n_trans

    def active(self, shape):
        if self.change_count is None or self.change_count.shape != shape:
            return np.zeros(shape, bool)
        return self.change_count > 0

    @property
    def self_colour(self):
        if not self.self_votes:
            return None
        c, v = self.self_votes.most_common(1)[0]
        return c if v >= 2 else None

    def locate_self(self, g):
        """The self's cells in g: the component of the self's colour nearest its last known cells."""
        c = self.self_colour
        if c is None or not (g == c).any():
            return None
        comps = muse._components(g == c)
        if self.self_cells:
            cy = sum(p[0] for p in self.self_cells) / len(self.self_cells)
            cx = sum(p[1] for p in self.self_cells) / len(self.self_cells)
        else:
            cy, cx = g.shape[0] / 2, g.shape[1] / 2

        def d(comp):
            return min((p[0] - cy) ** 2 + (p[1] - cx) ** 2 for p in comp), -len(comp)
        return frozenset(min(comps, key=d))

    # ------------------------------------------------------------------------------------------- features
    def action_of(self, act):
        """(id, logical click cell or None) of a GameAction."""
        aid = int(getattr(act, "value", act))
        xy = None
        if aid == 6 and self.G is not None:
            d = act.action_data.model_dump() if hasattr(act, "action_data") else {}
            xy = self.G.cell_of_pixel(int(d.get("x", 0)), int(d.get("y", 0)))
        return aid, xy

    @staticmethod
    def act_features(aid, xy, g, comp_size):
        f = np.zeros(N_ACT)
        if 1 <= aid <= 7:
            f[aid - 1] = 1.0
        if xy is not None:
            x, y = xy
            if 0 <= y < g.shape[0] and 0 <= x < g.shape[1]:
                f[7 + int(g[y, x])] = 1.0
                s = comp_size.get((y, x), 1)
                f[23] = math.log1p(s) / 5.0
                f[24] = 1.0 if s == 1 else 0.0
        return f

    def mode_disp(self, aid):
        c = self.disp.get(aid)
        if not c:
            return None
        (d, n), = c.most_common(1)
        return d if n >= 1 else None

    def ahead(self, g, aid, cells):
        """The colour the self would enter under the action's usual displacement (one-hot), out of the grid, unknown."""
        f = np.zeros(N_AHEAD)
        d = self.mode_disp(aid) if cells else None
        if d is None or d == (0, 0):
            f[17] = 1.0
            return f, None
        dx, dy = d
        h, w = g.shape
        new = [(y + dy, x + dx) for y, x in cells]
        if any(not (0 <= y < h and 0 <= x < w) for y, x in new):
            f[16] = 1.0
            return f, None
        entered = collections.Counter(int(g[y, x]) for y, x in new if (y, x) not in cells)
        if entered:
            f[entered.most_common(1)[0][0]] = 1.0
        return f, frozenset(new)

    # ------------------------------------------------------------------------------------------- predict
    def expect(self, g, aid, xy, light=False):
        """Every fact predicted for (g, action): list of (type, key, features, p). p > 0 means yes. `light`: only the
        global facts and the self (what an agent asks for each candidate action)."""
        out = []
        bg, objs = self.objects_of(g)
        comp_size = {}
        clicked_obj = None
        for c, cells in objs:
            for p in cells:
                comp_size[p] = len(cells)
        if xy is not None:
            x, y = xy
            for o in objs:
                if (y, x) in o[1]:
                    clicked_obj = o
                    break
        fa = self.act_features(aid, xy, g, comp_size)
        aid_hot = one_hot(aid - 1, 7)
        cells_self = self.locate_self(g)
        fahead, new_self = self.ahead(g, aid, cells_self)
        flife = self.life_features(g)
        for t, f in (("effect", fa), ("death", np.concatenate([fa, fahead, flife])), ("level", fa),
                     ("reset", np.concatenate([fa, fahead, flife])), ("clock_tick", fa),
                     ("novel", np.concatenate([fa, fahead]))):
            out.append((t, None, f, self.heads[t].predict(f)))
        # the self
        moves_p = None
        if cells_self is not None:
            f = np.concatenate([aid_hot, fahead])
            moves_p = self.heads["self_moves"].predict(f)
            out.append(("self_moves", None, f, moves_p))
            d = self.mode_disp(aid) if moves_p > 0 and new_self is not None else (0, 0)
            pred_self = new_self if (moves_p > 0 and new_self is not None) else cells_self
            out.append(("self_pos", d, None, 1.0))
            out.append(("self_pos_base", (0, 0), None, 1.0))     # the baseline: the self stays
            touch = self._touches(g, pred_self, bg)
            out.append(("contact", None, None, 1.0 if touch else -1.0))
        else:
            pred_self = None
        if light:
            return out
        # active cells
        act = self.active(g.shape)
        clk = self.clock(g.shape)
        last = self.last_diff if self.last_diff is not None and self.last_diff.shape == g.shape else None
        rate = (self.change_count / max(1, self.n_trans)) if act.any() else None
        ys, xs = np.nonzero(act)
        for y, x in list(zip(ys.tolist(), xs.tolist()))[:MAX_CELLS]:
            nb = 0
            if last is not None:
                for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if 0 <= yy < g.shape[0] and 0 <= xx < g.shape[1] and last[yy, xx]:
                        nb += 1
            in_self = cells_self is not None and (y, x) in cells_self
            near_self = cells_self is not None and any(abs(y - p[0]) + abs(x - p[1]) == 1 for p in cells_self)
            f = np.concatenate([aid_hot, [rate[y, x], float(last is not None and last[y, x]), nb / 4.0,
                                          float(clk[y, x]), float(in_self), float(near_self),
                                          float(xy is not None and (y, x) == (xy[1], xy[0])),
                                          float(clicked_obj is not None and (y, x) in clicked_obj[1]),
                                          float(pred_self is not None and (y, x) in pred_self),
                                          float(int(g[y, x]) == bg)]])
            p = self.heads["cell_change"].predict(f)
            out.append(("cell_change", (y, x), f, p))
            out.append(("cell_value", (y, x), None, self._value(y, x, int(g[y, x]))))
        # objects
        sc = self.self_colour
        for c, cells in objs:
            if cells_self is not None and cells == cells_self:
                continue
            touched = pred_self is not None and bool(cells & pred_self)
            adj = pred_self is not None and any((y + dy, x + dx) in cells for y, x in pred_self
                                                 for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)))
            ch = last is not None and any(last[p] for p in cells)
            f = np.concatenate([aid_hot, [float(clicked_obj is not None and clicked_obj[1] == cells),
                                          float(clicked_obj is not None and clicked_obj[0] == c),
                                          float(touched), float(adj), math.log1p(len(cells)) / 5.0,
                                          float(len(cells) == 1), float(ch), float(c == sc)], one_hot(c, 16)])
            for t in ("obj_gone", "obj_moved", "obj_recolour", "obj_reshape"):
                out.append((t, (c, cells), f, self.heads[t].predict(f)))
        return out

    def life_features(self, g):
        f = np.zeros(N_LIFE)
        f[0] = math.log1p(self.life) / 7.0
        if self.lives:
            m = sum(self.lives[-8:]) / len(self.lives[-8:])
            f[1] = min(2.0, self.life / max(1.0, m))
            f[2] = 1.0 if self.life + 1 >= 0.9 * min(self.lives[-8:]) else 0.0
        clk = self.clock(g.shape)
        if clk.any() and self.life_start is not None and self.life_start.shape == g.shape:
            f[3] = float((clk & (g != self.life_start)).sum()) / float(clk.sum())
        else:
            f[4] = 1.0
        return f

    def _value(self, y, x, old):
        """The predicted new value of a cell if it changes (learned table: this cell's own history, then the game's)."""
        cnt = self.value_cell.get((y, x, old)) or self.value_game.get(old)
        return cnt.most_common(1)[0][0] if cnt else None

    @staticmethod
    def _touches(g, cells, bg):
        h, w = g.shape
        for y, x in cells:
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in cells and int(g[yy, xx]) != bg:
                    return True
        return False

    def before(self, latest, act):
        """Called with the frame the agent saw and the action it chose: predicts and keeps the facts."""
        self.pending = None
        if not getattr(latest, "frame", None) or int(getattr(act, "value", act)) == 0:
            return
        g = self.grid_of(latest.frame[-1])
        aid, xy = self.action_of(act)
        if self.start is None:
            self.start = g.copy()
        if self.life_start is None:
            self.life_start = g.copy()
        fs = self.expect(g, aid, xy, light=self.lite)
        self.pending = dict(frame=np.asarray(latest.frame[-1], dtype=np.int64), g=g, aid=aid, xy=xy,
                            G=self.G, facts=fs, cells_self=self.locate_self(g), img=self.imagine(g, fs))

    # ------------------------------------------------------------------------------------------- check
    def after(self, latest):
        """Called with the next frame: checks every pending fact, learns, updates the level state."""
        from arcengine import GameState
        p, self.pending = self.pending, None
        if p is None:
            return
        self.cur = collections.Counter()
        lv = int(getattr(latest, "levels_completed", 0) or 0)
        st = getattr(latest, "state", None)
        dead = st is GameState.GAME_OVER
        up = lv > self.level
        conf = self.conf[self.level]
        self.steps[self.level] += 1
        g0 = p["g"]
        g1 = None
        if not dead and not up and getattr(latest, "frame", None) and self.G is p["G"]:
            g1 = self.read(np.asarray(latest.frame[-1], dtype=np.int64))
            if g1.shape != g0.shape:
                g1 = None
        truth = dict(death=dead, level=up)
        if g1 is not None:
            d = g0 != g1
            clk = self.clock(g0.shape)
            act = self.active(g0.shape)
            truth["effect"] = bool((d & ~clk).any())
            truth["clock_tick"] = bool((d & clk).any())
            truth["novel"] = bool((d & ~act).any())
            st0 = self.start if self.start is not None and self.start.shape == g0.shape else None
            truth["reset"] = st0 is not None and bool(((g1 == st0) | clk).all()) and not bool(((g0 == st0) | clk).all())
        facts = p["facts"]
        for t, key, f, pr in facts:
            if t in ("death", "level"):
                self._score(conf, t, f, pr, truth[t])
            elif g1 is None:
                continue
            elif t in ("effect", "clock_tick", "novel", "reset"):
                self._score(conf, t, f, pr, truth[t])
            elif t == "cell_change":
                self._score(conf, t, f, pr, bool(d[key]))
            elif t == "cell_value":
                y, x = key
                if d[y, x]:
                    self._add(conf, t, pr is not None and pr == int(g1[y, x]), True, 1.0)
        if g1 is not None:
            img, unk, flags = p["img"]
            m = self.img[self.level]
            known_ok = bool((img[~unk] == g1[~unk]).all())
            m["n"] += 1
            m["known_right"] += known_ok                       # every committed cell right
            m["full_right"] += known_ok and not unk.any()      # the whole frame committed and right
            m["unknown_cells"] += int(unk.sum())
            m["cells"] += int(unk.size)
            m["wrong_cells"] += int((img[~unk] != g1[~unk]).sum())
            m["persist_right"] += bool((g0 == g1).all())       # the baseline: nothing changes
            m["novel_unknown"] += flags["novel"] is not False
            m["death_verdict"] += flags["death"] is not None
            m["death_verdict_right"] += flags["death"] is not None and flags["death"] == dead
            self._check_self_and_objects(conf, p, g0, g1, d, facts)
            # learn the level's dynamics (after scoring: predictions never see their own answer)
            if self.change_count is None or self.change_count.shape != d.shape:
                self.change_count, self.n_trans = np.zeros(d.shape), 0
            self.change_count += d
            self.n_trans += 1
            self.last_diff = d
            for y, x in zip(*np.nonzero(d)):
                old, new = int(g0[y, x]), int(g1[y, x])
                self.value_cell[(int(y), int(x), old)][new] += 1
                self.value_game[old][new] += 1
        for t, f, yes in ([] if self.frozen else self.replay[-8:]):   # arm +4: replay the latest refuted confident facts
            self.heads[t].learn(f, yes, times=self.replay_surprise)
        self.replay = []
        tot = sum(self.steps.values())
        c = self.cur
        c["steps"] += 1
        c["wasted"] += int(not c["refuted"] and c["err_drop"] < 1e-3 and not up)
        self.info[(tot - 1) // 100].update(c)
        if up:
            self.level_up_at.append(tot)
        if tot in (100, 300):                               # U at a fixed horizon: agents compared on equal data
            u, _ = understanding(self.game)
            self.horizon[str(tot)] = dict(U=_r(u), auc={t: _r(c.aucf()) for t, c in self.game.items()})
        self.life += 1
        if up or dead:
            if dead:
                self.lives.append(self.life)
            self.life, self.life_start = 0, None
        if up:
            self.level = lv
            self._new_level()
        elif dead:
            self.self_cells = None

    def _add(self, conf, t, pred, truth, c, p=None, raw=None):
        conf[t].add(pred, truth, c, p, raw)
        self.game[t].add(pred, truth, c, p, raw)
        self.win[sum(self.steps.values()) // WIN][t].add(pred, truth, c, p, raw)
        self.cur["predicted"] += 1
        self.cur["confirmed" if pred == truth else "refuted"] += 1

    def _score(self, conf, t, f, pr, yes):
        pred = pr > 0
        self._add(conf, t, pred, yes, abs(pr), prob(pr), pr)
        P = prob(pr)
        self.cur["bits"] += -math.log2(max(1e-3, P if yes else 1.0 - P))
        if self.frozen:
            return
        self.heads[t].learn(f, yes)
        tgt = 0.9 if yes else -0.9
        self.cur["err_drop"] += (tgt - pr) ** 2 - (tgt - self.heads[t].predict(f)) ** 2
        if self.replay_surprise and pred != yes and abs(pr) >= 0.5:
            self.replay.append((t, f, yes))

    def _check_self_and_objects(self, conf, p, g0, g1, d, facts):
        aid = p["aid"]
        # which colours moved rigidly: votes for the self (non-click actions only)
        moved = {}
        for c in [int(v) for v in np.unique(g0[d])] if d.any() else []:
            tr = muse.translation(g0, g1, c)
            if tr is not None:
                moved[c] = tr
        if aid != 6:
            for c in moved:
                self.self_votes[c] += 1
        cs = p["cells_self"]
        sc = self.self_colour
        if cs is not None and sc is not None:
            col = next(iter(g0[y, x] for y, x in cs))
            disp = moved.get(int(col), (0, 0)) if int(col) == sc else (0, 0)
            if int(col) == sc and disp == (0, 0) and any(d[y, x] for y, x in cs):
                disp = None                                 # changed but not rigidly: neither moved nor still
            for t, key, f, pr in facts:
                if t == "self_moves" and disp is not None:
                    self._score(conf, t, f, pr, disp != (0, 0))
                elif t in ("self_pos", "self_pos_base") and disp is not None:
                    self._add(conf, t, True, key == disp, 1.0)
                elif t == "contact" and disp is not None:
                    actual = frozenset((y + disp[1], x + disp[0]) for y, x in cs)
                    if all(0 <= y < g1.shape[0] and 0 <= x < g1.shape[1] for y, x in actual):
                        self._add(conf, t, pr > 0, self._touches(g1, actual, background(g1)), 1.0, prob(pr), pr)
            if disp is not None and disp != (0, 0) and aid != 6:
                self.disp[aid][disp] += 1
            if disp is not None:
                self.self_cells = frozenset((y + disp[1], x + disp[0]) for y, x in cs)
        elif cs is None and sc is not None:
            self.self_cells = None
        # objects (the new grid's components, once per colour)
        cache = {}

        def comps(c):
            if c not in cache:
                cache[c] = [frozenset(cp) for cp in muse._components(g1 == c)]
            return cache[c]
        for t, key, f, pr in facts:
            if not t.startswith("obj_"):
                continue
            c, cells = key
            self._score(conf, t, f, pr, self._obj_truth(t, c, cells, g0, g1, comps))

    @staticmethod
    def _obj_truth(t, c, cells, g0, g1, comps):
        keep = [p for p in cells if g1[p] == c]
        if t == "obj_recolour":
            vals = {int(g1[p]) for p in cells}
            return len(vals) == 1 and vals != {c}
        shape, (y0, x0) = shape_key(cells)
        if t in ("obj_moved", "obj_gone"):
            found_moved = False
            for comp in comps(c):
                if len(comp) != len(cells):
                    continue
                s2, (y1, x1) = shape_key(comp)
                if s2 == shape and (y1, x1) != (y0, x0):
                    found_moved = True
                    break
            if t == "obj_moved":
                return found_moved and len(keep) < len(cells)
            return not keep and not found_moved
        # obj_reshape: the component of colour c over the object's kept cells is another set, not a translation
        if not keep:
            return False
        if len(keep) == len(cells):
            comp = next((cp for cp in comps(c) if keep[0] in cp), None)
            return comp is not None and comp != cells
        return True

    # ------------------------------------------------------------------------------------------- rates and report
    def rate(self, t):
        """A fact type's verified rate over the game so far: its AUC (a learned type) or its accuracy (a composed
        one), None while it has fewer than 20 checks or (AUC) has not seen both answers."""
        c = self.game.get(t)
        if c is None or c.n < 20:
            return None
        return c.acc() if t in ("self_pos", "self_pos_base", "cell_value") else c.auc()

    def trusted(self, t):
        r = self.rate(t)
        return r is not None and r >= self.tau

    def verdict(self, t, p):
        """A fact's answer when it is TRUSTED, else None. Trust is verified, not assumed: predictions of type t that
        fell in the same probability bin (of 20) were right >= tau of the time over this game, with >= 10 checks."""
        c = self.game.get(t)
        if c is None:
            return None
        b = min(NB - 1, max(0, int(prob(p) * NB)))
        n = c.pos[b] + c.neg[b]
        if n < 10:
            return None
        if c.pos[b] >= self.tau * n:
            return True
        if c.neg[b] >= self.tau * n:
            return False
        return None

    def bin_rate(self, t, p):
        """The share of "yes" among this game's checked predictions of type t in p's probability bin (Laplace), or
        None under 10 checks: a verified probability, where `verdict` is a verified certainty."""
        c = self.game.get(t)
        if c is None:
            return None
        b = min(NB - 1, max(0, int(prob(p) * NB)))
        n = c.pos[b] + c.neg[b]
        return (c.pos[b] + 0.5) / (n + 1.0) if n >= 10 else None

    def imagine(self, g, facts):
        """The next frame composed from TRUSTED facts only: (grid, unknown mask, flags). A cell is committed only
        when every fact touching it has a verified answer (`verdict`); everything else is unknown."""
        img, unk = g.copy(), np.zeros(g.shape, bool)
        vals = {k: p for t, k, f, p in facts if t == "cell_value"}
        value_ok = (self.rate("cell_value") or 0) >= self.tau
        for t, key, f, p in facts:
            if t == "cell_change":
                v = self.verdict(t, p)
                if v is False:
                    continue
                if v is True and value_ok and vals.get(key) is not None:
                    img[key] = vals[key]
                else:
                    unk[key] = True
            elif t.startswith("obj_"):
                if self.verdict(t, p) is not False:
                    for q in key[1]:
                        unk[q] = True
        pr = {t: p for t, k, f, p in facts if k is None and f is not None}
        novel = self.verdict("novel", pr["novel"]) if "novel" in pr else None
        if novel is not False:
            unk |= ~self.active(g.shape)                    # a cell that never changed may change: unknown
        moves = pr.get("self_moves")
        disp = next((k for t, k, f, p in facts if t == "self_pos"), None)
        cs = self.locate_self(g)
        if cs is not None and moves is not None:
            v = self.verdict("self_moves", moves)
            pos_ok = (self.rate("self_pos") or 0) >= self.tau
            if v is True and pos_ok and disp not in (None, (0, 0)):
                dx, dy = disp
                new = {(y + dy, x + dx) for y, x in cs}
                if all(0 <= y < g.shape[0] and 0 <= x < g.shape[1] for y, x in new):
                    c = g[next(iter(cs))]
                    for q in cs:
                        if q not in new:
                            unk[q] = True                   # what the self uncovers is not predicted
                    for q in new:
                        img[q], unk[q] = c, False
                else:
                    v = None
            if v is None or (v is True and not pos_ok):
                for y, x in cs:
                    for yy, xx in ((y, x), (y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                        if 0 <= yy < g.shape[0] and 0 <= xx < g.shape[1]:
                            unk[yy, xx] = True
        flags = dict(novel=novel, death=self.verdict("death", pr.get("death", 0.0)),
                     level=self.verdict("level", pr.get("level", 0.0)))
        return img, unk, flags

    def report(self):
        tot = collections.defaultdict(Confusion)
        levels = {}
        for lv, cs in sorted(self.conf.items()):
            for t, c in cs.items():
                tot[t].merge(c)
            u, ua = understanding(cs)
            levels[str(lv)] = dict(U=_r(u), U_acc=_r(ua), steps=self.steps[lv], imagine=dict(self.img[lv]),
                                   types={t: c.row() for t, c in sorted(cs.items())})
        u, ua = understanding(tot)
        im = collections.Counter()
        for c in self.img.values():
            im.update(c)
        return dict(U=_r(u), U_acc=_r(ua), steps=sum(self.steps.values()), self_colour=self.self_colour,
                    imagine=dict(im), tau=self.tau, horizon=self.horizon, level_up_at=self.level_up_at,
                    info={str(b): {k: round(v, 3) for k, v in c.items()} for b, c in sorted(self.info.items())},
                    windows={str(w): dict(U=_r(understanding(cs)[0]),
                                          auc={t: _r(c.aucf()) for t, c in cs.items() if c.aucf() is not None})
                             for w, cs in sorted(self.win.items())},
                    types={t: c.row() for t, c in sorted(tot.items())}, levels=levels)


def probe(v, path, steps=300):
    """Score v's readouts, frozen, on a shared reference trajectory (`path`, an .npz written by `Verified` with
    HY_RECORD): the understanding score of what v learned, on the SAME transitions for every agent. v is not changed:
    the probe has its own level state and copies of v's tables; its readouts are v's, never trained."""
    import copy
    from arcengine import GameAction, GameState
    z = np.load(path)
    q = Verifier()
    q.heads, q.frozen = v.heads, True
    q.self_votes, q.disp = copy.deepcopy(v.self_votes), copy.deepcopy(v.disp)
    q.value_game, q.lives = copy.deepcopy(v.value_game), list(v.lives)

    class F:
        pass
    n = min(steps, len(z["actions"]))
    for i in range(n):
        f = F()
        f.frame, f.state, f.levels_completed = [z["frames"][i]], GameState[str(z["states"][i])], int(z["levels"][i])
        q.after(f)
        a = GameAction.from_id(int(z["actions"][i]))
        if int(z["actions"][i]) == 6:
            a.set_data({"x": int(z["xy"][i][0]), "y": int(z["xy"][i][1])})
        if f.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            q.pending = None
            continue
        q.before(f, a)
    u, ua = understanding(q.game)
    return dict(U=_r(u), U_acc=_r(ua), steps=n, auc={t: _r(c.aucf()) for t, c in q.game.items()})


class Verified:
    """Wraps a policy (`choose_action(frames, latest)` / `is_done`) with a Verifier that observes every step. The
    Verifier is exposed as `.agent` with `stats()` and `carry()`, the hooks the gate's `replay.py` reads into its row."""

    def __init__(self, policy, verifier, game_id=""):
        self.policy, self.v, self.game_id = policy, verifier, game_id
        self.agent = self
        self.record = os.environ.get("HY_RECORD", "")         # write this game's trajectory (the probe set)
        self.probe_dir = os.environ.get("HY_PROBE", "")       # score the readouts on a recorded trajectory
        self.tape = []
        self.probes = {}

    def is_done(self, frames, latest):
        return self.policy.is_done(frames, latest)

    def choose_action(self, frames, latest):
        try:
            self.v.after(latest)
        except Exception as e:  # the observer must never break the agent
            self._error(e)
        if self.probe_dir and sum(self.v.steps.values()) == 300 and "300" not in self.probes:
            self._probe("300")
        act = self.policy.choose_action(frames, latest)
        if self.record and getattr(latest, "frame", None):
            d = act.action_data.model_dump() if hasattr(act, "action_data") else {}
            self.tape.append((np.asarray(latest.frame[-1], dtype=np.int8), latest.state.name,
                              int(latest.levels_completed or 0), int(act.value), (int(d.get("x", 0)), int(d.get("y", 0)))))
        try:
            self.v.before(latest, act)
        except Exception as e:
            self._error(e)
        return act

    def _error(self, e):
        import traceback
        self.v.pending = None
        self.v.errors += 1
        self.v.last_error = traceback.format_exc()[-600:]

    def _probe(self, key):
        path = os.path.join(self.probe_dir, f"{self.game_id}.npz")
        if os.path.exists(path):
            try:
                self.probes[key] = probe(self.v, path)
            except Exception as e:
                self._error(e)

    def stats(self):
        if self.record and self.tape:
            os.makedirs(self.record, exist_ok=True)
            fr, st, lv, ac, xy = zip(*self.tape)
            np.savez_compressed(os.path.join(self.record, f"{self.game_id}.npz"), frames=np.stack(fr),
                                states=np.array(st), levels=np.array(lv), actions=np.array(ac), xy=np.array(xy))
        if self.probe_dir:
            self._probe("end")
        r = self.v.report()
        r["probe"] = self.probes
        if hasattr(self.policy, "hy_stats"):
            r["hy"] = dict(self.policy.hy_stats)
        if getattr(self.policy, "trace", None):
            r["hy_trace"] = self.policy.trace
        r["errors"] = self.v.errors
        if getattr(self.v, "last_error", None):
            r["last_error"] = self.v.last_error
        return r

    def carry(self):
        return json.dumps({})


In [ ]:
%%writefile /kaggle/working/hy_arms.py
# SPDX-License-Identifier: MIT-0
"""Kit HY's acting arms: kit X's explore v3 with one added term in its choice, read from the fact verifier.

`HY_ARM` (the gate rows say which arm ran):
  B     explore v3 unchanged (the verifier only observes);
  1     + TRUSTED facts gate the choice: an action whose death verdict is a verified yes is skipped while another is
        left, and an action predicted to change a never-changed cell gains `HY_W1` x P(novel), only once the novel
        fact's verified rate (AUC) is >= 0.75;
  13    + 1 + EXPERIMENTS: each candidate gains `HY_W3` x its expected information, the sum over the facts that touch
        the self, the counters, the effect and death of H(P) x (1 - skill), where H is the binary entropy of the
        predicted probability and skill = (AUC - 0.5) / 0.5 of that fact type over the game so far (0 while unrated),
        decaying as 1 / (1 + steps / 500): early in a game and fewer as the rates rise;
  134   + 13 + learn fastest from SURPRISE: a refuted confident fact is replayed `HY_REPLAY` (default 3) more passes.
AUTHORED: the arm rules above (a death is bad, a never-changed cell changing is worth seeing, the facts that matter),
the weights and the decay. LEARNED: every probability and every rate. Nothing names a game, a colour or a mechanic.
"""
import collections
import math
import os

import numpy as np

import explore
import facts
from explore import features


def entropy(P):
    P = min(1 - 1e-6, max(1e-6, P))
    return -(P * math.log2(P) + (1 - P) * math.log2(1 - P))


MATTER = ("effect", "self_moves", "death", "clock_tick", "novel")


class HyExplore(explore.ExploreAgent):
    """Explore v3, whose final scoring line adds `self.bonus(a)`; every other line is v3's."""

    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed)
        self.v, self.arm = verifier, arm
        self.w1 = float(os.environ.get("HY_W1", "0.5"))
        self.w3 = float(os.environ.get("HY_W3", "0.5"))
        self.n_steps = 0
        self._gv = None

    def bonus(self, a, cache):
        if self.arm == "B" or self._gv is None:
            return 0.0
        if a in cache:
            return cache[a]
        if " " in a:
            _, x, y = a.split()
            aid, xy = 6, (int(x), int(y))
        else:
            aid, xy = int(a), None
        if self._gv.shape != self.G.grid(self._frame).shape:
            return 0.0
        fs = {t: p for t, k, f, p in self.v.expect(self._gv, aid, xy, light=True) if f is not None}
        b = 0.0
        if self.v.verdict("death", fs["death"]) is True:
            b -= 5.0
        r = self.v.rate("novel")
        if r is not None and r >= 0.75:
            b += self.w1 * facts.prob(fs["novel"])
        if "3" in self.arm:
            info = 0.0
            for t in MATTER:
                if t in fs:
                    r = self.v.rate(t)
                    skill = 0.0 if r is None else max(0.0, (r - 0.5) / 0.5)
                    info += entropy(facts.prob(fs[t])) * (1.0 - skill)
            b += self.w3 * info / (1.0 + self.n_steps / 500.0)
        cache[a] = b
        return b

    def choose_action(self, frames, latest):
        self.n_steps += 1
        self._gv = None
        if self.v is not None and latest.frame:
            try:
                self._frame = np.asarray(latest.frame[-1], dtype=np.int64)
                self._gv = self.v.grid_of(latest.frame[-1])
            except Exception:
                self._gv = None
        return self._choose(frames, latest)

    # ---- v3's choose_action, with `+ self.bonus(a, cache)` in the score (marked HY)
    def _choose(self, frames, latest):
        import collections
        import muse
        from arcengine import GameAction, GameState
        from play import click_cells
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            if self.prev is not None and latest.state is GameState.GAME_OVER:
                g0, a0, f0 = self.prev
                self.model.learn(f0, -0.9)
                self.stats["deaths"] += 1
            self.prev = None
            return GameAction.RESET
        g = self.grid_of(latest.frame[-1])
        lv = int(latest.levels_completed or 0)
        if self.prev is not None:
            g0, a0, f0 = self.prev
            if lv > self.level:
                self.model.learn(f0, 0.9)
                for i, f in enumerate(reversed((self.recent + [f0])[-20:])):
                    self.progress.learn(f, 0.9 * (0.85 ** i))
                self.recent = []
                self.level = lv
                self.frames, self.G, self.change_count = [], None, None
                self.tried = collections.defaultdict(set)
                self.graph, self.acts_of, self.route = collections.defaultdict(dict), {}, []
                g = self.grid_of(latest.frame[-1])
                self.stats["levels"] += 1
            else:
                eff = self.effect(g0, g)
                self.model.learn(f0, 0.9 if eff else -0.9)
                self.stats["effective" if eff else "no_effect"] += 1
                self.graph[self.key(g0)][a0] = self.key(g)
        avail = sorted(set(int(v) for v in (latest.available_actions or [])))
        acts = [str(a) for a in avail if a not in (0, 6)]
        comp_of = {}
        if 6 in avail:
            cells = click_cells(g, cap=32)
            for x, y in cells:
                c = int(g[y, x])
                size = next((len(comp) for comp in muse._components(g == c) if (y, x) in comp), 1)
                comp_of[(x, y)] = size
            acts += [f"6 {x} {y}" for x, y in cells]
        key = self.key(g)
        self.acts_of[key] = acts
        untried = [a for a in acts if a not in self.tried[key]]
        cache = {}
        if self.arm != "B":                                       # HY: a verified-death action is not "untried"
            safe = [a for a in untried if self.bonus(a, cache) > -4.0]
            untried = safe if safe else untried
        if self.frontier and not untried:
            if not self.route:
                self.route = self.route_to_frontier(key)
            if self.route:
                a = self.route.pop(0)
                if a in acts:
                    self.stats["route_steps"] += 1
                    self.prev = (g, a, features(a, g, comp_of))
                    return self.to_action(a)
                self.route = []
        else:
            self.route = []
        pool = untried or acts
        scored = [(self.model.predict(features(a, g, comp_of))
                   + self.w_progress * self.progress.predict(features(a, g, comp_of))
                   + (0 if untried else self.rng.random() * 0.3)
                   + self.bonus(a, cache), -i, a)                 # HY
                  for i, a in enumerate(pool)]
        a = max(scored)[2]
        self.tried[key].add(a)
        self.prev = (g, a, features(a, g, comp_of))
        self.recent.append(self.prev[2])
        return self.to_action(a)


# ---------------------------------------------------------------------------------------------------------------------
# The arms on TK's planner (kit TK, `think.ThinkAgent`): B = kit TK itself. The search reads `predict_next(...)`'s
# p_new and p_death; for the CURRENT state only (the verifier has facts for the grid it sees, not for remembered
# states) the arms replace TK's per-type rates with the verifier's facts:
#   1    a TRUSTED verdict wins where one exists: effect verified "no" -> p_new 0 (nothing changes, so no new state);
#        novel verified "yes" -> p_new 1; death verified "yes" -> p_death 1, "no" -> p_death min(TK's, 1 - tau);
#        otherwise TK's rates;
#   13   + the expected information (as for explore, MATTER facts, decaying with steps) added to p_new;
#   134  + surprise replay.
#   1v   as 1, but death is not a verdict: p_death = the death rate VERIFIED in this prediction's probability bin over
#        the game (Laplace, >= 10 checks), else TK's. A "no" verdict on a rare fact says nothing beyond its base rate
#        (sp80: every one of 21 deaths fell under a trusted "no"), so arm 1 made risky moves look cheap.
# AUTHORED: these substitutions. LEARNED: every probability, rate and verdict.
# ---------------------------------------------------------------------------------------------------------------------
import think  # noqa: E402


class HyThink(think.ThinkAgent):
    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed)
        self.v, self.arm = verifier, arm
        self.w3 = float(os.environ.get("HY_W3", "0.5"))
        self._plan_key, self._gv, self._cache = None, None, {}
        self.hy_stats = collections.Counter()
        self.trace = [] if os.environ.get("HY_TRACE") else None   # (step, action, p_new TK -> HY, p_death TK -> HY)

    def choose_action(self, frames, latest):
        self._gv, self._cache = None, {}
        if self.v is not None and latest.frame:
            try:
                self._gv = self.v.grid_of(latest.frame[-1])
            except Exception:
                self._gv = None
        return super().choose_action(frames, latest)

    def plan(self, key, acts_here, g, comp_of):
        self._plan_key = key if (self._gv is not None and not self.raw and g.shape == self._gv.shape) else None
        return super().plan(key, acts_here, g, comp_of)

    def facts_for(self, a):
        if a in self._cache:
            return self._cache[a]
        if " " in a:
            _, x, y = a.split()
            aid, xy = 6, (int(x), int(y))
        else:
            aid, xy = int(a), None
        fs = {t: p for t, k, f, p in self.v.expect(self._gv, aid, xy, light=True) if f is not None}
        self._cache[a] = fs
        return fs

    def predict_next(self, n, a, t, par, graph=True):
        pred = super().predict_next(n, a, t, par, graph)
        if self.arm == "B" or self.v is None or n != self._plan_key or pred["source"] != "rates":
            return pred
        fs = self.facts_for(a)
        v = self.v
        p_new, p_death = pred["p_new"], pred["p_death"]
        if v.verdict("effect", fs["effect"]) is False:
            p_new = 0.0
            self.hy_stats["effect_no"] += 1
        elif v.verdict("novel", fs["novel"]) is True:
            p_new = 1.0
            self.hy_stats["novel_yes"] += 1
        dv = v.verdict("death", fs["death"])
        if "v" in self.arm:                                       # arm 1v: the verified rate of this bin, not a verdict
            r = v.bin_rate("death", fs["death"])
            if r is not None:
                p_death = r
                self.hy_stats["death_bin"] += 1
        elif dv is True:
            p_death = 1.0
            self.hy_stats["death_yes"] += 1
        elif dv is False:
            p_death = min(p_death, 1.0 - v.tau)
            self.hy_stats["death_no"] += 1
        if "3" in self.arm:
            info = 0.0
            for ft in MATTER:
                if ft in fs:
                    r = v.rate(ft)
                    skill = 0.0 if r is None else max(0.0, (r - 0.5) / 0.5)
                    info += entropy(facts.prob(fs[ft])) * (1.0 - skill)
            p_new = p_new + self.w3 * info / (1.0 + self.n_steps / 500.0)
        if self.trace is not None and len(self.trace) < 4000 and (p_new, p_death) != (pred["p_new"], pred["p_death"]):
            self.trace.append((self.n_steps, a, round(pred["p_new"], 3), round(p_new, 3), round(pred["p_death"], 3),
                               round(p_death, 3)))
        return dict(pred, p_new=p_new, p_death=p_death)


In [ ]:
%%writefile /kaggle/working/bs.py
# SPDX-License-Identifier: MIT-0
"""Stream BS's effect-graph pieces as switches over kit HV's planner (`hy_arms.HyThink`, itself kit TK's `think.
ThinkAgent` over kit X's explore v3). A port of `crates/openmorph-perception/src/effect_graph.rs` (PR #813) to the
logical grid. Both switches are off by default; off, `BsThink` plays exactly as `HyThink`.

* `BS_KEY=1` -- EFFECT-EQUIVALENT STATES. The state key also paints out the cells that tests show never change what
  an action does:
  - TICK TEST: a change kind (a cell's old and new colour) that was a transition's only change under two simple
    actions, or (click-only) that appeared in every one of >= 8 transitions of the level and once alone: every cell
    it changes, grown to its same-colour run (and its bounding box when small), is painted out;
  - EFFECT TEST: two grids that differ in a few cells, under the same action, changed the same cells the same way
    elsewhere (and changed something): a vote for painting those cells out. A disagreement on a one-part region votes
    against. A cell is painted out with support from two action classes and more than twice its contradictions, or
    four non-empty agreements and no contradiction. Votes are judged against the ticks (a tick is no effect).
  When the painted set grows, every state-keyed structure of the planner is remapped to the new keys.
  The painted cells are not an effect either (v3's effect head and TK's outcome rates read `effect`).
* `BS_CLICK=1` -- CLICK PROBING BY EFFECT. Click candidates: a cell of every object (no 32 cap) and a coarse grid of
  cells; a click that changed nothing on a colour is presumed inert where that colour is under it again, and every
  click on a colour inert at 3 places and effective nowhere; two clicks that took one state to the same next state
  are one button (only its first is offered).

AUTHORED (labelled, as the rules ask): the tests, the vote rule, the thresholds, the grid spacing. LEARNED per game:
the painted cells, the buttons, the inert colours. Nothing names a game, a colour or a mechanic.
"""
import collections
import hashlib
import os

import numpy as np

import hy_arms as hy

SOLO = 4            # non-empty agreements that paint a cell under one action class, with no contradiction
EVERY = 8           # click-only: transitions of a level before a kind seen in all of them is a tick
INERT_COLOUR = 3    # places where clicks on one colour changed nothing before that colour is presumed inert


def _neighbours(p, w, h):
    y, x = divmod(p, w)
    if x > 0:
        yield p - 1
    if x + 1 < w:
        yield p + 1
    if y > 0:
        yield p - w
    if y + 1 < h:
        yield p + w


def _components(cells, w, h):
    cells = set(cells)
    seen, out = set(), []
    for p in sorted(cells):
        if p in seen:
            continue
        comp, stack = [p], [p]
        seen.add(p)
        while stack:
            c = stack.pop()
            for n in _neighbours(c, w, h):
                if n in cells and n not in seen:
                    seen.add(n)
                    comp.append(n)
                    stack.append(n)
        out.append(comp)
    return out


def _flood(f, p, w, h, cap):
    col = f[p]
    seen, stack = {p}, [p]
    while stack:
        c = stack.pop()
        for n in _neighbours(c, w, h):
            if n not in seen and f[n] == col:
                seen.add(n)
                if len(seen) > cap:
                    return None
                stack.append(n)
    return seen


class LifeClock:
    """Stream HY's life-periodic filter (from `Verifier.time_log` / `time_mask`, same rule and
    defaults): cells driven by life time, not by actions -- across >= `lives` finished lives of this level, each changed
    at exactly the same life-relative steps in every life since it first changed (a budget bar). A fading floor changes
    where and when the self has been, so it fails the test unless the agent replays the same path every life.
    AUTHORED rule; the cells are learned per level."""

    def __init__(self, lives=3, min_cells=4):
        self.lives, self.min_cells = lives, min_cells
        self.reset()

    def reset(self):                        # at each level-up
        self.log, self.life, self.step, self._cache = {}, 0, 0, (None, None)

    def observe(self, g0, g1):              # a transition within a life (not the death, not the level-up)
        if g0.shape == g1.shape:
            for y, x in zip(*np.nonzero(g0 != g1)):
                self.log.setdefault((int(y), int(x)), {}).setdefault(self.life, []).append(self.step)
        self.step += 1

    def end_life(self):                     # a death: the next life starts its step count at 0
        self.life += 1
        self.step = 0

    def mask(self, shape):
        key = (shape, self.life)
        if self._cache[0] == key:
            return self._cache[1]
        m = np.zeros(shape, bool)
        for (y, x), per in self.log.items():
            if not (0 <= y < shape[0] and 0 <= x < shape[1]):
                continue
            done = {li: ts for li, ts in per.items() if li < self.life}      # finished lives only
            if len(done) < self.lives:
                continue
            first = min(done)
            if any(li not in done for li in range(first, self.life)):        # it must change in every life since
                continue
            if len({tuple(ts) for ts in done.values()}) == 1:                # at the same steps of each life
                m[y, x] = True
        if m.sum() < self.min_cells:                                         # a bar is many cells
            m[:] = False
        self._cache = (key, m)
        return m


class BsThink(hy.HyThink):
    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed, verifier=verifier, arm=arm)
        self.bs_key = bool(int(os.environ.get("BS_KEY", "0")))
        self.bs_click = bool(int(os.environ.get("BS_CLICK", "0")))
        # which key tests: "all" (the tick test and the effect test), "tick" (the tick test only), "tick+life" (a
        # ticking cell painted out only once HY's LifeClock also finds it life-periodic), "tick+chain" (that only for
        # cells painted by a colour chain a->b->c, plain ticks as "tick")
        self.bs_key_mode = os.environ.get("BS_KEY_MODE", "all")
        self.bs_life = LifeClock()
        # which candidates: "all" (every object + a coarse grid), "nogrid" (every object), "cap" (v3's first 32
        # objects); the inert and button filters apply to each
        self.bs_click_mode = os.environ.get("BS_CLICK_MODE", "all")
        # which inert rule: "both" (the position on a colour, and a colour inert at 3 places), "pos" (the position on
        # a colour only), "none"; "soft" / "softpos": as "both" / "pos", but at a state whose other clicks were all
        # tried, the presumed-inert ones are offered again
        self.bs_click_inert = os.environ.get("BS_CLICK_INERT", "both")
        self._bs_reset(None)
        self.bs_stats = collections.Counter()

    # ---- per-shape state (a new level or the raw reading may change the grid's shape) ----------------------------
    def _bs_reset(self, shape):
        self.bs_shape = shape
        n = 0 if shape is None else int(np.prod(shape))
        self.bs_mask = np.zeros(n, bool)
        self.bs_tick = np.zeros(n, bool)
        self.bs_tick_kinds = collections.defaultdict(set)   # cell -> the tick kinds that painted it
        self.bs_votes = []                          # (cells, class bits, support?, effect cells)
        self.bs_alone = collections.defaultdict(int)
        self.bs_ticks = set()
        self.bs_kind_n = collections.Counter()
        self.bs_at_click = set()                    # kinds seen at or next to the clicked cell
        self.bs_kind_clicks = collections.defaultdict(set)   # kind -> the clicks it appeared under (up to 3)
        self.bs_trans = []                          # this shape's transitions: (flat g0, action, flat g1)
        self.bs_by_act = collections.defaultdict(list)
        self.bs_key_grid = {}                       # key -> the grid that made it
        # clicks
        self.bs_inert = set()                       # (action, colour)
        self.bs_col_inert = collections.defaultdict(set)
        self.bs_col_eff = set()
        self.bs_group = {}                          # click -> its button's first click

    def _bs_shape(self, shape):
        if shape != self.bs_shape:
            self._bs_reset(shape)

    # ---- the key ---------------------------------------------------------------------------------------------------
    def key(self, g):
        if not self.bs_key:
            return super().key(g)
        self._bs_shape(g.shape)
        h = g.copy()
        h[self.clock_mask(g.shape)] = -1
        h.reshape(-1)[self.bs_mask] = -1
        k = hashlib.blake2b(h.tobytes(), digest_size=8).hexdigest()
        if k not in self.bs_key_grid:
            self.bs_key_grid[k] = g.copy()
        return k

    # ---- the hook: every non-terminal transition passes through `effect(g0, g1)` with `self.prev` its action ----------
    def effect(self, a, b):
        e = super().effect(a, b)
        if (not self.bs_key and not self.bs_click) or a.shape != b.shape or self.prev is None:
            return e
        act = self.prev[1]
        self._bs_shape(a.shape)
        if getattr(self, "_bs_level", None) != self.level:
            # a new level: its transitions and clicks are its own (the painted cells and the votes are kept)
            self._bs_level = self.level
            self.bs_trans, self.bs_by_act = [], collections.defaultdict(list)
            self.bs_inert, self.bs_col_inert, self.bs_col_eff, self.bs_group = (
                set(), collections.defaultdict(set), set(), {})
            self.bs_kind_n = collections.Counter()
            self.bs_at_click = set()
            self.bs_kind_clicks = collections.defaultdict(set)
            self.bs_life.reset()
        d = (a != b).reshape(-1)
        outside = d & ~self.clock_mask(a.shape).reshape(-1)
        if self.bs_key:
            outside &= ~self.bs_mask
        changed = bool(outside.any())
        if self.bs_click and " " in act:
            self._note_click(a, act, changed)
        if self.bs_key:
            if self.bs_key_mode in ("tick+life", "tick+chain"):
                self.bs_life.observe(a, b)
            f0, f1 = a.reshape(-1).copy(), b.reshape(-1).copy()
            self.bs_trans.append((f0, act, f1))
            self.bs_by_act[act].append(len(self.bs_trans) - 1)
            grew = self._ticks(f0, act, f1)
            if self.bs_key_mode != "tick":
                grew |= self._pair(len(self.bs_trans) - 1)
            if grew:
                self._remap()
            return changed
        return e

    # ---- clicks ------------------------------------------------------------------------------------------------------
    def _note_click(self, g, act, changed):
        _, x, y = act.split()
        c = int(g[int(y), int(x)])
        if changed:
            self.bs_col_eff.add(c)
        else:
            self.bs_inert.add((act, c))
            self.bs_col_inert[c].add(act)

    def _presumed_inert(self, g, act):
        _, x, y = act.split()
        c = int(g[int(y), int(x)])
        if self.bs_click_inert == "none":
            return False
        if self.bs_click_inert in ("pos", "softpos"):
            return (act, c) in self.bs_inert
        return (act, c) in self.bs_inert or (c not in self.bs_col_eff and len(self.bs_col_inert[c]) >= INERT_COLOUR)

    def cells(self, g, cap=32):
        """Click candidates: every object's cell (no cap), then a coarse grid, without presumed-inert clicks and
        without a button's other clicks."""
        import muse
        self._bs_shape(g.shape) if self.bs_key else None
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
        objs = []
        for c in [int(v) for v in vals if v != bg]:
            for comp in muse._components(g == c):
                my = sum(p[0] for p in comp) / len(comp)
                mx = sum(p[1] for p in comp) / len(comp)
                y, x = min(comp, key=lambda p: ((p[0] - my) ** 2 + (p[1] - mx) ** 2, p))
                objs.append((x, y))
        objs.sort(key=lambda p: (p[1], p[0]))
        if self.bs_click_mode == "cap":
            objs = objs[:cap]
        h, w = g.shape
        s = max(1, min(h, w) // 4)
        grid = [] if self.bs_click_mode != "all" else [
            (x, y) for y in range(s // 2, h, s) for x in range(s // 2, w, s)]
        out, kept, seen = [], [], set()
        for x, y in objs + grid:
            if (x, y) in seen:
                continue
            seen.add((x, y))
            act = f"6 {x} {y}"
            if self.bs_group.get(act, act) != act:
                continue
            kept.append((x, y))
            if self._presumed_inert(g, act):
                continue
            out.append((x, y))
        if self.bs_click_inert.startswith("soft"):
            # soft: presumed-inert clicks come back at a state once its other clicks were all tried there
            tried = self.tried.get(self.key(g), set())
            if all(f"6 {x} {y}" in tried for x, y in out):
                return kept
        return out

    def _think(self, latest):
        if not self.bs_click:
            return super()._think(latest)
        import play
        orig = play.click_cells
        play.click_cells = lambda g, cap=32: self.cells(g, cap)
        try:
            return super()._think(latest)
        finally:
            play.click_cells = orig

    def observe(self, k_now, died, leveled):
        # a button: two clicks that took one state to the same next state
        if self.bs_click and self.pend is not None and not died and not leveled and k_now is not None:
            k0, a = self.pend[0], self.pend[1]
            if " " in a and k_now != k0 and a not in self.bs_group:
                other = next((b for b, n in self.graph.get(k0, {}).items() if b != a and " " in b and n == k_now),
                             None)
                self.bs_group[a] = self.bs_group.get(other, other) if other else a
        super().observe(k_now, died, leveled)

    # ---- the tick test -----------------------------------------------------------------------------------------------
    @staticmethod
    def _class(act):
        if " " in act:
            return 1 << (8 + int(hashlib.blake2b(act.encode(), digest_size=2).hexdigest(), 16) % 8)
        return 1 << min(int(act), 7)

    def _ticks(self, f0, act, f1):
        idx = np.nonzero(f0 != f1)[0]
        kinds = {(int(f0[p]), int(f1[p])) for p in idx}
        for k in kinds:
            self.bs_kind_n[k] += 1
        for k in kinds:
            if " " in act and len(self.bs_kind_clicks[k]) < 3:
                self.bs_kind_clicks[k].add(act)
        if " " in act:
            # a kind that changed the clicked cell or a cell next to it is the click's own effect, never a clock
            _, x, y = act.split()
            h, w = self.bs_shape
            c = int(y) * w + int(x)
            near = {c} | set(_neighbours(c, w, h)) if c < h * w else set()
            for p in idx:
                if int(p) in near:
                    self.bs_at_click.add((int(f0[p]), int(f1[p])))
        if len(kinds) == 1:
            k = next(iter(kinds))
            self.bs_alone[k] |= 1 if " " in act else self._class(act)
        n = len(self.bs_trans)
        only_clicks = all(" " in a for a in self.bs_by_act)
        new = [k for k in kinds if k not in self.bs_ticks and (
            bin(self.bs_alone[k] & ~1).count("1") >= 2
            or (only_clicks and self.bs_alone[k] and n >= EVERY and self.bs_kind_n[k] == n
                and k not in self.bs_at_click and len(self.bs_kind_clicks[k]) >= 3))]
        span = range(n) if new else range(n - 1, n)
        self.bs_ticks.update(new)
        h, w = self.bs_shape
        cap = max(1, h * w // 16)
        bg = int(np.bincount(f0).argmax())
        grew = False
        for i in span:
            g0, _, g1 = self.bs_trans[i]
            seed = [int(p) for p in np.nonzero(g0 != g1)[0]
                    if (int(g0[p]), int(g1[p])) in self.bs_ticks and not self.bs_tick[p]]
            kinds_here = {(int(g0[p]), int(g1[p])) for p in seed}
            for comp in _components(seed, w, h):
                for p in self._grow(comp, (g0, g1), cap, bg):
                    self.bs_tick_kinds[p] |= kinds_here
                    if not self.bs_tick[p]:
                        self.bs_tick[p] = True
                        grew = True
        if grew:
            self._revote()
        return grew and self._recompute()

    def _grow(self, seed, frames, cap, bg):
        h, w = self.bs_shape
        region = set(seed)
        for p in seed:
            for f in frames:
                if f[p] != bg:
                    run = _flood(f, p, w, h, cap)
                    if run is not None:
                        region |= run
        ys, xs = zip(*(divmod(p, w) for p in region))
        if (max(xs) - min(xs) + 1) * (max(ys) - min(ys) + 1) <= cap:
            region = {y * w + x for y in range(min(ys), max(ys) + 1) for x in range(min(xs), max(xs) + 1)}
        elif len(region) > cap:
            region = set(seed)
        return region

    # ---- the effect test ---------------------------------------------------------------------------------------------
    def _pair(self, ti):
        g, act, g2 = self.bs_trans[ti]
        h, w = self.bs_shape
        limit = max(4, h * w // 128)
        clock = self.clock_mask(self.bs_shape).reshape(-1)
        free = ~self.bs_mask & ~clock
        dg = (g != g2) & free
        votes = []
        for oi in self.bs_by_act[act][:-1]:
            f, _, f2 = self.bs_trans[oi]
            r = (f != g) & free
            nr = int(r.sum())
            if nr == 0 or nr > limit:
                continue
            region = set(int(p) for p in np.nonzero(r)[0])
            df = (f != f2) & free
            # disagreements: cells changed in only one move, or to different colours
            dis = np.nonzero((df != dg) | (df & dg & (f2 != g2)))[0]
            dis = [int(p) for p in dis if p not in region]
            grew = True
            while grew and dis:
                grew = False
                keep = []
                for p in dis:
                    if any(n in region for n in _neighbours(p, w, h)):
                        region.add(p)
                        grew = True
                    else:
                        keep.append(p)
                dis = keep
            reg = np.zeros(h * w, bool)
            reg[list(region)] = True
            ef, eg = df & ~reg, dg & ~reg
            same = bool((ef == eg).all() and (f2[ef] == g2[ef]).all())
            if same:
                effect = [int(p) for p in np.nonzero(ef)[0]]
            else:
                if len(_components(region, w, h)) > 1:
                    continue
                effect = [int(p) for p in np.nonzero((ef != eg) | (ef & eg & (f2 != g2)))[0]]
            votes.append((sorted(region), same, effect))
        if not votes:
            return False
        bits = self._class(act)
        for region, support, effect in votes:
            self.bs_votes.append((region, bits, support, effect))
        self._revote()
        return self._recompute()

    def _revote(self):
        n = self.bs_mask.size
        self.bs_sup = np.zeros(n, np.int64)
        self.bs_sup_n = np.zeros(n, np.int64)
        self.bs_full_n = np.zeros(n, np.int64)
        self.bs_con_n = np.zeros(n, np.int64)
        for region, bits, support, effect in self.bs_votes:
            full = any(not self.bs_tick[p] for p in effect)
            if not full:
                continue
            if support:
                self.bs_sup[region] |= bits
                self.bs_full_n[region] += 1
            else:
                self.bs_con_n[region] += 1

    def _recompute(self):
        if not hasattr(self, "bs_sup") or self.bs_sup.size != self.bs_mask.size:
            self._revote()
        classes = np.array([bin(int(b)).count("1") for b in self.bs_sup])
        m = ((classes >= 2) & (self.bs_full_n > 2 * self.bs_con_n)) \
            | ((self.bs_full_n >= SOLO) & (self.bs_con_n == 0)) \
            | (self._tick_mask() & (self.bs_con_n == 0))
        changed = bool((m != self.bs_mask).any())
        self.bs_mask = m
        if changed:
            self.bs_stats["mask_changes"] += 1
        return changed

    def _tick_mask(self):
        """The tick cells as the key mode reads them (HY's life filter in the "+life" / "+chain" modes)."""
        if self.bs_key_mode not in ("tick+life", "tick+chain") or self.bs_shape is None:
            return self.bs_tick
        life = self.bs_life.mask(self.bs_shape).reshape(-1)
        if self.bs_key_mode == "tick+life":
            return self.bs_tick & life
        chain = {k for k in self.bs_ticks if any(j[1] == k[0] or k[1] == j[0] for j in self.bs_ticks if j != k)}
        out = self.bs_tick.copy()
        for p, ks in self.bs_tick_kinds.items():
            if ks & chain and not life[p]:
                out[p] = False
        return out

    def choose_action(self, frames, latest):
        if self.bs_key and self.bs_key_mode in ("tick+life", "tick+chain"):
            from arcengine import GameState
            if latest.state is GameState.GAME_OVER and self.prev is not None:
                self.bs_life.end_life()
                if self.bs_shape is not None and self._recompute():
                    self._remap()
        return super().choose_action(frames, latest)

    # ---- remap every state-keyed structure to the new keys ------------------------------------------------------------
    def _remap(self):
        old = dict(self.bs_key_grid)
        self.bs_key_grid = {}
        m = {k: self.key(g) for k, g in old.items()}
        f = lambda k: m.get(k, k)  # noqa: E731
        tried = collections.defaultdict(set)
        for k, v in self.tried.items():
            tried[f(k)] |= v
        self.tried = tried
        graph = collections.defaultdict(dict)
        for k, edges in self.graph.items():
            for a, n in edges.items():
                graph[f(k)].setdefault(a, f(n))
        self.graph = graph
        acts, kinds = {}, {}
        for k, v in self.acts_of.items():
            acts.setdefault(f(k), v)
        for k, v in self.kinds_of.items():
            kinds.setdefault(f(k), v)
        self.acts_of, self.kinds_of = acts, kinds
        parent = {}
        for k, (pk, t) in self.parent.items():
            if f(k) != f(pk):
                parent.setdefault(f(k), (f(pk), t))
        self.parent = parent
        self.dead = {(f(k), a) for k, a in self.dead}
        if self.start_key is not None:
            self.start_key = f(self.start_key)
        if self.goal is not None:
            self.goal = (f(self.goal[0]), self.goal[1])
        if self.pend is not None:
            k0, a, t, first, par = self.pend
            self.pend = (f(k0), a, t, first, None if par is None else (f(par[0]), par[1]))
        if self.pend_key is not None:
            self.pend_key = f(self.pend_key)
        if self.pend_pred is not None:
            src, nxt, cls = self.pend_pred
            self.pend_pred = (src, f(nxt) if isinstance(nxt, str) else nxt, cls)
        self.route = []
        self.bs_stats["remaps"] += 1


In [ ]:
%%writefile /kaggle/working/dg3.py
# SPDX-License-Identifier: MIT-0
"""Stream DG3's switch over kit BH (`../kit-bh/bs.py`, `BsThink`). Off by default; off, `Dg3Think` plays exactly as
`BsThink`. (A second switch, no click candidate on a painted cell, cost su15's level on the public 25 and was deleted:
`docs/research/1.9-streams/dg3-diagnosis.md`.)

* `DG3_BAR=1` -- A BUDGET BAR IS NOT STATE. A bar that shortens by a cell every action (or every few) changes WITH the
  action's own effect, never alone, so the tick test (a kind that was a move's only change) never fires on it and no
  cell of it changes in most transitions (v3's clock). Every state is then new: the planner's P(new) stays near 1, a
  toggle pressed twice (on, off) or a move and its undo look like discoveries, and the search never closes a loop.
  The test, per change kind (a cell's old and new colour), over the current life:
  - ISOLATED: only transitions where its cells form small parts (<= 3 cells) touching no other change count;
  - MONOTONE: a cell it changed did not change again in the life (a refill, the inverse kind, is ignored), and its
    inverse never came in the same transition (an object's leading and trailing edges);
  - A LINE: every cell it changed this life lies in one row or one column, the cells a run without gaps over 2;
  - >= 4 times in the life under >= 2 action classes (keys, or distinct clicks), or >= 8 times under one.
  A kind that passes paints out its line's run of its two colours (BS's tick cells: the key and the effect drop them,
  and every state-keyed structure is remapped). A kind that broke MONOTONE in this level never passes in it.

AUTHORED (labelled, as the rules ask): the four conditions and their numbers. LEARNED per game: which kinds, which
cells. Nothing names a game, a colour or a mechanic.
"""
import collections
import os

import numpy as np

import bs

MIN_TIMES = 4       # occurrences of a kind in one life before it can be a bar
MAX_PART = 3        # cells in one isolated part of one transition
MAX_GAP = 2         # largest gap between the bar's cells along its line


class Dg3Think(bs.BsThink):
    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed, verifier=verifier, arm=arm)
        self.dg3_bar = bool(int(os.environ.get("DG3_BAR", "0")))
        self.dg3_stats = collections.Counter()
        self._dg3_reset_level(None)

    def _dg3_reset_level(self, shape):
        self.dg3_shape, self.dg3_level = shape, getattr(self, "level", 0)
        self.dg3_banned = set()          # kinds that broke MONOTONE in this level
        self.dg3_bars = set()            # kinds painted in this level
        self._dg3_reset_life()

    def _dg3_reset_life(self):
        self.dg3_cells = collections.defaultdict(set)      # kind -> cells it changed this life
        self.dg3_times = collections.Counter()             # kind -> transitions it appeared in this life
        self.dg3_classes = collections.defaultdict(set)    # kind -> action classes it appeared under
        self.dg3_touched = collections.defaultdict(set)    # cell -> the kinds that changed it this life (no refill)

    def choose_action(self, frames, latest):
        if self.dg3_bar:
            from arcengine import GameState
            if latest.state is GameState.GAME_OVER:
                self._dg3_reset_life()
        return super().choose_action(frames, latest)

    def effect(self, a, b):
        e = super().effect(a, b)
        if not self.dg3_bar or not self.bs_key or a.shape != b.shape or self.prev is None:
            return e
        if self.dg3_shape != a.shape or self.dg3_level != self.level:
            self._dg3_reset_level(a.shape)
        if self._dg3_observe(a, b, self.prev[1]) and self._recompute():
            self._remap()
            # the effect as BS reads it, with the new painted cells
            d = (a != b).reshape(-1) & ~self.clock_mask(a.shape).reshape(-1) & ~self.bs_mask
            return bool(d.any())
        return e

    def _dg3_observe(self, a, b, act):
        h, w = a.shape
        d = a != b
        if not d.any():
            return False
        ys, xs = np.nonzero(d)
        cells = [int(y) * w + int(x) for y, x in zip(ys, xs)]
        kind_of = {p: (int(a.flat[p]), int(b.flat[p])) for p in cells}
        parts = bs._components(cells, w, h)
        isolated = collections.defaultdict(lambda: True)
        here = collections.defaultdict(set)
        for comp in parts:
            ks = {kind_of[p] for p in comp}
            for k in ks:
                here[k].update(p for p in comp if kind_of[p] == k)
                if len(ks) > 1 or len(comp) > MAX_PART:
                    isolated[k] = False
        cls = act if " " in act else self._class(act)
        for k, ps in here.items():
            inv = (k[1], k[0])
            if inv in here:                                 # an object's leading and trailing edges: never a bar
                self.dg3_banned |= {k, inv}
                continue
            if inv in self.dg3_cells and ps <= self.dg3_cells[inv]:
                continue                                    # a refill of a bar: not a contradiction, not a tick
            for p in ps:
                if self.dg3_touched[p]:                     # changed twice in one life: no kind there is a bar
                    self.dg3_banned |= self.dg3_touched[p] | {k}
                self.dg3_touched[p].add(k)
            if not isolated[k]:
                continue                                    # next to another change: not counted
            self.dg3_cells[k] |= ps
            self.dg3_times[k] += 1
            self.dg3_classes[k].add(cls)
        new = []
        for k in here:
            if k in self.dg3_banned or k in self.dg3_bars:
                continue
            if self.dg3_times[k] < MIN_TIMES or (len(self.dg3_classes[k]) < 2 and self.dg3_times[k] < 2 * MIN_TIMES):
                continue
            line = self._line(self.dg3_cells[k], w)
            if line is not None:
                new.append((k, line))
        if not new:
            return False
        grew = False
        for k, (axis, idx, lo, hi) in new:
            self.dg3_bars.add(k)
            self.dg3_stats["bars"] += 1
            for t in range(lo, hi + 1):
                p = idx * w + t if axis == "row" else t * w + idx
                if int(b.flat[p]) in k and not self.bs_tick[p]:
                    self.bs_tick[p] = True
                    grew = True
            # extend along the line over the run of the bar's two colours
            for step in (-1, 1):
                t = lo - 1 if step < 0 else hi + 1
                while 0 <= t < (w if axis == "row" else h):
                    p = idx * w + t if axis == "row" else t * w + idx
                    if int(b.flat[p]) not in k:
                        break
                    if not self.bs_tick[p]:
                        self.bs_tick[p] = True
                        grew = True
                    t += step
        if grew:
            self.dg3_stats["painted"] = int(self.bs_tick.sum())
            self._revote()
        return grew

    @staticmethod
    def _line(cells, w):
        ys = {p // w for p in cells}
        xs = {p % w for p in cells}
        if len(ys) == 1:
            axis, idx, along = "row", next(iter(ys)), sorted(xs)
        elif len(xs) == 1:
            axis, idx, along = "col", next(iter(xs)), sorted(ys)
        else:
            return None
        if len(along) < 2 or any(b - a > MAX_GAP + 1 for a, b in zip(along, along[1:])):
            return None
        return axis, idx, along[0], along[-1]

    def dump(self):
        super().dump()
        import json
        rec = dict(step=self.n_steps, level=self.level, raw=self.raw, shape=self.dg3_shape,
                   bars=sorted(map(list, self.dg3_bars)), stats=dict(self.dg3_stats),
                   ticks=sorted(map(list, self.bs_ticks)), tick_cells=int(self.bs_tick.sum()))
        with open(os.path.join(self.diag, f"{self.game_id}.dg3.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")


In [ ]:
%%writefile /kaggle/working/dg1.py
# SPDX-License-Identifier: MIT-0
"""Stream DG1's switch over kit BH (`bs.BsThink`), off by default; off, `DgThink` plays exactly as `BsThink`.

* `DG1_REL=1` -- RELATIONAL NOVELTY. A state's relation cell is the set of colour pairs that touch in its grid (the
  cells the state key paints out excluded). The planner's P(new state) of an untried action at a reached state is
  multiplied by 1 + w / sqrt(n), n the decisions this level taken in states of the same relation cell (w =
  `DG1_REL_W`, default 1), once the level has taken `DG1_REL_AFTER` actions (default 300). Untried actions where
  objects touch in a way seldom seen are worth more than one more position in open space (count-based exploration
  over a coarse cell, as Go-Explore's). Before 300 actions the level plays as kit BH: the early hits carry the score.

AUTHORED (labelled): the relation cell (touching colour pairs), the weight form and the 300-action threshold.
LEARNED per level: the counts. Nothing names a game, a colour or a mechanic.
"""
import collections
import os

import numpy as np

import bs


def relation_cell(h):
    """The set of unordered colour pairs that touch (4-neighbours) in `h`; cells < 0 (painted out) take no part."""
    out = set()
    for a, b in ((h[:, 1:], h[:, :-1]), (h[1:, :], h[:-1, :])):
        m = (a != b) & (a >= 0) & (b >= 0)
        if m.any():
            lo, hi = np.minimum(a[m], b[m]), np.maximum(a[m], b[m])
            out |= set(np.unique(lo * 64 + hi).tolist())
    return frozenset(out)


class DgThink(bs.BsThink):
    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed, verifier=verifier, arm=arm)
        self.dg_rel = bool(int(os.environ.get("DG1_REL", "0")))
        self.dg_w = float(os.environ.get("DG1_REL_W", "1"))
        self.dg_after = int(os.environ.get("DG1_REL_AFTER", "300"))   # actions into a level before the weight applies
        self.dg_cell = {}                               # state key -> relation cell
        self.dg_n = collections.Counter()               # relation cell -> decisions this level
        self.dg_level = None
        self.dg_start = 0
        self.dg_stats = collections.Counter()

    def _masked(self, g):
        h = g.copy()
        cm = self.clock_mask(g.shape)
        if cm.shape == g.shape:
            h[cm] = -1
        if self.bs_key and getattr(self, "bs_shape", None) == g.shape and self.bs_mask.size == h.size:
            h.reshape(-1)[self.bs_mask] = -1
        return h

    def plan(self, key, acts_here, g, comp_of):
        if self.dg_rel:
            if self.dg_level is None or self.dg_level[0] != self.level:
                self.dg_start = self.n_steps
            if self.dg_level != (self.level, self.raw):
                self.dg_level = (self.level, self.raw)
                self.dg_cell, self.dg_n = {}, collections.Counter()
            c = relation_cell(self._masked(g))
            if key not in self.dg_cell:
                self.dg_stats["cells"] += c not in self.dg_n
            self.dg_cell[key] = c
            self.dg_n[c] += 1
        return super().plan(key, acts_here, g, comp_of)

    def predict_next(self, n, a, t, par, graph=True):
        pred = super().predict_next(n, a, t, par, graph)
        if self.dg_rel and not graph and pred["source"] == "rates" and self.dg_steps() >= self.dg_after:
            c = self.dg_cell.get(n)
            if c is not None:
                pred = dict(pred, p_new=pred["p_new"] * (1.0 + self.dg_w / np.sqrt(self.dg_n[c])))
        return pred

    def dg_steps(self):
        """Actions this level (the raw switch restarts TK's level count, not this one)."""
        return self.n_steps - self.dg_start

    def _remap(self):
        super()._remap()
        if self.dg_rel:
            # keys changed: re-read every known state's cell from the grid that made it
            self.dg_cell = {k: relation_cell(self._masked(g)) for k, g in self.bs_key_grid.items()}


In [ ]:
%%writefile /kaggle/working/dg13.py
# SPDX-License-Identifier: MIT-0
"""Kit DG13's policy: kit DG1's relational novelty (`dg1.DgThink`, frozen at 241797d2) over kit DG3's bar test
(`dg3.Dg3Think`, frozen at b69bd659), composed by cooperative inheritance. Both files are used byte for byte; this
class adds nothing. Method order: DgThink (plan, predict_next, _remap), then Dg3Think (choose_action, effect, the bar
test), then BS's BsThink and the rest of kit BH."""
import dg1
import dg3


class Dg13Think(dg1.DgThink, dg3.Dg3Think):
    pass


In [ ]:
%%writefile /kaggle/working/rules.py
# SPDX-License-Identifier: MIT-0
"""Stream RU: an object-level rule model, learned from every transition and kept across levels.

Kit DG13 learns STATES (a graph of whole-screen hashes, wiped at every level). This module learns RULES: what an
action does to the objects, keyed by the self's relation to the object it meets, so a rule seen on level 1 predicts a
screen never seen on level 2.

    m = RuleModel()
    m.observe(g0, action, g1, died=False)        # every transition (g1 the next grid; None after a death)
    m.observe_win(g0, action, g_final)           # the transition that completed a level (g_final = frame[0])
    g1_hat, info = m.predict(g0, action)         # the predicted next grid, or None when no rule covers it

The world, as this module reads it (AUTHORED, game-blind):
* OBJECTS: 4-connected components of one colour on the logical grid; the background is the most common colour; the
  type of an object is its colour (colours keep their meaning across the levels of one game). Cells < 0 are masked.
* THE SELF: the colour that moves rigidly most often under non-click actions (HY's rule); its displacement per action
  is the mode of the observed ones, stored in units of the self's own extent (a level may draw cells larger).
* THE RELATION (the rule's key): for a move, the colour the self would enter (`ahead`), the colour one step further
  (`ahead2`), and whether each is the last object of its colour; for a click, the clicked colour; for any other
  action, the action alone. Keys back off from the full relation to `ahead` alone.
* THE EFFECT (the rule's outcome), relative to the self: how far the self went (0, 1 step, or `slide`: until the next
  non-background cell), what happened to the object ahead (`same`, `push`: moved one step with the self, `gone`,
  `recolour` to a colour), the remote colour changes (old -> new, `all` cells of the old colour or a count) and death.
LEARNED: the self, its displacements, every rule's outcome counts and its VERIFIED rate (each rule predicts before
it learns; a prediction is checked against the next grid), and the win condition (`WinModel`).
"""
import collections
from fractions import Fraction

import numpy as np

EDGE = -2          # outside the grid
MASK = -1          # a masked cell (the clock)


# ------------------------------------------------------------------------------------------------ objects (AUTHORED)
def background(g):
    v = g[g >= 0]
    if not v.size:
        return 0
    vals, cnt = np.unique(v, return_counts=True)
    return int(vals[np.argmax(cnt)])


try:
    from scipy import ndimage as _ndi
except Exception:  # pragma: no cover - the pure-python reading below is the same components
    _ndi = None


def components(mask):
    """4-connected components of a boolean mask: a label array (-1 outside) and the list of (ys, xs) arrays."""
    if _ndi is not None:
        lab, n = _ndi.label(mask)
        lab = lab.astype(np.int32) - 1
        if n == 0:
            return lab, []
        ys, xs = np.nonzero(lab >= 0)
        ids = lab[ys, xs]
        order = np.argsort(ids, kind="stable")
        ys, xs, ids = ys[order], xs[order], ids[order]
        cuts = np.flatnonzero(np.diff(ids)) + 1
        # components ordered by their first cell in row-major order (as the python reading below)
        comps = list(zip(np.split(ys, cuts), np.split(xs, cuts)))
        return lab, comps
    h, w = mask.shape
    lab = -np.ones((h, w), np.int32)
    comps = []
    ys, xs = np.nonzero(mask)
    for y0, x0 in zip(ys.tolist(), xs.tolist()):
        if lab[y0, x0] >= 0:
            continue
        i = len(comps)
        stack = [(y0, x0)]
        lab[y0, x0] = i
        cy, cx = [], []
        while stack:
            y, x = stack.pop()
            cy.append(y)
            cx.append(x)
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < h and 0 <= xx < w and mask[yy, xx] and lab[yy, xx] < 0:
                    lab[yy, xx] = i
                    stack.append((yy, xx))
        comps.append((np.array(cy), np.array(cx)))
    return lab, comps


def surround(g, ys, xs, exclude=()):
    """The most common colour 4-adjacent to the cells (ys, xs), outside them and not in `exclude` (masked excluded)."""
    h, w = g.shape
    own = set(zip(ys.tolist(), xs.tolist()))
    c = collections.Counter()
    for y, x in own:
        for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
            if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in own:
                v = int(g[yy, xx])
                if v >= 0 and v not in exclude:
                    c[v] += 1
    return c.most_common(1)[0][0] if c else None


def colour_at(g, y, x):
    h, w = g.shape
    return int(g[y, x]) if 0 <= y < h and 0 <= x < w else EDGE


class Scene:
    """One grid read as objects (cached per grid)."""

    def __init__(self, g):
        self.g = g
        self.bg = background(g)
        v = g[g >= 0].ravel()
        vals, first, cnt = np.unique(v, return_index=True, return_counts=True)
        order = np.argsort(first, kind="stable")             # first-appearance order, as a Counter built by a scan
        self.counts = collections.Counter({int(vals[i]): int(cnt[i]) for i in order})
        self._lab = {}
        self._n = {}
        self._self = {}

    def comps(self, c):
        if c not in self._lab:
            self._lab[c] = components(self.g == c)
        return self._lab[c]

    def object_at(self, y, x):
        """(colour, ys, xs) of the component through (y, x), or None for background / outside / masked."""
        c = colour_at(self.g, y, x)
        if c < 0 or c == self.bg:
            return None
        lab, comps = self.comps(c)
        ys, xs = comps[lab[y, x]]
        return c, ys, xs

    def n_objects(self, c):
        if c in self._lab:
            return len(self._lab[c][1])
        if c not in self._n:
            self._n[c] = _ndi.label(self.g == c)[1] if _ndi is not None else len(self.comps(c)[1])
        return self._n[c]


# ------------------------------------------------------------------------------------------------ the model
def _sign(v):
    return (v > 0) - (v < 0)


class RuleModel:
    def __init__(self):
        self.self_votes = collections.Counter()         # colour -> rigid moves under non-click actions
        self.disp = collections.defaultdict(collections.Counter)   # action -> Counter((dy/h, dx/w) as Fractions)
        self.rules = collections.defaultdict(collections.Counter)  # key -> Counter(outcome)
        self.checked = collections.defaultdict(lambda: [0, 0])     # key -> [right, checked] (verified rate)
        self.self_cells = None                                       # (ys, xs) of the self last seen
        self.floor = None                                            # the static layer: last colour seen per cell
        self.movers = set()                                          # colours seen pushed (they leave floor behind)
        self.pair_steps = collections.Counter()                      # remote (old, new) -> steps it happened in
        self.n_steps = 0                                              # steps read (the autonomous rate's base)
        self.ptr = {}                                                 # (old, new) -> (last cells, step delta)
        self._rem = {}
        self.pair_noop = collections.Counter()                       # ... steps where the action itself did nothing
        self.pair_keys = collections.defaultdict(set)                # ... the rule keys it came under
        self.pair_gap = collections.defaultdict(collections.Counter)  # ... steps between two of its changes
        self.pair_last = {}                                           # ... the step it last happened
        self.ptr_ok = collections.defaultdict(lambda: [0, 0])         # ... its pointer's predictions [right, made]
        self.n_obs = 0
        self.log = collections.Counter()

    # ---- the self ------------------------------------------------------------------------------------------------
    @property
    def self_colour(self):
        if not self.self_votes:
            return None
        c, v = self.self_votes.most_common(1)[0]
        return c if v >= 1.5 else None

    def locate_self(self, sc):
        c = self.self_colour
        sk = self.self_cells
        key = (c, None if sk is None else (sk[0].tobytes(), sk[1].tobytes()))
        hit = sc._self.get(key, False)
        if hit is not False:
            return hit                                   # the same reading of the same scene (a search's actions)
        if c is None or not (sc.g == c).any():
            sc._self[key] = None
            return None
        lab, comps = sc.comps(c)
        if sk is not None and len(sk[0]):
            cy, cx = sk[0].mean(), sk[1].mean()
        else:
            cy, cx = sc.g.shape[0] / 2, sc.g.shape[1] / 2
        out = min(comps, key=lambda p: (np.min((p[0] - cy) ** 2 + (p[1] - cx) ** 2), -len(p[0])))
        sc._self[key] = out
        return out

    @staticmethod
    def extent(cells):
        ys, xs = cells
        return int(ys.max() - ys.min() + 1), int(xs.max() - xs.min() + 1)

    def step_of(self, a, cells):
        """The self's usual displacement (dy, dx) in cells for action a, scaled to its current extent; None if unknown."""
        c = self.disp.get(a)
        if not c or cells is None:
            return None
        (fy, fx), _ = c.most_common(1)[0]
        h, w = self.extent(cells)
        dy, dx = fy * h, fx * w
        if dy.denominator != 1 or dx.denominator != 1:
            return None
        return int(dy), int(dx)

    # ---- the relation: what the self meets -----------------------------------------------------------------------
    def relation(self, sc, a, xy=None):
        """(keys from specific to general, context) for action a in scene sc."""
        if a == 6:
            if xy is None:
                return [("click", None)], dict(kind="click")
            x, y = xy
            o = sc.object_at(y, x)
            c = colour_at(sc.g, y, x)
            last = o is not None and sc.n_objects(o[0]) == 1
            return [("click", c, last), ("click", c)], dict(kind="click", obj=o, xy=xy)
        cells = self.locate_self(sc)
        d = self.step_of(a, cells)
        if d is None or d == (0, 0):
            return [("act", a)], dict(kind="act", cells=cells, d=d)
        ys, xs = cells
        dy, dx = d
        own = set(zip(ys.tolist(), xs.tolist()))
        ahead = [(y + dy, x + dx) for y, x in own if (y + dy, x + dx) not in own]
        # a jump (a move longer than the self): the colour passed over decides as much as where it lands
        n = max(abs(dy), abs(dx))
        via = None
        if n > 1:
            sy, sx = _sign(dy), _sign(dx)
            land = set(ahead)
            vc = collections.Counter(colour_at(sc.g, y + j * sy, x + j * sx) for y, x in own for j in range(1, n)
                                     if (y + j * sy, x + j * sx) not in own and (y + j * sy, x + j * sx) not in land)
            if vc:
                via = vc.most_common(1)[0][0]
                via = "bg" if via == sc.bg else via
        cols = collections.Counter(colour_at(sc.g, y, x) for y, x in ahead)
        c1 = cols.most_common(1)[0][0] if cols else EDGE
        o1 = None
        if c1 >= 0 and c1 != sc.bg:
            y, x = next(p for p in ahead if colour_at(sc.g, *p) == c1)
            o1 = sc.object_at(y, x)
        # one step beyond the object ahead (or beyond the self when the way is free)
        if o1 is not None:
            oc = set(zip(o1[1].tolist(), o1[2].tolist()))
            beyond = [(y + dy, x + dx) for y, x in oc if (y + dy, x + dx) not in oc and (y + dy, x + dx) not in own]
        else:
            beyond = [(y + 2 * dy, x + 2 * dx) for y, x in own if (y + 2 * dy, x + 2 * dx) not in own]
        cols2 = collections.Counter(colour_at(sc.g, y, x) for y, x in beyond)
        c2 = cols2.most_common(1)[0][0] if cols2 else EDGE
        k1 = "bg" if c1 == sc.bg else c1
        k2 = "bg" if c2 == sc.bg else c2
        last1 = o1 is not None and sc.n_objects(o1[0]) == 1
        o2 = None
        if c2 >= 0 and c2 != sc.bg and beyond:
            y, x = next(p for p in beyond if colour_at(sc.g, *p) == c2)
            o2 = sc.object_at(y, x)
        last2 = o2 is not None and sc.n_objects(o2[0]) == 1
        if via is not None:
            k1 = (via, k1)                             # a jump: what it passes over, then where it lands
        keys = [("mv", k1, k2, last1, last2), ("mv", k1, k2)]
        cls2 = self.solidity(k2)
        if cls2 is not None:
            keys.append(("mv", k1, cls2))              # what lies beyond, by what it does to the self (generalises)
        keys.append(("mv", k1))
        return keys, dict(kind="mv", cells=cells, d=d, o1=o1, c1=c1, c2=c2)

    def solidity(self, k):
        """How a colour treats the self, learned: `solid` (the self stays), `free` (it enters), `edge`, else None."""
        if k == EDGE:
            return "edge"
        c = self.rules.get(("mv", k))
        if not c:
            return None
        moves = sum(n for o, n in c.items() if o[0] not in ("death",) and o[0] != 0)
        stays = sum(n for o, n in c.items() if o[0] == 0)
        if moves + stays < 2:
            return None
        return "free" if moves > stays else "solid"

    # ---- the effect of an observed transition ----------------------------------------------------------------------
    def outcome(self, sc0, g1, ctx, died):
        """The observed effect, in the model's terms (a hashable tuple), or None when it cannot be read."""
        if died:
            return ("death",)
        g0 = sc0.g
        if g1 is None or g1.shape != g0.shape:
            return None
        valid = (g0 >= 0) & (g1 >= 0)
        diff = (g0 != g1) & valid
        local = np.zeros(g0.shape, bool)
        k, eff1 = 0, "same"
        if ctx["kind"] == "mv":
            ys, xs = ctx["cells"]
            dy, dx = ctx["d"]
            sc1 = Scene(g1)
            c = self.self_colour
            # how far the self went along d (0 .. the grid's size)
            span = max(g0.shape)
            k = None
            for j in range(0, span):
                yy, xx = ys + j * dy, xs + j * dx
                if yy.min() < 0 or xx.min() < 0 or yy.max() >= g0.shape[0] or xx.max() >= g0.shape[1]:
                    break
                if (g1[yy, xx] == c).all():
                    k = j
                    if j > 0:
                        break
                    # stayed? only if no displaced copy exists further along
                    continue
            if k is None:
                return None
            # the self's old and new cells are local
            for j in (0, k):
                local[np.clip(ys + j * dy, 0, g0.shape[0] - 1), np.clip(xs + j * dx, 0, g0.shape[1] - 1)] = True
            if k > 1:
                # a slide: the cells passed over are local too, and the slide is `slide` if it stopped at a non-bg cell
                for j in range(1, k):
                    local[ys + j * dy, xs + j * dx] = True
                nxt = [colour_at(g0, y, x) for y, x in zip((ys + (k + 1) * dy).tolist(), (xs + (k + 1) * dx).tolist())]
                k = "slide" if any(v != sc0.bg and v not in (MASK,) for v in nxt if (v != c)) else k
            o1 = ctx["o1"]
            if o1 is not None:
                col, oy, ox = o1
                local[oy, ox] = True
                ny, nx = oy + dy, ox + dx
                inside = ny.min() >= 0 and nx.min() >= 0 and ny.max() < g0.shape[0] and nx.max() < g0.shape[1]
                dest = g1[ny, nx] if inside else None
                moved = inside and k == 1 and not (g1[oy, ox] == col).all() and len(set(dest.tolist())) == 1 \
                    and int(dest[0]) != c and (g0[ny, nx] != dest).any() \
                    and (int(dest[0]) == col or (g0[ny, nx] != dest).all())
                if moved:
                    eff1 = "push" if int(dest[0]) == col else ("push", int(dest[0]))
                    local[ny, nx] = True
                elif (g1[oy, ox] == col).all():
                    eff1 = "same"
                else:
                    covered = (g1[oy, ox] == c)
                    rest = g1[oy, ox][~covered]
                    if (rest == col).all() and rest.size:
                        eff1 = "same"                          # partly under the self
                    elif rest.size == 0:
                        eff1 = "under"                         # wholly under the self: unknown
                    else:
                        vals = collections.Counter(int(v) for v in rest)
                        nv = vals.most_common(1)[0][0]
                        fl = surround(g0, oy, ox, (c, col))
                        eff1 = "gone" if nv in (sc1.bg, fl) else ("recolour", nv)
        elif ctx["kind"] == "click":
            o = ctx.get("obj")
            if o is not None:
                col, oy, ox = o
                local[oy, ox] = True
                vals = collections.Counter(int(v) for v in g1[oy, ox])
                nv = vals.most_common(1)[0][0]
                eff1 = "same" if nv == col else ("gone" if nv in (sc0.bg, surround(g0, oy, ox, (col,))) else ("recolour", nv))
        # remote changes: colour pairs, `all` when every cell of the old colour outside the local region changed
        rem = diff & ~local
        ry, rx = np.nonzero(rem)
        cells = collections.defaultdict(list)
        for y, x in zip(ry.tolist(), rx.tolist()):
            cells[(int(g0[y, x]), int(g1[y, x]))].append((y, x))
        self._rem = {p: frozenset(v) for p, v in cells.items()}
        remote = []
        for (o, n), v in sorted(cells.items()):
            if self.autonomous((o, n)):
                continue                                     # the world's own step, not the action's (Ticks)
            tot = int(((g0 == o) & ~local & valid).sum())
            remote.append((o, n, "all" if len(v) == tot else "some"))
        return (k, eff1, tuple(remote))

    # ---- the world's own steps (pointers) ------------------------------------------------------------------------
    def autonomous(self, p):
        """A remote change that happens whatever the action (a bar, a clock, a patrol): in most steps, or on steps
        where the action itself did nothing and under three or more relations."""
        if self.n_steps >= 6 and self.pair_steps[p] >= 0.5 * self.n_steps:
            return True
        return self.pair_noop[p] >= 2 and len(self.pair_keys[p]) >= 3

    def due(self, p):
        """Whether an autonomous change is due on the next step: its usual gap since its last one has passed."""
        gaps = self.pair_gap.get(p)
        if not gaps or p not in self.pair_last:
            return True
        gap = gaps.most_common(1)[0][0]
        return self.n_steps - self.pair_last[p] >= gap

    def learn_ticks(self, key=None, noop=False):
        """After a read step: count every remote pair, move its pointer (the cells it changed, and their shift)."""
        self.n_steps += 1
        for p, cs in self._rem.items():
            self.pair_steps[p] += 1
            self.pair_noop[p] += int(noop)
            if key is not None:
                self.pair_keys[p].add(key)
            if p in self.pair_last:
                self.pair_gap[p][self.n_steps - self.pair_last[p]] += 1
            self.pair_last[p] = self.n_steps
            last = self.ptr.get(p)
            delta = None
            if last is not None and len(last[0]) == len(cs):
                a = np.array(sorted(last[0]))
                b = np.array(sorted(cs))
                dd = b.mean(0) - a.mean(0)
                if np.allclose(dd, np.round(dd)):
                    delta = (int(round(dd[0])), int(round(dd[1])))
            self.ptr[p] = (cs, delta if delta is not None else (last[1] if last is not None else None))

    def tick_cells(self, g, p):
        """Where pair p = (old, new) changes next: its last cells shifted by its last step, among cells of the old
        colour; reversed when that is empty (a bounce); the last cells' neighbours of the old colour otherwise."""
        last = self.ptr.get(p)
        if last is None:
            return []
        cs, d = last
        h, w = g.shape
        o = p[0]

        def ok(c):
            return [(y, x) for y, x in c if 0 <= y < h and 0 <= x < w and g[y, x] == o]
        if d is not None and d != (0, 0):
            for s in (1, -1):
                c = ok([(y + s * d[0], x + s * d[1]) for y, x in cs])
                if len(c) == len(cs):
                    return c
        return []

    # ---- learning --------------------------------------------------------------------------------------------------
    def learn_self(self, g0, a, g1):
        """HY's self rule: colours that moved rigidly under a non-click action vote; the self's displacement."""
        if g1 is None or g1.shape != g0.shape or a in (0, 6):
            return
        d = (g0 != g1) & (g0 >= 0) & (g1 >= 0)
        if not d.any():
            return
        for c in set(int(v) for v in g0[d]) | set(int(v) for v in g1[d]):
            t = _translation(g0, g1, c)
            if t is not None and _whole(g0, g1, c, t):
                # one vote per step, shared by the colour's objects: the self is one thing, a dotted floor many
                self.self_votes[c] += 1.0 / max(1, len(components(g0 == c)[1]))
        c = self.self_colour
        if c is None:
            return
        t = _translation(g0, g1, c)
        sc = Scene(g0)
        cells = self.locate_self(sc)
        if t is not None and cells is not None:
            h, w = self.extent(cells)
            dy, dx = t
            self.disp[a][(Fraction(dy, h), Fraction(dx, w))] += 1

    def observe(self, g0, a, g1, died=False, xy=None):
        """Learn from one transition: check the prediction first (the verified rate), then count the outcome."""
        self.n_obs += 1
        if not died:
            self.learn_self(g0, a, g1)
        sc0 = Scene(g0)
        keys, ctx = self.relation(sc0, a, xy)
        out = self.outcome(sc0, g1, ctx, died)
        if out is None:
            self.log["unreadable"] += 1
            return None
        pred = self.best(keys)
        if pred is not None:
            key, o, _ = pred
            r = self.checked[key]
            r[0] += int(o == out)
            r[1] += 1
        for k in keys:
            self.rules[k][out] += 1
        if ctx["kind"] == "mv" and not died and g1 is not None:
            sc1 = Scene(g1)
            self.self_cells = self.locate_self(sc1)
        if out[0] != "death" and len(out) > 1 and isinstance(out[1], tuple) and out[1][0] == "push":
            self.movers.add(ctx["o1"][0])
        elif out[0] != "death" and out[1] == "push":
            self.movers.add(ctx["o1"][0])
        self.see(g0)
        if g1 is not None:
            self.see(g1)
            for p in list(self.ptr):
                if not self.due(p):
                    continue
                pc = frozenset(self.tick_cells(g0, p))
                if pc:
                    r = self.ptr_ok[p]
                    r[0] += int(pc == self._rem.get(p, frozenset()))
                    r[1] += 1
            self.learn_ticks(keys[-1], out != ("death",) and out[0] == 0 and out[1] == "same")
        return out

    def see(self, g):
        """Update the static layer: every cell not covered by the self or a mover keeps its last colour."""
        if self.floor is None or self.floor.shape != g.shape:
            self.floor = np.full(g.shape, MASK, np.int64)
        movers = set(self.movers)
        if self.self_colour is not None:
            movers.add(self.self_colour)
        m = (g >= 0) & ~np.isin(g, list(movers))
        self.floor[m] = g[m]

    def new_level(self):
        """Positions reset at a level change; the rules, the self and the movers are kept."""
        self.floor = None
        self.self_cells = None
        self.ptr = {}
        self.pair_last = {}

    def best(self, keys):
        """(key, outcome, its share) of the most specific key with data."""
        for k in keys:
            c = self.rules.get(k)
            if c:
                o, n = c.most_common(1)[0]
                return k, o, n / sum(c.values())
        return None

    def rate(self, key):
        r, n = self.checked[key]
        return (r + 1.0) / (n + 2.0)

    # ---- prediction ------------------------------------------------------------------------------------------------
    def predict(self, g0, a, xy=None, ticks=True):
        """(predicted next grid or None, info). info: key, outcome, share, verified rate, death."""
        sc0 = Scene(g0)
        keys, ctx = self.relation(sc0, a, xy)
        b = self.best(keys)
        if b is None:
            return None, dict(key=None)
        key, out, share = b
        info = dict(key=key, outcome=out, share=share, rate=self.rate(key), death=out == ("death",))
        if out == ("death",):
            return None, info
        g = apply(sc0, ctx, out, self.self_colour, self.floor)
        if g is None:
            return None, info
        # the world's own steps and the action's counted changes, where their pointers say
        if not ticks:
            return g, info
        todo = [p for p in self.ptr if self.autonomous(p) and self.due(p)] + \
            [(o, n) for o, n, how in out[2] if how != "all"]
        for p in dict.fromkeys(todo):
            r = self.ptr_ok[p]
            if r[1] < 3 or r[0] < 0.6 * r[1]:
                continue                                    # a pointer is used only once verified
            for y, x in self.tick_cells(g0, p):
                if g[y, x] == g0[y, x]:
                    g[y, x] = p[1]
        return g, info


def apply(sc0, ctx, out, self_colour, floor=None):
    """The grid an outcome produces from scene sc0 (the model's imagination)."""
    g0 = sc0.g
    g = g0.copy()
    k, eff1, remote = out
    local = np.zeros(g0.shape, bool)
    fl = floor if floor is not None and floor.shape == g0.shape else None

    def vacate(ys, xs, col=None, under=None):
        dflt = under if under is not None else surround(g0, ys, xs, (self_colour, col))
        dflt = sc0.bg if dflt is None else dflt
        if fl is None:
            g[ys, xs] = dflt
        else:
            v = fl[ys, xs]
            g[ys, xs] = np.where((v >= 0) & (v != col), v, dflt)
    if ctx["kind"] == "mv":
        ys, xs = ctx["cells"]
        dy, dx = ctx["d"]
        o1 = ctx["o1"]
        h, w = g0.shape
        if k == "slide":
            j = 0
            while True:
                ny, nx = ys + (j + 1) * dy, xs + (j + 1) * dx
                if ny.min() < 0 or nx.min() < 0 or ny.max() >= h or nx.max() >= w:
                    break
                v = g0[ny, nx]
                if ((v != sc0.bg) & (v != self_colour) & (v >= 0)).any():
                    break
                j += 1
            k = j
        if o1 is not None:
            col, oy, ox = o1
            local[oy, ox] = True
            if eff1 == "push" or (isinstance(eff1, tuple) and eff1[0] == "push"):
                ny, nx = oy + dy, ox + dx
                if ny.min() < 0 or nx.min() < 0 or ny.max() >= h or nx.max() >= w:
                    return None
                vacate(oy, ox, col)
                g[ny, nx] = col if eff1 == "push" else eff1[1]
                local[ny, nx] = True
            elif eff1 == "gone":
                vacate(oy, ox, col)
            elif isinstance(eff1, tuple) and eff1[0] == "recolour":
                g[oy, ox] = eff1[1]
        if k:
            ny, nx = ys + k * dy, xs + k * dx
            if ny.min() < 0 or nx.min() < 0 or ny.max() >= h or nx.max() >= w:
                return None
            c1 = ctx.get("c1")
            # what the self leaves behind, where never seen: the floor it steps onto, if it steps onto floor
            under = c1 if (o1 is None or eff1 == "under") and c1 is not None and c1 >= 0 else None
            vacate(ys, xs, under=under)
            g[ny, nx] = self_colour
            for j in range(0, k + 1):
                local[ys + j * dy, xs + j * dx] = True
        else:
            local[ys, xs] = True
    elif ctx["kind"] == "click":
        o = ctx.get("obj")
        if o is not None:
            col, oy, ox = o
            local[oy, ox] = True
            if eff1 == "gone":
                vacate(oy, ox, col)
            elif isinstance(eff1, tuple):
                g[oy, ox] = eff1[1]
    # remote colour changes, simultaneously (a swap is two pairs); a counted change is not localised: left as is
    out_g = g.copy()
    for o, n, how in remote:
        if how == "all":
            out_g[(g0 == o) & ~local] = n
    return out_g


def _whole(a, b, c, t):
    """Whether the objects of colour c that moved moved whole: each component the move left, shifted by t, lies on
    colour c in b (a moving thing; the floor the self walks over "moves" the other way, but not whole)."""
    A = a == c
    gone = A & ~(b == c)
    lab, comps = components(A)
    h, w = a.shape
    dy, dx = t
    for i in set(lab[gone].tolist()):
        ys, xs = comps[i]
        ny, nx = ys + dy, xs + dx
        if ny.min() < 0 or nx.min() < 0 or ny.max() >= h or nx.max() >= w or not (b[ny, nx] == c).all():
            return False
    return True


def _translation(a, b, c):
    """(dy, dx) by which the cells of colour c that changed moved rigidly from a to b, else None (muse.translation's
    test, on the changed part of the colour)."""
    A, B = a == c, b == c
    gone, new = A & ~B, B & ~A
    if not gone.any() or not new.any() or gone.sum() != new.sum():
        return None
    gy, gx = np.nonzero(gone)
    ny, nx = np.nonzero(new)
    for dy, dx in {(int(ny[0] - y), int(nx[0] - x)) for y, x in zip(gy.tolist(), gx.tolist())}:
        if dy == 0 and dx == 0:
            continue
        sh = np.zeros_like(A)
        h, w = A.shape
        ys, xs = np.nonzero(A)
        yy, xx = ys + dy, xs + dx
        ok = (yy >= 0) & (yy < h) & (xx >= 0) & (xx < w)
        moved = A & ~(A & B & ~gone)          # every cell of the colour
        # the moved part: the old cells that left, shifted, must equal the new cells plus the old cells kept under it
        src = np.zeros_like(A)
        src[gy, gx] = True
        # include old cells that stay covered by the moved copy (overlap of old and new positions)
        for y, x in zip(ys.tolist(), xs.tolist()):
            if B[y, x] and 0 <= y - dy < h and 0 <= x - dx < w and A[y - dy, x - dx] and not gone[y, x]:
                pass
        sy, sx = np.nonzero(src)
        ty, tx = sy + dy, sx + dx
        if (ty < 0).any() or (tx < 0).any() or (ty >= h).any() or (tx >= w).any():
            continue
        sh[ty, tx] = True
        if (sh & ~B).any():
            continue
        if ((B & ~A) & ~sh).any():
            continue
        del moved, ok, yy, xx
        return dy, dx
    return None


In [ ]:
%%writefile /kaggle/working/ru.py
# SPDX-License-Identifier: MIT-0
"""Stream RU's agent: kit DG13 with an object-level rule model that is kept across levels and planned over.

`RuThink` is kit DG13's policy (`dg13.Dg13Think`) plus three things, all in this kit (nothing in a crate):
* LEARN RULES (`rules.RuleModel`): every transition, on the logical grid with DG13's clock and bars painted out,
  updates the rules (what an action does to the objects, keyed by the self's relation to what it meets). Rules,
  the self and its displacements are KEPT across levels; only positions reset.
* LEARN THE WIN (`WinModel`): at each level completion, which object-level conditions became true (AUTHORED
  vocabulary: the self entered a colour, a colour reached zero cells, a colour was clicked); a condition that became
  true on a step that did not finish the level is refuted once; the conditions true at every win so far, least
  refuted first, are the goal.
* PLAN WITH THEM (`imagine`): at a level whose start the rules cover, a breadth-first search over PREDICTED grids
  (the rules as the model; predicted deaths pruned; the life's verified budget respected) for an action sequence
  whose last step makes the goal true. The plan is executed while each step's prediction comes true; on a miss the
  rules learn and the plan is searched again (predict, verify, learn). No plan: kit DG13 decides, as before.
AUTHORED (labelled): the object reading, the relation key, the effect classes, the win vocabulary, the search and its
caps. GROWN: the rules, their verified rates, the win condition and which conditions matter.

Switches (fixed in `my_agent.py` for a kit; the environment for screens): RU_PLAN (1: plan to the learned win from
level 2 on), RU_NODES (search cap, 2000), RU_OPT (1: when the verified rules give no plan, one more search in which
one untested move per plan steps onto what it meets; three such plans a level at most, none after one missed), RU_PROBE (0: also walk to
the nearest untested relation), RU_L1 (0: also plan on level 1), RU_PRUNE (0: the rules score DG13's untried
actions), RU_DEBUG (0). An optimistic plan that misses ends optimism for its level.
"""
import collections
import os
import time

import numpy as np

import dg13
import rules


class WinModel:
    """The level's win condition as object-level hypotheses, supported at wins and refuted on steps."""

    def __init__(self):
        self.support = collections.Counter()
        self.refute = collections.Counter()
        self.level_refute = collections.Counter()     # refutations in the current level (reset at each level)
        self.level_dead = collections.Counter()       # deaths while a plan toward it was walked, this level
        self.wins = 0

    @staticmethod
    def hyps(m, g0, a, xy, g1, final):
        """The conditions the transition g0 -a-> g1 made true (`final`: g1 is the completed level's frame)."""
        out = set()
        sc0 = rules.Scene(g0)
        keys, ctx = m.relation(sc0, a, xy)
        sc = m.self_colour
        if ctx["kind"] == "mv" and ctx.get("c1") is not None and ctx["c1"] >= 0 and ctx["c1"] != sc0.bg:
            if final:
                out.add(("enter", int(ctx["c1"])))
            elif g1 is not None:
                o = m.outcome(sc0, g1, ctx, False)
                if o is not None and o != ("death",) and o[0] != 0:
                    out.add(("enter", int(ctx["c1"])))
        if ctx["kind"] == "click" and xy is not None:
            out.add(("click", rules.colour_at(g0, xy[1], xy[0])))
        if g1 is not None and g1.shape == g0.shape:
            for c in set(int(v) for v in np.unique(g0)) - set(int(v) for v in np.unique(g1)):
                if c >= 0 and c != sc and c != sc0.bg:
                    out.add(("gone", c))
        return out

    def observe_win(self, hs):
        self.wins += 1
        self.support.update(hs)
        self.level_refute = collections.Counter()
        self.level_dead = collections.Counter()

    def observe_step(self, hs):
        for h in hs:
            if self.support[h]:
                self.refute[h] += 1
                self.level_refute[h] += 1

    def goals(self):
        """Conditions true at every win so far, least refuted first (ties: the support, then the name)."""
        if not self.wins:
            return []
        cand = [h for h, n in self.support.items()
                if n == self.wins and self.level_refute[h] < 2 and self.level_dead[h] < 3]
        return sorted(cand, key=lambda h: (self.refute[h] / (self.support[h] + self.refute[h]), str(h)))


def holds(goal, ctx, pred_g, out, a, xy, g0):
    """Whether an imagined step makes the goal true."""
    kind, c = goal
    if kind == "enter":
        return ctx["kind"] == "mv" and ctx.get("c1") == c and out is not None and out[0] != 0
    if kind == "gone":
        return pred_g is not None and not (pred_g == c).any() and (g0 == c).any()
    if kind == "click":
        return xy is not None and rules.colour_at(g0, xy[1], xy[0]) == c
    return False


class RuThink(dg13.Dg13Think):
    def __init__(self, game_id, seed=1, verifier=None, arm="B"):
        super().__init__(game_id, seed, verifier=verifier, arm=arm)
        self.ru = rules.RuleModel()
        self.win = WinModel()
        self.ru_on = bool(int(os.environ.get("RU_PLAN", "1")))
        self.ru_nodes = int(os.environ.get("RU_NODES", "2000"))
        self.ru_probe = bool(int(os.environ.get("RU_PROBE", "0")))
        self.ru_l1 = bool(int(os.environ.get("RU_L1", "0")))
        self.ru_debug = bool(int(os.environ.get("RU_DEBUG", "0")))     # print the first failed searches (off)
        self.ru_prune = bool(int(os.environ.get("RU_PRUNE", "0")))     # the rules score DG13's untried actions (off)
        self.ru_opt = bool(int(os.environ.get("RU_OPT", "1")))         # an optimistic re-search (on)
        self.ru_plan_bold = False         # the plan in hand came from the optimistic search
        self.ru_goals = []                # the goals the plan in hand was searched for
        self._ru_cache = {}
        self.ru_bold = {}                # level -> optimistic plans made
        self.ru_prev = None              # (G, raw grid, action, xy) of the last decision
        self.ru_plan = []                # [(action string, action id, xy, predicted masked grid)]
        self.ru_fail = None              # (level, model size) at the last failed search: no search until it changes
        self.ru_stats = collections.Counter()
        self.ru_s = 0.0
        self.ru_level = 0

    # ---- reading ------------------------------------------------------------------------------------------------
    def _ru_mask(self, g):
        try:
            return self._masked(g)
        except Exception:
            return g.copy()

    @staticmethod
    def _ru_action(a):
        if " " in a:
            _, x, y = a.split()
            return 6, (int(x), int(y))
        return int(a), None

    # ---- the hook: every step's outcome --------------------------------------------------------------------------
    def choose_action(self, frames, latest):
        t0 = time.perf_counter()
        try:
            self._ru_observe(latest)
        except Exception as e:  # pragma: no cover - the rule layer must never break the agent
            self.ru_stats["errors"] += 1
            self.ru_last_error = repr(e)
            self.ru_prev = None
        self.ru_s += time.perf_counter() - t0
        act = super().choose_action(frames, latest)
        return act

    def _ru_observe(self, latest):
        from arcengine import GameState
        p, self.ru_prev = self.ru_prev, None
        if p is None:
            return
        G, g0raw, a, xy = p
        aid, axy = self._ru_action(a)
        g0 = self._ru_mask(g0raw)
        lv = int(latest.levels_completed or 0)
        if latest.state is GameState.GAME_OVER:
            self.ru.observe(g0, aid, None, died=True, xy=axy)
            if self.ru_plan:
                for h in self.ru_goals:
                    self.win.level_dead[h] += 1               # a life lost on the way: three, and it is dropped
            self.ru_plan = []
            self.ru.self_cells = None
            return
        if not latest.frame:
            return
        if lv > self.ru_level:
            # the completed level's last frame is the response's first frame
            g1 = None
            try:
                g1raw = G.grid(np.asarray(latest.frame[0], dtype=np.int64))
                if g1raw.shape == g0raw.shape:
                    g1 = self._ru_mask_like(g1raw, g0raw, g0)
            except Exception:
                g1 = None
            hs = WinModel.hyps(self.ru, g0, aid, axy, g1, True)
            self.win.observe_win(hs)
            self.ru_stats["wins"] += 1
            if g1 is not None:
                self.ru.observe(g0, aid, g1, xy=axy)
            self.ru_level = lv
            self.ru.new_level()
            self.ru_plan, self.ru_fail = [], None
            return
        g1raw = G.grid(np.asarray(latest.frame[-1], dtype=np.int64))
        if g1raw.shape != g0raw.shape:
            return
        g1 = self._ru_mask_like(g1raw, g0raw, g0)
        self.win.observe_step(WinModel.hyps(self.ru, g0, aid, axy, g1, False))
        self.ru.observe(g0, aid, g1, xy=axy)

    @staticmethod
    def _ru_mask_like(g1raw, g0raw, g0):
        g1 = g1raw.copy()
        g1[g0 < 0] = -1
        return g1

    # ---- the decision ---------------------------------------------------------------------------------------------
    def plan(self, key, acts_here, g, comp_of):
        a = None
        t0 = time.perf_counter()
        try:
            a = self._ru_plan(g, acts_here)
        except Exception as e:  # pragma: no cover
            self.ru_stats["errors"] += 1
            self.ru_last_error = repr(e)
            self.ru_plan = []
        self.ru_s += time.perf_counter() - t0
        if a is not None:
            self.ru_stats["ru_steps"] += 1
            self.goal = None
            out = a
        else:
            out = super().plan(key, acts_here, g, comp_of)
        return out

    def _think(self, latest):
        a = super()._think(latest)
        # remember the decision (the grid it was taken on and its reading) for the next step's observation
        if self.prev is not None and not self.raw:
            self.ru_prev = (self.G, self.prev[0], self.prev[1], None)
        else:
            self.ru_prev = None
        return a

    def _ru_plan(self, g, acts_here):
        if not self.ru_on or self.raw:
            return None
        gm = self._ru_mask(g)
        # on track: the last planned step came true
        if self.ru_plan:
            want = self.ru_plan[0][3]
            if want is not None and want.shape == gm.shape and ((want == gm) | (want < 0) | (gm < 0)).all():
                self.ru_plan.pop(0)
            else:
                self.ru_stats["plan_miss"] += 1
                if self.ru_plan_bold:
                    self.ru_bold[self.level] = 3             # an optimistic plan missed: no more optimism this level
                self.ru_plan = []
        if self.ru_plan:
            if self.ru_plan and self.ru_plan[0][0] == "#done":
                self.ru_plan = []
            else:
                a = self._ru_match(self.ru_plan[0][0], acts_here, gm)
                if a is not None:
                    self.ru_plan[0] = (a,) + self.ru_plan[0][1:]
                    self.ru_stats["plan_steps"] += 1
                    return a
                self.ru_plan = []
        goals = self.win.goals()
        if self.level == 0 and not self.ru_l1:
            goals = []
        if self.ru_probe:
            goals = goals[:3] + [("unknown", None)]
        if not goals:
            return None
        sig = (self.level, len(self.ru.rules), sum(sum(c.values()) for c in self.ru.rules.values()) // 25)
        if self.ru_fail == sig:
            return None
        budget = self.budget()
        left = None if budget is None else budget - self.life
        path = self.imagine(gm, acts_here, goals, left)
        if not path:
            self.ru_fail = sig
            self.ru_stats["search_fail"] += 1
            if self.ru_debug and self.ru_stats["search_fail"] <= 3:
                import sys
                print(f"RU_DEBUG {self.game_id} level {self.level} goals {goals} self {self.ru.self_colour} "
                      f"cells {self.ru.self_cells} left {left}\n" + "\n".join(" ".join(f"{v:2d}" for v in r) for r in gm),
                      file=sys.stderr, flush=True)
                for a in self._ru_actions(gm, acts_here)[:6]:
                    aid, axy = self._ru_action(a)
                    keys, ctx = self.ru.relation(rules.Scene(gm), aid, axy)
                    b = self.ru.best(keys)
                    print("   ", a, keys[0], b, None if b is None else round(self.ru.rate(b[0]), 2), file=sys.stderr)
            return None
        self.ru_stats["search_ok"] += 1
        self.ru_goals = [h for h in goals if h[0] != "unknown"]
        self.ru_plan = path
        a = self._ru_match(path[0][0], acts_here, gm)
        if a is None:
            self.ru_plan = []
            return None
        self.ru_plan[0] = (a,) + self.ru_plan[0][1:]
        return a

    @staticmethod
    def _ru_match(a, acts_here, g):
        """The real candidate for an imagined action: itself, or for a click another candidate on the same object."""
        if a in acts_here:
            return a
        if " " not in a:
            return None
        _, x, y = a.split()
        x, y = int(x), int(y)
        c = rules.colour_at(g, y, x)
        sc = rules.Scene(g)
        o = sc.object_at(y, x)
        if o is None:
            return None
        cells = set(zip(o[1].tolist(), o[2].tolist()))
        for b in acts_here:
            if " " in b:
                _, bx, by = b.split()
                if (int(by), int(bx)) in cells and rules.colour_at(g, int(by), int(bx)) == c:
                    return b
        return None

    def _ru_actions(self, g, acts_here):
        """The imagined action set: the keyboard actions here, and one click per object (its first cell)."""
        keys = [a for a in acts_here if " " not in a]
        if any(" " in a for a in acts_here):
            sc = rules.Scene(g)
            clicks = []
            for c in sorted(sc.counts):
                if c == sc.bg or c < 0:
                    continue
                for ys, xs in sc.comps(c)[1][:12]:
                    clicks.append(f"6 {int(xs[0])} {int(ys[0])}")
            keys += clicks[:48]
        return keys

    def imagine(self, g, acts_here, goals, left):
        """Breadth-first over predicted grids (`imagine`, below), with this agent's action set and cap."""
        acts = lambda gs: self._ru_actions(gs, acts_here)  # noqa: E731
        path, n, unknown = imagine(self.ru, g, acts, goals, left, self.ru_nodes)
        self.ru_plan_bold = False
        if not path and unknown and self.ru_opt and self.ru_bold.get(self.level, 0) < 3:
            # no plan the verified rules give: one more search where one untested move steps onto what it meets
            # (at most three such plans a level: a wrong one costs the life it is walked in)
            path, n2, _ = imagine(self.ru, g, acts, goals, left, self.ru_nodes, optimist=True)
            n += n2
            self.ru_stats["optimist_ok" if path else "optimist_fail"] += 1
            if path:
                self.ru_bold[self.level] = self.ru_bold.get(self.level, 0) + 1
                self.ru_plan_bold = True
        self.ru_stats["nodes"] += n
        return path

    # ---- the rules inside DG13's own explorer (RU_PRUNE) ------------------------------------------------------------
    def predict_next(self, n, a, t, par, graph=True):
        """DG13's prediction for an untried action, unless the rules predict its outcome confidently (a verified rate
        and a share of at least 0.8): then a predicted death is a death, a predicted known screen (or no change) is
        not new, and a predicted unseen screen is new."""
        pred = super().predict_next(n, a, t, par, graph)
        if not self.ru_prune or graph or pred["source"] != "rates" or self.raw:
            return pred
        g = getattr(self, "bs_key_grid", {}).get(n)
        if g is None:
            return pred
        ck = (n, a, self.ru.n_obs // 16)
        hit = self._ru_cache.get(ck)
        if hit is None:
            hit = ("?",)
            gm = self._ru_mask(g)
            aid, xy = self._ru_action(a)
            keep = self.ru.self_cells
            self.ru.self_cells = None
            try:
                pg, info = self.ru.predict(gm, aid, xy, ticks=False)
            finally:
                self.ru.self_cells = keep
            k = info.get("key")
            if k is not None and self.ru.rate(k) >= 0.8 and info.get("share", 0) >= 0.8:
                if info.get("death"):
                    hit = ("death",)
                elif pg is not None and pg.shape == g.shape:
                    h = g.copy()
                    h[pg >= 0] = pg[pg >= 0]
                    hit = ("next", self.key(h))
            if len(self._ru_cache) > 50000:
                self._ru_cache.clear()
            self._ru_cache[ck] = hit
        if hit[0] == "death":
            self.ru_stats["prune_death"] += 1
            return dict(pred, p_new=0.0, p_death=1.0)
        if hit[0] == "next":
            k = hit[1]
            known = k == n or k in self.graph or k in self.parent or k == self.start_key
            self.ru_stats["prune_known" if known else "prune_new"] += 1
            return dict(pred, p_new=0.0 if known else 1.0)
        return pred

    def ru_report(self):
        g = self.win.goals()
        return dict(stats=dict(self.ru_stats), ms=round(1000 * self.ru_s / max(1, self.n_steps), 2),
                    rules=len(self.ru.rules), self_colour=self.ru.self_colour, wins=self.win.wins,
                    goals=[list(map(str, h)) for h in g[:4]], unreadable=self.ru.log["unreadable"],
                    error=getattr(self, "ru_last_error", None))


def _action(a):
    if " " in a:
        _, x, y = a.split()
        return 6, (int(x), int(y))
    return int(a), None


def imagine(m, g, actions, goals, left, max_nodes, optimist=False):
    """Breadth-first over the grids the rule model `m` predicts from g: the shortest action sequence whose last step
    makes a goal true (`holds`). `actions(grid)` lists the imagined actions; predicted deaths and rules verified under
    1/3 are not taken; `left` caps the depth (the life's budget). Returns ([(action, id, xy, predicted grid)], the
    predictions made, whether an untested move was met). A goal ("unknown", None) also accepts a keyboard step no rule covers (a probe)."""
    probe = any(gl[0] == "unknown" for gl in goals)
    goals = [gl for gl in goals if gl[0] != "unknown"]
    keep = m.self_cells
    start = g.tobytes()
    parent = {start: None}
    grids = {start: (g, keep, rules.Scene(g))}
    q = collections.deque([start])
    depth = {start: 0}
    bold = {start: False}                                    # the path to this grid took an untested step
    unknown = [False]                                        # an untested move was met (else optimism adds nothing)
    n = 0
    found = None
    try:
        while q and n < max_nodes and found is None:
            s = q.popleft()
            gs, cells, sc0 = grids[s]
            if left is not None and depth[s] + 1 >= left:
                continue
            for a in actions(gs):
                aid, xy = _action(a)
                m.self_cells = cells
                keys, ctx = m.relation(sc0, aid, xy)
                b = m.best(keys)
                n += 1
                if b is None:
                    if ctx["kind"] == "mv":
                        unknown[0] = True
                    if probe and " " not in a:
                        found = (s, a, aid, xy, None)          # an untested relation: go and see
                        break
                    if not (optimist and ctx["kind"] == "mv") or bold[s]:
                        continue
                    out = (1, "under", ())                     # optimism, once per plan: the self steps onto it
                else:
                    key, out, share = b
                    if out == ("death",) or m.rate(key) < 0.34:
                        continue
                pg = rules.apply(sc0, ctx, out, m.self_colour, m.floor)
                if any(holds(gl, ctx, pg, out, aid, xy, gs) for gl in goals):
                    found = (s, a, aid, xy, pg)
                    break
                if pg is None:
                    continue
                t = pg.tobytes()
                if t in parent:
                    continue
                parent[t] = (s, a, aid, xy)
                m.self_cells = cells
                sc1 = rules.Scene(pg)                              # one reading per imagined grid
                ncells = m.locate_self(sc1)
                grids[t] = (pg, ncells, sc1)
                depth[t] = depth[s] + 1
                bold[t] = bold[s] or b is None
                q.append(t)
    finally:
        m.self_cells = keep
    if found is None:
        return [], n, unknown[0]
    s, a, aid, xy, pg = found
    path = [(a, aid, xy, pg)]
    while parent[s] is not None:
        ps, pa, paid, pxy = parent[s]
        path.append((pa, paid, pxy, grids[s][0]))
        s = ps
    path.reverse()
    return path, n, unknown[0]


In [ ]:
%%writefile /kaggle/working/threadsafe.py
# SPDX-License-Identifier: MIT-0
"""(Taken byte for byte in logic from kit OL's threadsafe.py, stream OL-W6.)
Per-thread module attributes, for the one global that kit RU's decision swaps: `play.click_cells`.

Kit BH's `bs.BsThink._think` (inherited by kit RU) sets `play.click_cells` to the game's own click generator for the
length of one decision and restores it after. The competition framework plays every game as a thread of one
process, so two games' decisions can overlap. One game can then read another's generator. And when the restores
interleave (A sets, B sets, A restores, B restores A's), the module keeps a dead game's generator for every later
decision. Kit OL makes the window longer (an LLM call can sit inside a decision), so it makes the attribute per
thread: an assignment is seen only by the thread that made it; a thread that set nothing reads the module's own
function. Run in one thread, every read returns what it returned before (the gate rows are unchanged), and the
module's own globals are untouched.

`install(play, "click_cells")` is idempotent. `install_action_data(GameAction)` (stream RU) does the same for the
click data arcengine's `GameAction` members carry (below).
"""
import threading
import types


class _PerThread(types.ModuleType):
    def __getattribute__(self, name):
        d = types.ModuleType.__getattribute__(self, "__dict__")
        if name in d.get("_ol_names", ()):
            tl = d["_ol_tl"].__dict__
            if name in tl:
                return tl[name]
        return types.ModuleType.__getattribute__(self, name)

    def __setattr__(self, name, value):
        d = types.ModuleType.__getattribute__(self, "__dict__")
        if name in d.get("_ol_names", ()):
            setattr(d["_ol_tl"], name, value)
            return
        types.ModuleType.__setattr__(self, name, value)


def install(module, *names):
    """Make `names` per-thread attributes of `module` (reads fall back to the module's own value)."""
    d = module.__dict__
    if "_ol_tl" not in d:
        d["_ol_tl"] = threading.local()
        d["_ol_names"] = frozenset()
    d["_ol_names"] = d["_ol_names"] | frozenset(names)
    if not isinstance(module, _PerThread):
        module.__class__ = _PerThread
    return module


def install_action_data(game_action_cls):
    """Make `action_data` per thread on the members of arcengine's `GameAction` enum (stream RU, 2026-10-28).

    `GameAction.ACTION6.set_data({...})` writes the click's coordinates onto the enum member, one object shared by
    every game of the process. With games as threads, one game's `set_data` can land between another's and that
    game's read of its own action (the verifier's read, the framework's submission). A thread now reads the data it
    set itself; a thread that set nothing reads the member's own default. Idempotent; one thread reads as before."""
    if getattr(game_action_cls, "_ru_tl", None) is not None:
        return game_action_cls
    tl = threading.local()

    def _get(self):
        d = tl.__dict__.get("data")
        if d is not None and self.name in d:
            return d[self.name]
        return self.__dict__["action_data"]

    def _set(self, value):
        d = tl.__dict__.setdefault("data", {})
        d[self.name] = value

    type.__setattr__(game_action_cls, "_ru_tl", tl)
    type.__setattr__(game_action_cls, "action_data", property(_get, _set))
    return game_action_cls


In [ ]:
%%writefile /kaggle/working/my_agent.py
# SPDX-License-Identifier: MIT-0
"""Kit RU: kit DG13 with stream RU's object-level rule model (`ru.RuThink`, `rules.py`; docs/research/1.9-streams/
ru-rules.md): rules learned from every transition and kept across levels, the win condition learned at each level
completion, and a search over predicted grids that plans toward it from level 2 on. Everything below is kit DG13's.

Kit DG13: kit DG3 composed with kit DG1, fixed in code: relational novelty (`dg1.DgThink`, stream DG1;
docs/research/1.9-streams/dg1-goal.md, read 3) over the bar test (`dg3.Dg3Think`), as `dg13.Dg13Think`. Once a level has
taken 300 actions, the planner's P(new state) for an untried action at a reached state is multiplied by
1 + 1 / sqrt(n), n the decisions this level in states whose touching colour pairs are the same.

Kit DG3: kit BH with one change, fixed in code: a budget bar is not state (`dg3.Dg3Think`, stream DG3;
docs/research/1.9-streams/dg3-diagnosis.md). A colour change that comes in small isolated parts, changes no cell
twice in a life, never comes with its inverse and stays on one row or column is painted out of the state key, like
BS's ticks (a bar shortens with every action's own effect, never alone, so the tick test misses it).

Kit BH: kit HV with stream BS's effect-graph switches (`bs.py`), fixed in code: the tick test paints out what every
action changes the same way (BS_KEY, tick only), and clicks are probed by effect (BS_CLICK: every object, the soft inert
rule). `BS_FREE=1` reads the switches from the environment instead; both off = kit HV.

Kit HV: kit TK (below) with stream HY's verified facts (`facts.py`, `agent.py`; `docs/research/1.9-streams/
hy-verify.md` reads 6-7). Before each action a verifier predicts the next frame's step facts and the self, checks them
after the action, and keeps per-fact verified rates. TK's search reads, for the current state, the death rate VERIFIED
in each prediction's probability bin (arm 1v) and the trusted effect / novelty verdicts, in place of its per-type rates.
Settings, fixed in code (`make_policy`): arm 1v, and the verifier in lite mode (it checks only the facts arm 1v reads).
Everything else is kit TK's wrapper as it was.

Kit TK: kit X (below) with stream TK's planner, `muse/think.py` (`ThinkAgent`, a subclass of v3's `ExploreAgent`;
`docs/research/1.9-streams/tk-think-prereg.md`).

Kit X: the gpu-track `explore` agent, v3, as it scored 0.238 on the public 25 (commit `a446b4d1`, default switches: `EXPLORE_PROGRESS=1.0`, `EXPLORE_FRONTIER=1`, seed 1).

`muse/` holds v3's five modules byte for byte (`explore.py`, `muse.py`, `play.py`, `ask.py`, `weights/eval.py`).
This file only adapts `explore.ExploreAgent` to the competition framework's `Agent`:
* `MAX_ACTIONS = 3000` (the framework's default is 80). v3 was measured at 1,000; at 3,000 its first 1,000 actions
  are the same play, three more public-25 levels fall after action 1,000 and no game scores less (README);
* a global 8.3 h guard, like every earlier kit (v3's own `is_done` stops only on a WIN);
* on an exception in a step (none on the public 25), a logged legal move instead of a dead game thread.
The seed is v3's: `random.Random(f"{game_id}:1:explore")`, which Python seeds by SHA-512, not by `hash()`, so it
repeats across processes. Nothing here changes a decision v3 makes.
"""
import logging
import os
import sys
import time

_HERE = os.path.dirname(os.path.abspath(__file__))
for _root in (os.environ.get("OM_KIT_X_ROOT"), _HERE, os.path.join(_HERE, "..", "kit-x"), "/kaggle/working"):
    if _root and os.path.isdir(os.path.join(_root, "muse")):
        sys.path.insert(0, os.path.join(_root, "muse"))
        break

os.environ.setdefault("TK_ARBITER", "1")   # kit TK's default since its gate read (tk-think-prereg.md §8.1)
for _root in (os.environ.get("OM_KIT_HV_ROOT"), _HERE, os.path.join(_HERE, "..", "kit-hy"), "/kaggle/working"):
    if _root and os.path.isfile(os.path.join(_root, "facts.py")):
        sys.path.insert(0, _root)
        break

import explore  # noqa: E402  (v3; it puts muse/ and muse/weights on sys.path itself)
import think  # noqa: E402  (stream TK's planner over v3's own models)
import hy_arms as hy  # noqa: E402  (stream HY: the arms over think.ThinkAgent)
import facts  # noqa: E402  (stream HY: the fact verifier)
sys.path.insert(0, _HERE)
if os.path.isfile(os.path.join(_HERE, "..", "kit-bh", "bs.py")):   # the repo layout; on Kaggle bs.py sits beside
    sys.path.insert(0, os.path.join(_HERE, "..", "kit-bh"))
for _kit in ("kit-dg13", "kit-dg3", "kit-dg1"):                                  # the repo layout; on Kaggle these sit beside
    if os.path.isdir(os.path.join(_HERE, "..", _kit)):
        sys.path.insert(0, os.path.join(_HERE, "..", _kit))
import bs  # noqa: E402  (stream BS: the effect-graph switches, BS_KEY / BS_CLICK, off by default)
import dg3  # noqa: E402  (stream DG3: a budget bar is not state)
import dg13  # noqa: E402  (stream DG1: kit DG1 over kit DG3)
import ru  # noqa: E402  (stream RU: the rule model over kit DG13)
import play  # noqa: E402  (kit X's click candidates; kit BH's decision swaps its click_cells)
import threadsafe  # noqa: E402  (stream OL-W6: that swap made per game thread; one thread reads as before)
threadsafe.install(play, "click_cells")
from arcengine import FrameData, GameAction, GameState  # noqa: E402
threadsafe.install_action_data(GameAction)   # stream RU: a click's data per game thread (GameAction members are shared)

try:
    from agents.agent import Agent  # provided by the competition kernel
except Exception:
    Agent = object  # type: ignore[assignment,misc]

logger = logging.getLogger()
_RUN_START = time.time()
_BUDGET_S = float(os.environ.get("OM_BUDGET_S", str(8.3 * 3600)))
import openmorph as _om  # noqa: E402
MAX_ACTIONS = int(os.environ.get("KIT_X_MAX_ACTIONS", "3000"))
print(f">>> kit RU-T2 | rules + win + plan over kit DG13, click_cells and click data per thread | explore v3 (a446b4d1) + think + HY facts (arm 1v) + BS ticks + clicks (nogrid, soft) + DG3 bars + DG1 relational novelty | openmorph {_om.__version__} | "
      f"MAX_ACTIONS {MAX_ACTIONS}", flush=True)


def make_policy(game_id):
    """Kit BH's policy: kit HV's (kit TK's planner with HY's arm 1v and a lite verifier) with stream BS's states by the
    tick test and click probing by effect (`bs.py`: BS_KEY tick only; every object, the soft inert rule), fixed in code; `BS_FREE=1` reads the BS_* switches instead
    (both off = kit HV, row for row on the public 25)."""
    v = facts.Verifier()
    v.lite = True
    a = ru.RuThink(game_id, seed=1, verifier=v, arm="1v")
    a.ru_on, a.ru_nodes, a.ru_probe, a.ru_l1, a.ru_opt = True, 2000, False, False, True   # kit RU: plan from level 2
    a.dg3_bar = True   # kit DG3: a budget bar is not state
    a.dg_rel, a.dg_w, a.dg_after = True, 1.0, 300   # kit DG1: relational novelty after 300 actions of a level, w 1
    if not int(os.environ.get("BS_FREE", "0")):
        # kit BH's setting, fixed in code: states by the tick test (BS_KEY, tick only) and click probing by effect
        # (every object, the soft inert rule)
        a.bs_key, a.bs_key_mode = True, "tick"
        a.bs_click, a.bs_click_mode, a.bs_click_inert = True, "nogrid", "soft"
    return _RuVerified(a, v, game_id=game_id)


class _RuVerified(facts.Verified):
    """HY's observer wrapper, with the rule layer's report in the gate's row."""

    def stats(self):
        r = super().stats()
        try:
            r["ru"] = self.policy.ru_report()
        except Exception as e:  # pragma: no cover
            r["ru"] = dict(error=repr(e))
        return r


class MorphAgent(Agent):
    """The `explore` agent v3 with stream TK's planner and stream HY's verified facts, in the official framework."""

    MAX_ACTIONS = MAX_ACTIONS

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.policy = make_policy(self.game_id)
        self.errors = 0
        logger.info(f"MorphAgent kit HV: HyThink 1v over explore v3 (game={self.game_id})")

    def is_done(self, frames, latest_frame: FrameData) -> bool:
        return self.policy.is_done(frames, latest_frame) or (time.time() - _RUN_START) > _BUDGET_S

    def choose_action(self, frames, latest_frame: FrameData) -> GameAction:
        try:
            return self.policy.choose_action(frames, latest_frame)
        except Exception as e:  # pragma: no cover
            self.errors += 1
            logger.warning(f"kit HV choose_action error #{self.errors} (game={self.game_id}), legal fallback: {e!r}")
            avail = sorted(int(getattr(a, "value", a)) for a in (getattr(latest_frame, "available_actions", None) or []))
            simple = [a for a in avail if a not in (0, 6)]
            if simple:
                return GameAction.from_id(simple[0])
            if 6 in avail:
                act = GameAction.from_id(6)
                act.set_data({"x": 32, "y": 32})
                return act
            return GameAction.RESET


## 5. Run the agent (competition) / placeholder (commit)

In the **competition rerun** we drop `MorphAgent` into the official `ARC-AGI-3-Agents` framework and run
`main.py --agent morphagent`: it plays every hidden game against the live gateway, and the gateway's
scorecard is turned into the submission. Outside a rerun (an interactive commit) that block is skipped and
we emit a placeholder `submission.parquet` so the notebook commits cleanly.

In [ ]:
# SETTINGS FOR THIS SUBMISSION, kit RU-T2: kit DG13 + RU rules + click_cells and click data per thread, defaults. The games run in a subprocess (the run
# cell's `!python main.py`), which inherits these variables.
import os
for _k in ('EXPLORE_PROGRESS', 'EXPLORE_FRONTIER', 'OM_AGENT_CONFIG', 'KIT_X_MAX_ACTIONS', 'TK_DEPTH', 'TK_PROGRESS', 'TK_UNDO', 'TK_DEATH',
           'TK_RAW', 'TK_BUDGET', 'TK_COMMIT', 'TK_LINE', 'TK_DIAG', 'TK_ARBITER', 'TK_V3ORDER',
           'HY_BASE', 'HY_ARM', 'HY_LITE', 'HY_TAU', 'HY_RECORD', 'HY_PROBE', 'HY_TRACE', 'HY_REPLAY',
           'HY_W1', 'HY_W3', 'HY_MAX_CELLS', 'HY_MAX_OBJECTS', 'HY_DEATH', 'HY_DEATH_PRIOR',
           'BS_FREE', 'BS_KEY', 'BS_KEY_MODE', 'BS_CLICK', 'BS_CLICK_MODE', 'BS_CLICK_INERT',
           'DG3_BAR', 'DG1_REL', 'DG1_REL_W', 'DG1_REL_AFTER',
           'RU_PLAN', 'RU_NODES', 'RU_PROBE', 'RU_L1', 'RU_DEBUG', 'RU_PRUNE', 'RU_OPT'):
    os.environ.pop(_k, None)   # v3's defaults: progress 1.0, frontier routing on
print('kit RU-T2 | explore v3 a446b4d1 + think + HY facts (arm 1v) + BS ticks + clicks (nogrid, soft) + DG3 bars + DG1 relational novelty + RU rules | every switch at its default')


In [ ]:
# PRE-FLIGHT: load my_agent.py as the framework will, and build one agent. It must print
# '>>> kit RU-T2 | rules + win + plan over kit DG13, click_cells and click data per thread | explore v3 (a446b4d1)
# + think + HY facts (arm 1v) + BS ticks +
# clicks (nogrid, soft) + DG3 bars + DG1 relational novelty | openmorph 1.8.0 | MAX_ACTIONS 3000' and
# 'kit RU-T2 pre-flight OK'.
import importlib.util
_spec = importlib.util.spec_from_file_location('kit_ru_preflight', '/kaggle/working/my_agent.py')
_kit_hv = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_kit_hv)
_p = _kit_hv.make_policy('preflight-0')                       # the policy MorphAgent builds
_a, _v = _p.policy, _p.v
_a.model.learn([0.0] * _kit_hv.explore.N_FEAT, 0.9)          # the RLS readouts load and learn on this wheel
_v.heads['death'].learn([0.0] * (_kit_hv.facts.N_ACT + _kit_hv.facts.N_AHEAD + _kit_hv.facts.N_LIFE), False)
assert _kit_hv.MorphAgent.MAX_ACTIONS == 3000 and _a.w_progress == 1.0 and _a.frontier
assert (_a.depth, _a.line, _a.use_raw, _a.use_budget, _a.use_commit, _a.arbiter) == (16, 1, True, True, True, True)
assert type(_a).__name__ == 'RuThink' and _a.arm == '1v' and _a.v is _v and _v.lite and _v.tau == 0.95
assert _v.replay_surprise == 0 and not _v.frozen and _kit_hv.facts.MAX_CELLS == 160 and _kit_hv.facts.MAX_OBJECTS == 48
assert (_a.bs_key, _a.bs_key_mode, _a.bs_click, _a.bs_click_mode, _a.bs_click_inert) == (
    True, 'tick', True, 'nogrid', 'soft')
assert _a.dg3_bar is True
assert (_a.dg_rel, _a.dg_w, _a.dg_after) == (True, 1.0, 300)
assert (_a.ru_on, _a.ru_nodes, _a.ru_probe, _a.ru_l1, _a.ru_opt) == (True, 2000, False, False, True)
import numpy as _np
_g = _np.full((8, 8), 3); _g[2, 2] = 5; _g[2, 5] = 7
_h = _g.copy(); _h[2, 2] = 3; _h[2, 3] = 5
for _i in range(3):
    _a.ru.observe(_g, 4, _h)                                 # the rule model reads, learns and predicts
assert _a.ru.self_colour == 5 and _a.ru.predict(_g, 4)[0] is not None
print('kit RU-T2 pre-flight OK')


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # 1) wait for the ARC gateway to come up
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    # 2) copy the official framework to a writable dir + drop our agent (and its perception files) in
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    # 3) register ONLY our agent + random (avoid importing templates with heavy optional deps)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MorphAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "morphagent": MorphAgent}
''')
    # 4) point the framework at the gateway
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')
    # 5) play every hidden game — the gateway scorecard is turned into the submission
    # PYTHONHASHSEED: without it abs(hash(game_id)) differs every process and the SAME program
    # moved by 1704x offline. Pinned so a run is reproducible and A/Bs are valid.
    !cd /kaggle/working/ARC-AGI-3-Agents && PYTHONHASHSEED=0 MPLBACKEND=agg python main.py --agent morphagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame([['1_0', '1', True, 1]],
                 columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet(
        '/kaggle/working/submission.parquet', index=False)
    print('placeholder submission written (interactive commit)')

---

### Why OpenMorph?
No backprop, fully online, and it **grows** its own capacity — a compact, efficient substrate for agents
that must *learn on the fly*, which is exactly what ARC-AGI-3 demands. This agent is an honest v1 baseline
(a coarse perception + online value learning); the substrate underneath — grown perception, options,
free-energy acting — is where the real gains come from.

`pip install openmorph` • [github.com/openmorph/openmorph-arc](https://github.com/openmorph/openmorph-arc)